# Reporte post Partido con datos de WhoScore y Fotmob

In [ ]:
# fotmob_url = "https://www.fotmob.com/matches/chelsea-vs-manchester-city/2d55kw#4506271"
# whoscored_url = "https://es.whoscored.com/Matches/1821051/Live/"

# Put the Fotmob matchId here
fotmob_matchId = 4506271

# WhoScore
match_id = 1821051

# Instalar paquetes 

In [ ]:
# pip install highlight_text
# pip install mplsoccer

In [ ]:
!pip install unidecode

In [ ]:
# !pip install --force-reinstall unidecode

# Import Packages

In [ ]:
import json
# import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgba
import requests
from bs4 import BeautifulSoup
import matplotlib.patches as patches
from mplsoccer import Pitch, add_image
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as path_effects

from highlight_text import ax_text, fig_text
from PIL import Image
from urllib.request import urlopen
import os
import time
from unidecode import unidecode
from scipy.spatial import ConvexHull

from selenium import webdriver  # Importa el módulo webdriver de Selenium, utilizado para automatizar navegadores web.
from selenium.webdriver.support import expected_conditions as EC  # Importa el módulo expected_conditions, que contiene una serie de condiciones que se pueden esperar (esperar a que un elemento sea clicable, visible, etc.).

# Configuración
import warnings  # Importa el módulo warnings, que maneja los mensajes de advertencia en Python.
warnings.filterwarnings("ignore")  # Configura para ignorar todas las advertencias.
warnings.simplefilter(action='ignore', category=FutureWarning)  # Configura para ignorar las advertencias de tipo FutureWarning, que se utilizan para notificar sobre cambios que pueden ocurrir en versiones futuras de las librerías.
pd.set_option('display.max_columns', None)  # Configura pandas para mostrar todas las columnas de un DataFrame al imprimirlo, sin truncar la visualización.
pd.set_option('display.max_rows', None)  # Configura pandas para mostrar todas las filas de un DataFrame al imprimirlo, sin truncar la visualización.

# especificar colores personalizados para usar
green = '#69f900'
red = '#ff4b44'
blue = '#00a0de'
violet = '#a369ff'
bg_color= '#f5f5f5'
line_color= '#000000'
# bg_color= '#000000'
# line_color= '#ffffff'
col1 = '#ff4b44'
col2 = '#00a0de'

In [ ]:
# !pip install pandas
!pip install selenium

# Extracción de datos de Eventos

La función setup_driver() inicializa un controlador de Selenium para interactuar con el navegador Chrome. Utiliza la clase webdriver.Chrome() para crear una instancia del controlador y, una vez creado, lo devuelve para ser utilizado en el resto del código.

In [ ]:
def setup_driver():
    driver = webdriver.Chrome()
    return driver

La función `extract_json_from_html()` se encarga de extraer datos en formato JSON desde una página web relacionada con un partido de fútbol, utilizando Selenium para acceder a la página de WhoScored y BeautifulSoup para analizar el contenido HTML. A continuación, te explico paso a paso:

1. **Parámetros:**
   - `match_id`: es el identificador del partido que se utiliza para construir la URL desde donde se extraerán los datos.
   - `save_output`: indica si se deben guardar los datos extraídos en archivos (por defecto, es `False`).

2. **Funcionamiento:**
   - Se inicializa un controlador de Chrome con Selenium para navegar a la página dinámica de un partido.
   - Luego, se realiza una pausa de 5 segundos (`time.sleep(5)`) para esperar la carga del contenido dinámico en la página.
   - Con `BeautifulSoup`, se obtiene el contenido HTML de la página cargada.
   - Se buscan y analizan todos los elementos `<script>` en el HTML para identificar el script que contiene los datos del partido, que están dentro de la variable `matchCentreData`.
   - Si se encuentran los datos, estos se manipulan para formatear el JSON adecuadamente y luego se cargan en un objeto Python.
   - Dependiendo de la opción `save_output`, los datos JSON se guardan en un archivo `.json` y también en un archivo `.txt` para fines de referencia.

3. **Manejo de errores:**
   - Si ocurre algún error durante el proceso de extracción, se captura y se muestra un mensaje indicando el error.

4. **Salida:**
   - Devuelve los datos JSON extraídos si se encuentran. Si no hay datos o si ocurre un error, devuelve una cadena vacía.



In [ ]:
def extract_json_from_html(match_id, save_output=False):
    driver = webdriver.Chrome()
    
    url = f'https://es.whoscored.com/Matches/{match_id}/Live/'

    datos = ''

    try:
        # Acceder a la página
        driver.get(url)
        
        # Esperar un poco para que se cargue el contenido dinámico
        time.sleep(5)
        
        # Obtener el contenido de la página
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Cerrar el driver
        driver.quit()

        # Buscar el script que contiene los datos
        scripts = soup.find_all("script")
        
        for script in scripts:
            if script.string and "matchCentreData" in script.string:
                datos = script.string
                break
        
        if datos:
            # Extraer los datos JSON
            listVar = datos.split(';')
            matchCenterData = listVar[0].split('require.config.params["args"] = ')[1]
            matchCenterData = matchCenterData.replace('\n','').replace('matchId','"matchId"')
            matchCenterData = matchCenterData.replace('matchCentreData','"matchCentreData"')
            matchCenterData = matchCenterData.replace('formationIdNameMappings','"formationIdNameMappings"')
            matchCenterData = matchCenterData.replace('matchCentreEventTypeJson','"matchCentreEventTypeJson"')
            
            # Cargar los datos JSON
            data = json.loads(matchCenterData)
            
            # Crear el nombre del archivo
            file_name = f"{match_id}.json"
            
            # Guardar el JSON
            with open(os.path.join("FData", file_name), 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=4)

            # save JSON data to txt 
            output_file = open(os.path.join("FData", f"{match_id}.txt"), "wt", encoding='utf-8')
            data_txt = matchCenterData.replace('};', '}')
            n = output_file.write(data_txt)
            output_file.close()
            
            return data
        else:
            print(f"No se encontraron datos para el partido {match_id}")
    except Exception as e:
        print(f"Error al procesar el partido {match_id}: {str(e)}")

    return datos

La función `extract_data_from_dict()` se encarga de extraer datos específicos de un diccionario que contiene información de un partido de fútbol (previamente cargado en formato JSON). A continuación, te explico los pasos de la función:

1. **Entrada:**
   - El parámetro `data` es un diccionario que contiene la información completa del partido extraída previamente.

2. **Extracción de datos:**
   - Se extraen varios elementos importantes del diccionario:
     - `event_types_json`: contiene los tipos de eventos ocurridos durante el partido.
     - `formation_mappings`: mapea los IDs de las formaciones con sus respectivos nombres.
     - `events_dict`: es un diccionario con todos los eventos registrados en el partido (goles, pases, etc.).
     - `teams_dict`: es un diccionario que contiene los nombres de los equipos y sus IDs.
     - `players_dict`: contiene los nombres de los jugadores mapeados a sus IDs.

3. **Creación de dataframes:**
   - Se crean dos `DataFrames` de Pandas, uno para los jugadores del equipo local (`players_home_df`) y otro para los jugadores del equipo visitante (`players_away_df`), añadiendo la columna `teamId` para identificar a qué equipo pertenece cada jugador.
   - Estos dos `DataFrames` se concatenan en un solo `players_df` que contiene la información de todos los jugadores involucrados en el partido.

4. **Salida:**
   - La función devuelve tres elementos:
     - `events_dict`: un diccionario de eventos del partido.
     - `players_df`: un `DataFrame` con la información de todos los jugadores del partido.
     - `teams_dict`: un diccionario con los IDs y nombres de los equipos.

Esta función organiza y prepara los datos del partido para su posterior análisis, facilitando el trabajo con eventos, jugadores y equipos.

In [ ]:
def extract_data_from_dict(data):
    # load data from json
    event_types_json = data["matchCentreEventTypeJson"]
    formation_mappings = data["formationIdNameMappings"]
    events_dict = data["matchCentreData"]["events"]
    teams_dict = {data["matchCentreData"]['home']['teamId']: data["matchCentreData"]['home']['name'],
                  data["matchCentreData"]['away']['teamId']: data["matchCentreData"]['away']['name']}
    players_dict = data["matchCentreData"]["playerIdNameDictionary"]
    # create players dataframe
    players_home_df = pd.DataFrame(data["matchCentreData"]['home']['players'])
    players_home_df["teamId"] = data["matchCentreData"]['home']['teamId']
    players_away_df = pd.DataFrame(data["matchCentreData"]['away']['players'])
    players_away_df["teamId"] = data["matchCentreData"]['away']['teamId']
    players_df = pd.concat([players_home_df, players_away_df])
    players_ids = data["matchCentreData"]["playerIdNameDictionary"]
    return events_dict, players_df, teams_dict


In [ ]:
data = extract_json_from_html(match_id)

events_dict, players_df, teams_dict = extract_data_from_dict(data)

df = pd.DataFrame(events_dict)
dfp = pd.DataFrame(players_df)

Guarda los datos en archivos CSV

In [ ]:
df.to_csv(os.path.join("FData", f"{match_id}_EventData_whoscored.csv"))
df = pd.read_csv(os.path.join("FData", f"{match_id}_EventData_whoscored.csv"))
dfp.to_csv(os.path.join("FData", f"{match_id}_PlayerData_whoscored.csv"))
dfp = pd.read_csv(os.path.join("FData", f"{match_id}_PlayerData_whoscored.csv"))

In [ ]:
df.head()

1. **Extracción del valor 'displayName':**
   - Las tres primeras líneas de código están utilizando expresiones regulares para extraer el valor del campo `'displayName'` de las columnas `type`, `outcomeType`, y `period`. Esto se realiza con `str.extract()` de Pandas, que extrae el texto que coincide con el patrón dentro de una cadena. El patrón `r"'displayName': '([^']+)"` busca el valor que se encuentra entre comillas simples justo después de `'displayName': `.

2. **Asignación temporal de valores en la columna 'period':**
   - La última línea del código reemplaza los valores extraídos en la columna `period` por identificadores numéricos que corresponden a las distintas fases del partido. Por ejemplo:
     - `'FirstHalf'` se reemplaza por `1` (primer tiempo),
     - `'SecondHalf'` por `2` (segundo tiempo),
     - `'FirstPeriodOfExtraTime'` por `3` (primer período de tiempo extra),
     - `'PenaltyShootout'` por `5` (penales),
     - y otros valores que representan distintas fases del partido.
   

In [ ]:
df['type'] = df['type'].str.extract(r"'displayName': '([^']+)")
df['outcomeType'] = df['outcomeType'].str.extract(r"'displayName': '([^']+)")
df['period'] = df['period'].str.extract(r"'displayName': '([^']+)")

df['period'] = df['period'].replace({'FirstHalf': 1, 'SecondHalf': 2, 'FirstPeriodOfExtraTime': 3, 'SecondPeriodOfExtraTime': 4, 
                                     'PenaltyShootout': 5, 'PostGame': 14, 'PreMatch': 16})

La función `cumulative_match_mins()` calcula los minutos acumulados de juego en base a los eventos de un partido de fútbol, teniendo en cuenta el tiempo dentro de cada período del partido (primer tiempo, segundo tiempo, tiempos extra, etc.). A continuación, se detalla cómo funciona:

1. **Parámetro de entrada:**
   - `events_df`: es un `DataFrame` que contiene los eventos de un partido de fútbol, incluyendo las columnas `minute`, `second`, y `period`, que representan el minuto y segundo en que ocurrió el evento, y el período del partido (primer tiempo, segundo tiempo, etc.).

2. **Cálculo de los minutos acumulados:**
   - Se crea una copia del `DataFrame` de eventos y se añade una nueva columna llamada `cumulative_mins`, que calcula el tiempo total en minutos para cada evento combinando el minuto (`minute`) y los segundos convertidos a minutos (`second` dividido por 60).
   
3. **Ajuste de los minutos acumulados por período:**
   - Para cada período del partido (primer tiempo, segundo tiempo, etc.), se ajustan los minutos acumulados:
     - Si el período es mayor que 1, se calcula un incremento de tiempo (`t_delta`), que representa la diferencia entre el tiempo acumulado máximo del período anterior y el tiempo acumulado mínimo del período actual, para evitar solapamientos en los minutos.
     - Si el período es el primero o el de los penales (indicados por `period == 1` o `period == 5`), no se añade incremento de tiempo, ya que no hay un período previo con el que ajustar.
     - Luego, se actualizan los minutos acumulados en el `DataFrame` de eventos para cada período, sumando el incremento calculado.

4. **Reconstrucción del DataFrame:**
   - Finalmente, se concatenan los eventos ajustados en un nuevo `DataFrame` llamado `events_out`, que es el resultado final con los minutos acumulados correctamente ajustados para cada período del partido.

5. **Salida:**
   - Devuelve el `DataFrame` `events_out` con una nueva columna `cumulative_mins`, que contiene los minutos acumulados de juego para cada evento, teniendo en cuenta los distintos períodos del partido.


In [ ]:
def cumulative_match_mins(events_df):
    events_out = pd.DataFrame()
    # Add cumulative time to events data, resetting for each unique match
    match_events = events_df.copy()
    match_events['cumulative_mins'] = match_events['minute'] + (1/60) * match_events['second']
    # Add time increment to cumulative minutes based on period of game.
    for period in np.arange(1, match_events['period'].max() + 1, 1):
        if period > 1:
            t_delta = match_events[match_events['period'] == period - 1]['cumulative_mins'].max() - \
                                   match_events[match_events['period'] == period]['cumulative_mins'].min()
        elif period == 1 or period == 5:
            t_delta = 0
        else:
            t_delta = 0
        match_events.loc[match_events['period'] == period, 'cumulative_mins'] += t_delta
    # Rebuild events dataframe
    events_out = pd.concat([events_out, match_events])
    return events_out

In [ ]:
df = cumulative_match_mins(df)

La función `insert_ball_carries()` se encarga de identificar y añadir eventos de "conducción de balón" (ball carries) dentro de un `DataFrame` de eventos de fútbol. Estos eventos de "carry" ocurren cuando un jugador lleva el balón de un punto a otro sin perder la posesión, y la función tiene varios parámetros configurables para determinar cuándo se considera un "carry" válido. A continuación se detalla el funcionamiento:

### Parámetros de la función:
- **`events_df`**: Un `DataFrame` que contiene todos los eventos de un partido de fútbol.
- **`min_carry_length`**: La distancia mínima (en metros) que debe recorrer el balón para que se considere un "carry". Por defecto es 3 metros.
- **`max_carry_length`**: La distancia máxima que puede recorrer el balón para que se considere un "carry". Por defecto es 60 metros.
- **`min_carry_duration`**: La duración mínima en segundos para que un "carry" sea válido. Por defecto es 1 segundo.
- **`max_carry_duration`**: La duración máxima en segundos que un "carry" puede durar. Por defecto es 10 segundos.

### Funcionamiento de la función:

1. **Preparación del DataFrame de eventos:**
   - Se resetea el índice del `DataFrame` de eventos y se inicializan los parámetros configurables para las conducciones.
   
2. **Iteración sobre los eventos:**
   - La función recorre cada evento del `DataFrame`. Para cada evento, se verifica si el evento siguiente pertenece al mismo equipo y cumple ciertos criterios para ser considerado una conducción del balón ("carry").
   - Durante esta iteración, se consideran eventos de tipo "TakeOn" (encarar al rival) exitosos y algunos eventos no válidos como "TakeOn" no exitosos o faltas, que podrían interrumpir la conducción.

3. **Condiciones para un carry válido:**
   - **Mismo equipo**: El siguiente evento debe pertenecer al mismo equipo.
   - **Distancia mínima y máxima**: Se calcula la distancia entre el final del evento actual y el inicio del siguiente, asegurándose de que esté entre los valores de `min_carry_length` y `max_carry_length`.
   - **Duración mínima y máxima**: Se verifica que el tiempo entre los dos eventos sea mayor a `min_carry_duration` y menor a `max_carry_duration`.
   - **Mismo período del partido**: Ambos eventos deben ocurrir en el mismo período del partido (primer tiempo, segundo tiempo, etc.).

4. **Creación del evento de carry:**
   - Si se cumplen todas las condiciones, se crea un nuevo evento de tipo "carry". Este nuevo evento incluye información como el equipo, el jugador, las coordenadas iniciales y finales, y los tiempos acumulados (cumulative minutes).
   - Los "carries" generados se guardan en un `DataFrame` separado.

5. **Reorganización de los eventos:**
   - Los eventos de "carry" se combinan con los eventos originales del partido, y luego se ordenan por el período del partido y el tiempo acumulado para asegurarse de que todos los eventos están en orden cronológico.

6. **Salida:**
   - La función devuelve un `DataFrame` que incluye tanto los eventos originales del partido como los nuevos eventos de "carry" insertados.

In [ ]:
def insert_ball_carries(events_df, min_carry_length=3, max_carry_length=60, min_carry_duration=1, max_carry_duration=10):
    events_out = pd.DataFrame()
    # Carry conditions (convert from metres to opta)
    min_carry_length = 3.0
    max_carry_length = 60.0
    min_carry_duration = 1.0
    max_carry_duration = 10.0
    # match_events = events_df[events_df['match_id'] == match_id].reset_index()
    match_events = events_df.reset_index()
    match_carries = pd.DataFrame()
    
    for idx, match_event in match_events.iterrows():

        if idx < len(match_events) - 1:
            prev_evt_team = match_event['teamId']
            next_evt_idx = idx + 1
            init_next_evt = match_events.loc[next_evt_idx]
            take_ons = 0
            incorrect_next_evt = True

            while incorrect_next_evt:

                next_evt = match_events.loc[next_evt_idx]

                if next_evt['type'] == 'TakeOn' and next_evt['outcomeType'] == 'Successful':
                    take_ons += 1
                    incorrect_next_evt = True

                elif ((next_evt['type'] == 'TakeOn' and next_evt['outcomeType'] == 'Unsuccessful')
                      or (next_evt['teamId'] != prev_evt_team and next_evt['type'] == 'Challenge' and next_evt['outcomeType'] == 'Unsuccessful')
                      or (next_evt['type'] == 'Foul')):
                    incorrect_next_evt = True

                else:
                    incorrect_next_evt = False

                next_evt_idx += 1

            # Apply some conditioning to determine whether carry criteria is satisfied
            same_team = prev_evt_team == next_evt['teamId']
            not_ball_touch = match_event['type'] != 'BallTouch'
            dx = 105*(match_event['endX'] - next_evt['x'])/100
            dy = 68*(match_event['endY'] - next_evt['y'])/100
            far_enough = dx ** 2 + dy ** 2 >= min_carry_length ** 2
            not_too_far = dx ** 2 + dy ** 2 <= max_carry_length ** 2
            dt = 60 * (next_evt['cumulative_mins'] - match_event['cumulative_mins'])
            min_time = dt >= min_carry_duration
            same_phase = dt < max_carry_duration
            same_period = match_event['period'] == next_evt['period']

            valid_carry = same_team & not_ball_touch & far_enough & not_too_far & min_time & same_phase &same_period

            if valid_carry:
                carry = pd.DataFrame()
                prev = match_event
                nex = next_evt

                carry.loc[0, 'eventId'] = prev['eventId'] + 0.5
                carry['minute'] = np.floor(((init_next_evt['minute'] * 60 + init_next_evt['second']) + (
                        prev['minute'] * 60 + prev['second'])) / (2 * 60))
                carry['second'] = (((init_next_evt['minute'] * 60 + init_next_evt['second']) +
                                    (prev['minute'] * 60 + prev['second'])) / 2) - (carry['minute'] * 60)
                carry['teamId'] = nex['teamId']
                carry['x'] = prev['endX']
                carry['y'] = prev['endY']
                carry['expandedMinute'] = np.floor(((init_next_evt['expandedMinute'] * 60 + init_next_evt['second']) +
                                                    (prev['expandedMinute'] * 60 + prev['second'])) / (2 * 60))
                carry['period'] = nex['period']
                carry['type'] = carry.apply(lambda x: {'value': 99, 'displayName': 'Carry'}, axis=1)
                carry['outcomeType'] = 'Successful'
                carry['qualifiers'] = carry.apply(lambda x: {'type': {'value': 999, 'displayName': 'takeOns'}, 'value': str(take_ons)}, axis=1)
                carry['satisfiedEventsTypes'] = carry.apply(lambda x: [], axis=1)
                carry['isTouch'] = True
                carry['playerId'] = nex['playerId']
                carry['endX'] = nex['x']
                carry['endY'] = nex['y']
                carry['blockedX'] = np.nan
                carry['blockedY'] = np.nan
                carry['goalMouthZ'] = np.nan
                carry['goalMouthY'] = np.nan
                carry['isShot'] = np.nan
                carry['relatedEventId'] = nex['eventId']
                carry['relatedPlayerId'] = np.nan
                carry['isGoal'] = np.nan
                carry['cardType'] = np.nan
                carry['isOwnGoal'] = np.nan
                carry['type'] = 'Carry'
                carry['cumulative_mins'] = (prev['cumulative_mins'] + init_next_evt['cumulative_mins']) / 2

                match_carries = pd.concat([match_carries, carry], ignore_index=True, sort=False)

    match_events_and_carries = pd.concat([match_carries, match_events], ignore_index=True, sort=False)
    match_events_and_carries = match_events_and_carries.sort_values(['period', 'cumulative_mins']).reset_index(drop=True)

    # Rebuild events dataframe
    events_out = pd.concat([events_out, match_events_and_carries])

    return events_out

In [ ]:
df = insert_ball_carries(df, min_carry_length=3, max_carry_length=60, min_carry_duration=1, max_carry_duration=10)

### Resumen:
Este código reinicia los índices del `DataFrame`, filtra los eventos que son pases o conducciones exitosas, y luego asigna un valor de `xT` (expected threat) a cada evento basado en una cuadrícula de valores predefinida. Los valores de xT representan la amenaza potencial que se genera al mover el balón desde una zona del campo a otra. Finalmente, se eliminan las columnas innecesarias y se fusionan los resultados con el `DataFrame` original para tener una representación completa de los eventos con los cálculos de xT.

1. **Reiniciar el índice del DataFrame:**
   - `df = df.reset_index(drop=True)` reinicia el índice del `DataFrame` `df` sin conservar el índice anterior.
   - Luego, se asigna un nuevo índice secuencial a la columna `'index'` que comienza desde 1 y termina en la longitud del `DataFrame`.

2. **Reorganización de columnas:**
   - `df = df[['index'] + [col for col in df.columns if col != 'index']]` asegura que la columna `'index'` sea la primera en el `DataFrame`.

3. **Filtrado de eventos para xT:**
   - Se copia el `DataFrame` original en `dfxT`, y luego se filtran los eventos:
     - Se eliminan aquellos cuya columna `qualifiers` contenga "Corner".
     - Se mantienen solo los eventos de tipo `Pass` y `Carry` (conducción del balón) que hayan sido exitosos (`outcomeType=='Successful'`).

4. **Asignación de valores de la cuadrícula de xT:**
   - Se carga la cuadrícula de valores `xT` desde un archivo CSV.
   - El `DataFrame` se procesa para asignar valores a zonas específicas del campo (definidas por coordenadas `x` e `y`) utilizando la cuadrícula de xT.
     - Las columnas `x1_bin_xT`, `y1_bin_xT`, `x2_bin_xT`, y `y2_bin_xT` dividen las coordenadas iniciales y finales en "bins" (intervalos) de acuerdo con el tamaño de la cuadrícula.
     - Los valores iniciales y finales de las zonas se extraen de la cuadrícula de xT y se almacenan en `start_zone_value_xT` y `end_zone_value_xT`.

5. **Cálculo del xT:**
   - El valor de `xT` se calcula restando el valor de la zona inicial (`start_zone_value_xT`) del valor de la zona final (`end_zone_value_xT`).

6. **Limpieza del DataFrame:**
   - Las columnas irrelevantes, como coordenadas, detalles del evento, y otras, se eliminan usando `dfxT.drop(columns=columns_to_drop, inplace=True)`.

7. **Fusión de los datos:**
   - Finalmente, el `DataFrame` original `df` se fusiona con `dfxT` (que contiene los cálculos de xT) usando la columna `'index'` como clave.



In [ ]:
df = df.reset_index(drop=True)
df['index'] = range(1, len(df) + 1)
df = df[['index'] + [col for col in df.columns if col != 'index']]

# Assign xT values
df_base  = df
dfxT = df_base.copy()
dfxT['qualifiers'] = dfxT['qualifiers'].astype(str)
dfxT = dfxT[(~dfxT['qualifiers'].str.contains('Corner'))]
dfxT = dfxT[(dfxT['type'].isin(['Pass', 'Carry'])) & (dfxT['outcomeType']=='Successful')]

xT = pd.read_csv(f"FData/xT_Grid.csv", header=None)    # Usa esto si tienes tu propia cuadrícula de valores xT, luego coloca la ruta de tu archivo aquí.
xT = np.array(xT)
xT_rows, xT_cols = xT.shape

dfxT['x1_bin_xT'] = pd.cut(dfxT['x'], bins=xT_cols, labels=False)
dfxT['y1_bin_xT'] = pd.cut(dfxT['y'], bins=xT_rows, labels=False)
dfxT['x2_bin_xT'] = pd.cut(dfxT['endX'], bins=xT_cols, labels=False)
dfxT['y2_bin_xT'] = pd.cut(dfxT['endY'], bins=xT_rows, labels=False)

dfxT['start_zone_value_xT'] = dfxT[['x1_bin_xT', 'y1_bin_xT']].apply(lambda x: xT[x[1]][x[0]], axis=1)
dfxT['end_zone_value_xT'] = dfxT[['x2_bin_xT', 'y2_bin_xT']].apply(lambda x: xT[x[1]][x[0]], axis=1)

dfxT['xT'] = dfxT['end_zone_value_xT'] - dfxT['start_zone_value_xT']
columns_to_drop = ['id', 'eventId', 'minute', 'second', 'teamId', 'x', 'y', 'expandedMinute', 'period', 'outcomeType', 'qualifiers',  'type',
                   'satisfiedEventsTypes', 'isTouch', 'playerId', 'endX', 'endY', 'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY',
                   'goalMouthZ', 'goalMouthY', 'isShot', 'Unnamed: 0', 'cumulative_mins']
dfxT.drop(columns=columns_to_drop, inplace=True)

df = df.merge(dfxT, on='index', how='left')

### Resumen:
Este código agrega información útil a los eventos de un partido, como los nombres de los equipos, ajusta las coordenadas del campo a dimensiones estándar, calcula la distancia progresiva de pases y conducciones para identificar acciones relevantes, y normaliza los nombres de los jugadores. Todo esto contribuye a preparar los datos para un análisis más detallado del rendimiento en el juego.

1. **Nuevas columnas para nombres de equipos y equipos rivales:**
   - La columna `teamName` se crea mapeando los `teamId` a sus nombres correspondientes utilizando el diccionario `teams_dict`.
   - Luego, se genera un nuevo diccionario `opposition_dict`, que asocia a cada equipo con su equipo rival, y se añade una columna `oppositionTeamName` que contiene el nombre del equipo contrario.

2. **Redimensionamiento de las coordenadas del campo:**
   - Dado que se está utilizando una cancha con dimensiones de la UEFA, se ajustan las coordenadas `x`, `y`, `endX`, `endY`, y `goalMouthY` para que correspondan a las dimensiones de 105x68 metros. Las coordenadas originales están basadas en una escala de 100x100.

3. **Eliminación de columnas irrelevantes:**
   - Se eliminan varias columnas del `DataFrame` `dfp` que no son necesarias para el análisis, como datos del jugador (`height`, `weight`, `age`), estadísticas y eventos de sustitución.

4. **Añadir información de jugadores:**
   - El `DataFrame` original `df` se fusiona con `dfp` (que contiene la información de los jugadores) basándose en la columna `playerId`, para agregar detalles como nombres y números de camiseta.

5. **Cálculo de la distancia progresiva de pases y conducciones:**
   - Se calcula la distancia reducida por un pase o conducción como indicador de si el pase o conducción es progresivo.
     - Para los **pases** (`type == 'Pass'`), se mide la diferencia de distancia desde el punto inicial hacia el centro del arco contrario antes y después del pase. Si la distancia disminuye en más de 10 yardas, se considera un pase progresivo.
     - Para las **conducciones** (`type == 'Carry'`), se utiliza la misma lógica para medir la distancia cubierta.

6. **Ángulo del pase o conducción:**
   - Se calcula el ángulo del pase o la conducción en grados usando la función `arctan2` que toma las diferencias entre las coordenadas finales e iniciales en los ejes `y` y `x`.

7. **Normalización de los nombres de jugadores:**
   - Se convierte la columna `name` en cadena de texto y se utiliza la función `unidecode` para reemplazar caracteres no ASCII, como las letras acentuadas (ej. "Á" por "A"), asegurando que los nombres de los jugadores estén en formato de caracteres ingleses estándar.



In [ ]:
# New Column for Team Names and Oppositon TeamNames
df['teamName'] = df['teamId'].map(teams_dict)
team_names = list(teams_dict.values())
opposition_dict = {team_names[i]: team_names[1-i] for i in range(len(team_names))}
df['oppositionTeamName'] = df['teamName'].map(opposition_dict)

# Reshaping the data from 100x100 to 105x68, as I use the pitch_type='uefa', in the pitch function, you can consider according to your use
df['x'] = df['x']*1.05
df['y'] = df['y']*0.68
df['endX'] = df['endX']*1.05
df['endY'] = df['endY']*0.68
df['goalMouthY'] = df['goalMouthY']*0.68

columns_to_drop = ['Unnamed: 0','height', 'weight', 'age', 'isManOfTheMatch', 'field', 'stats', 
                   'subbedInPlayerId', 'subbedOutPeriod', 
                   'subbedOutExpandedMinute', 'subbedInPeriod', 'subbedInExpandedMinute', 'subbedOutPlayerId', 
                   'teamId']
dfp.drop(columns=columns_to_drop, inplace=True)

# adding player name, shirt no. etc info
df = df.merge(dfp, on='playerId', how='left')

df['qualifiers'] = df['qualifiers'].astype(str)
# Calculating passing distance, to find out progressive pass, this will just show the distance reduced by a pass, then will be able to filter passes which has reduced distance value more than 10yds as a progressive pass
df['prog_pass'] = np.where((df['type'] == 'Pass'), 
                           np.sqrt((105 - df['x'])**2 + (34 - df['y'])**2) - np.sqrt((105 - df['endX'])**2 + (34 - df['endY'])**2), 0)
# Calculating carrying distance, to find out progressive carry, this will just show the distance reduced by a carry, then will be able to filter carries which has reduced distance value more than 10yds as a progressive carry
df['prog_carry'] = np.where((df['type'] == 'Carry'), 
                            np.sqrt((105 - df['x'])**2 + (34 - df['y'])**2) - np.sqrt((105 - df['endX'])**2 + (34 - df['endY'])**2), 0)
df['pass_or_carry_angle'] = np.degrees(np.arctan2(df['endY'] - df['y'], df['endX'] - df['x']))

# Making all the alphabets in the name as English Alphabets only (for example: Á will be replaced by A)
df['name'] = df['name'].astype(str)
df['name'] = df['name'].apply(unidecode)

### Resumen:
La función `get_short_name()` crea versiones abreviadas de los nombres completos de los jugadores, lo cual es útil para mostrar información de forma más concisa en reportes o visualizaciones. Luego, esta función se aplica a la columna `'name'` del `DataFrame`, y se realizan algunas transformaciones adicionales para limpiar los datos.

### Función `get_short_name(full_name)`:

1. **Propósito:**
   - La función toma un nombre completo (`full_name`) y devuelve una versión abreviada (nombre corto) basada en las iniciales y el apellido. Este es un enfoque común en deportes y análisis de datos cuando se quiere mostrar los nombres de manera más compacta.

2. **Funcionamiento:**
   - Si el nombre completo es `NaN` (faltante), simplemente lo devuelve como está.
   - Si el nombre tiene solo una palabra, no se acorta y se devuelve el nombre completo.
   - Si el nombre tiene dos palabras (ej. nombre y apellido), devuelve la primera letra del nombre seguida de un punto y luego el apellido (ej. "Lionel Messi" se convierte en "L. Messi").
   - Si el nombre tiene más de dos palabras (ej. varios nombres), devuelve las iniciales de los dos primeros nombres seguidas del apellido completo (ej. "Luis Alberto Suárez Díaz" se convierte en "L. A. Suárez Díaz").

3. **Aplicación en el `DataFrame`:**
   - La función `get_short_name()` se aplica a la columna `'name'` de `df` para crear una nueva columna `'shortName'` que contiene las versiones abreviadas de los nombres de los jugadores.

4. **Tipo de datos y eliminación de columnas:**
   - Se asegura que la columna `'qualifiers'` sea de tipo cadena (`str`).
   - Finalmente, se eliminan las columnas `'Unnamed: 0'` y `'id'`, que parecen ser irrelevantes para el análisis, utilizando `df.drop()`.

In [ ]:
# Function to extract short names
def get_short_name(full_name):
    if pd.isna(full_name):
        return full_name
    parts = full_name.split()
    if len(parts) == 1:
        return full_name  # No need for short name if there's only one word
    elif len(parts) == 2:
        return parts[0][0] + ". " + parts[1]
    else:
        return parts[0][0] + ". " + parts[1][0] + ". " + " ".join(parts[2:])

# Applying the function to create 'shortName' column
df['shortName'] = df['name'].apply(get_short_name)

df['qualifiers'] = df['qualifiers'].astype(str)
columns_to_drop2 = ['Unnamed: 0', 'id']
df.drop(columns=columns_to_drop2, inplace=True)

### Resumen:
La función `get_possession_chains()` toma una lista de eventos de un partido de fútbol y determina las cadenas de posesión, asignando un identificador de posesión único y un equipo a cada cadena. Esto es útil para analizar cómo un equipo mantiene el balón, cuánto dura cada posesión, y en qué momento del partido se interrumpen las posesiones.

### Propósito:
Esta función identifica las "cadenas de posesión" en los eventos de un partido de fútbol. Una cadena de posesión representa la secuencia de eventos en los que un equipo mantiene la posesión del balón, interrumpida por pérdidas de balón, goles, o cambios de período (como medio tiempo o tiempo extra).

### Parámetros de entrada:
1. **`events_df`**: El `DataFrame` que contiene los eventos de un partido, donde cada fila es un evento y contiene información como el tipo de evento, el equipo, el jugador, el tiempo, etc.
2. **`chain_check`**: Número de eventos consecutivos que se deben verificar para ver si forman parte de la misma cadena de posesión.
3. **`suc_evts_in_chain`**: Número mínimo de eventos exitosos consecutivos que un equipo debe tener para formar una cadena de posesión válida.

### Flujo de la función:

1. **Inicialización y preparación de datos:**
   - Se crea un `DataFrame` vacío `events_out` para almacenar el resultado.
   - Se filtran los tipos de eventos que no contribuyen a la posesión (como tarjetas, sustituciones, goles, etc.) y se inicializan indicadores temporales, como:
     - **`outcomeBinary`**: Indica si el evento fue exitoso o no.
     - **`teamBinary`**: Representa el equipo del evento en un formato binario.
     - **`goalBinary`**: Detecta si ocurrió un gol.

2. **Identificación de secuencias de eventos en la misma posesión:**
   - Se crean varias columnas en `pos_chain_df` para verificar si los eventos consecutivos pertenecen al mismo equipo. Si el equipo cambia antes de que se completen los eventos requeridos en la cadena, la cadena de posesión se interrumpe.
   - También se verifica si hay cambios de período o goles que marcan el fin de una posesión.

3. **Determinación de inicios de posesión válidos:**
   - Se marca el inicio de una posesión cuando hay suficientes eventos consecutivos del mismo equipo, no interrumpidos por goles o cambios de período.
   - Los inicios de posesión también se pueden generar manualmente en el primer evento del partido o después de un gol o cambio de período.

4. **Asignación de identificadores de posesión:**
   - A cada posesión válida se le asigna un identificador único (`possession_id`) y se asigna el equipo que tenía la posesión (`possession_team`).
   - Si un equipo mantiene la posesión durante eventos consecutivos, se le asigna el mismo identificador de posesión.

5. **Fusión con los eventos originales:**
   - Los identificadores de posesión y los nombres de los equipos se fusionan de nuevo en el `DataFrame` original para etiquetar cada evento con la cadena de posesión correspondiente.
   - Los valores faltantes de `possession_id` y `possession_team` se rellenan hacia adelante y hacia atrás, asegurando que todos los eventos tengan una posesión asignada.

6. **Salida:**
   - La función devuelve un `DataFrame` que contiene los eventos originales del partido, pero ahora con columnas adicionales para el identificador de la posesión y el equipo que tenía la posesión del balón en ese momento.



In [ ]:
def get_possession_chains(events_df, chain_check, suc_evts_in_chain):
    # Initialise output
    events_out = pd.DataFrame()
    match_events_df = df.reset_index()

    # Isolate valid event types that contribute to possession
    match_pos_events_df = match_events_df[~match_events_df['type'].isin(['OffsideGiven', 'CornerAwarded','Start', 'Card', 'SubstitutionOff',
                                                                                  'SubstitutionOn', 'FormationChange','FormationSet', 'End'])].copy()

    # Add temporary binary outcome and team identifiers
    match_pos_events_df['outcomeBinary'] = (match_pos_events_df['outcomeType']
                                                .apply(lambda x: 1 if x == 'Successful' else 0))
    match_pos_events_df['teamBinary'] = (match_pos_events_df['teamName']
                         .apply(lambda x: 1 if x == min(match_pos_events_df['teamName']) else 0))
    match_pos_events_df['goalBinary'] = ((match_pos_events_df['type'] == 'Goal')
                         .astype(int).diff(periods=1).apply(lambda x: 1 if x < 0 else 0))

    # Create a dataframe to investigate possessions chains
    pos_chain_df = pd.DataFrame()

    # Check whether each event is completed by same team as the next (check_evts-1) events
    for n in np.arange(1, chain_check):
        pos_chain_df[f'evt_{n}_same_team'] = abs(match_pos_events_df['teamBinary'].diff(periods=-n))
        pos_chain_df[f'evt_{n}_same_team'] = pos_chain_df[f'evt_{n}_same_team'].apply(lambda x: 1 if x > 1 else x)
    pos_chain_df['enough_evt_same_team'] = pos_chain_df.sum(axis=1).apply(lambda x: 1 if x < chain_check - suc_evts_in_chain else 0)
    pos_chain_df['enough_evt_same_team'] = pos_chain_df['enough_evt_same_team'].diff(periods=1)
    pos_chain_df[pos_chain_df['enough_evt_same_team'] < 0] = 0

    match_pos_events_df['period'] = pd.to_numeric(match_pos_events_df['period'], errors='coerce')
    # Check there are no kick-offs in the upcoming (check_evts-1) events
    pos_chain_df['upcoming_ko'] = 0
    for ko in match_pos_events_df[(match_pos_events_df['goalBinary'] == 1) | (match_pos_events_df['period'].diff(periods=1))].index.values:
        ko_pos = match_pos_events_df.index.to_list().index(ko)
        pos_chain_df.iloc[ko_pos - suc_evts_in_chain:ko_pos, 5] = 1

    # Determine valid possession starts based on event team and upcoming kick-offs
    pos_chain_df['valid_pos_start'] = (pos_chain_df.fillna(0)['enough_evt_same_team'] - pos_chain_df.fillna(0)['upcoming_ko'])

    # Add in possession starts due to kick-offs (period changes and goals).
    pos_chain_df['kick_off_period_change'] = match_pos_events_df['period'].diff(periods=1)
    pos_chain_df['kick_off_goal'] = ((match_pos_events_df['type'] == 'Goal')
                     .astype(int).diff(periods=1).apply(lambda x: 1 if x < 0 else 0))
    pos_chain_df.loc[pos_chain_df['kick_off_period_change'] == 1, 'valid_pos_start'] = 1
    pos_chain_df.loc[pos_chain_df['kick_off_goal'] == 1, 'valid_pos_start'] = 1

    # Add first possession manually
    pos_chain_df['teamName'] = match_pos_events_df['teamName']
    pos_chain_df.loc[pos_chain_df.head(1).index, 'valid_pos_start'] = 1
    pos_chain_df.loc[pos_chain_df.head(1).index, 'possession_id'] = 1
    pos_chain_df.loc[pos_chain_df.head(1).index, 'possession_team'] = pos_chain_df.loc[pos_chain_df.head(1).index, 'teamName']

    # Iterate through valid possession starts and assign them possession ids
    valid_pos_start_id = pos_chain_df[pos_chain_df['valid_pos_start'] > 0].index

    possession_id = 2
    for idx in np.arange(1, len(valid_pos_start_id)):
        current_team = pos_chain_df.loc[valid_pos_start_id[idx], 'teamName']
        previous_team = pos_chain_df.loc[valid_pos_start_id[idx - 1], 'teamName']
        if ((previous_team == current_team) & (pos_chain_df.loc[valid_pos_start_id[idx], 'kick_off_goal'] != 1) &
                (pos_chain_df.loc[valid_pos_start_id[idx], 'kick_off_period_change'] != 1)):
            pos_chain_df.loc[valid_pos_start_id[idx], 'possession_id'] = np.nan
        else:
            pos_chain_df.loc[valid_pos_start_id[idx], 'possession_id'] = possession_id
            pos_chain_df.loc[valid_pos_start_id[idx], 'possession_team'] = pos_chain_df.loc[valid_pos_start_id[idx], 'teamName']
            possession_id += 1

    # Assign possession id and team back to events dataframe
    match_events_df = pd.merge(match_events_df, pos_chain_df[['possession_id', 'possession_team']], how='left', left_index=True, right_index=True)

    # Fill in possession ids and possession team
    match_events_df[['possession_id', 'possession_team']] = (match_events_df[['possession_id', 'possession_team']].fillna(method='ffill'))
    match_events_df[['possession_id', 'possession_team']] = (match_events_df[['possession_id', 'possession_team']].fillna(method='bfill'))

    # Rebuild events dataframe
    events_out = pd.concat([events_out, match_events_df])

    return events_out

In [ ]:
df = get_possession_chains(df, 5, 3)

### Resumen:
Este código prepara el `DataFrame` final para su análisis, reemplazando los valores de los periodos del partido con sus descripciones textuales, eliminando los eventos que ocurrieron durante la tanda de penales, y extrayendo los nombres y IDs de los equipos local y visitante.

1. **Reemplazo de los valores de la columna `period`:**
   - La columna `period` en el `DataFrame` `df` contiene números que representan los diferentes periodos de un partido. Esta línea de código reemplaza esos valores numéricos por sus descripciones correspondientes:
     - `1` se reemplaza por `'FirstHalf'` (primer tiempo),
     - `2` por `'SecondHalf'` (segundo tiempo),
     - `3` por `'FirstPeriodOfExtraTime'` (primer período de tiempo extra),
     - `4` por `'SecondPeriodOfExtraTime'` (segundo período de tiempo extra),
     - `5` por `'PenaltyShootout'` (penales),
     - `14` por `'PostGame'` (post-partido),
     - `16` por `'PreMatch'` (pre-partido).

2. **Filtrado del `DataFrame` para excluir los penales:**
   - `df = df[df['period'] != 'PenaltyShootout']`: Se eliminan los eventos que ocurrieron durante la tanda de penales, manteniendo solo los eventos de las fases regulares y los tiempos extra del partido.
   - Luego, se reinicia el índice del `DataFrame` con `reset_index(drop=True)` para asegurar que los índices sean consecutivos tras el filtrado.

3. **Asignación de equipos locales y visitantes:**
   - Se selecciona el ID del equipo local (`hteamID`) y el equipo visitante (`ateamID`) utilizando el diccionario `teams_dict`, que mapea los IDs de los equipos a sus nombres.
     - `hteamID` toma el primer valor del diccionario (el equipo local).
     - `ateamID` toma el segundo valor del diccionario (el equipo visitante).
   - A continuación, se obtienen los nombres de ambos equipos usando estos IDs:
     - `hteamName` es el nombre del equipo local.
     - `ateamName` es el nombre del equipo visitante.

In [ ]:
df['period'] = df['period'].replace({1: 'FirstHalf', 2: 'SecondHalf', 3: 'FirstPeriodOfExtraTime', 4: 'SecondPeriodOfExtraTime', 
                                     5: 'PenaltyShootout', 14: 'PostGame', 16: 'PreMatch'})

# final df is ready here
df = df[df['period']!='PenaltyShootout']
df = df.reset_index(drop=True)

hteamID = list(teams_dict.keys())[0]  # selected home team
ateamID = list(teams_dict.keys())[1]  # selected away team
hteamName= teams_dict[hteamID]
ateamName= teams_dict[ateamID]

### Resumen:
Este código extrae los datos de tiros de un partido de Fotmob usando una API, luego los transforma en un `DataFrame` de Pandas. Además, combina esos datos con un archivo CSV que mapea los `teamId` de Fotmob a los nombres de los equipos, lo que facilita el análisis posterior de los datos de tiros con información adicional sobre los equipos.

1. **Función `scrape_shots(mi)`:**
   - **Propósito:** Extraer datos sobre los tiros de un partido específico en la plataforma **Fotmob**.
   - **Parámetros:** 
     - `mi`: Es el ID del partido (match ID) que se pasa a la API de Fotmob para obtener los detalles del partido.
   - **Proceso:**
     - Se utiliza la librería `requests` para hacer una solicitud a la API de **Fotmob** (`https://www.fotmob.com/api/matchDetails`) con el `matchId` como parámetro.
     - La respuesta se convierte en un formato JSON y se extrae la información del mapa de tiros (`shotmap`) bajo la clave `'shots'`.
     - La lista de tiros se convierte en un `DataFrame` de pandas.
     - Se agrega una columna `matchId` al `DataFrame` para identificar a qué partido pertenece cada tiro.
   - **Salida:** La función devuelve un `DataFrame` que contiene todos los tiros del partido especificado.

2. **Carga del archivo CSV de equipos:**
   - Se carga un archivo CSV llamado `"teams_name_and_id.csv"`, que contiene los nombres de los equipos y sus correspondientes `teamId` de Fotmob. Este archivo se lee usando `pd.read_csv()`.
   - El archivo CSV sirve como un mapeo entre los IDs de equipos de Fotmob y sus nombres, para facilitar la identificación de equipos en los datos de los tiros.

3. **Proceso de combinación de datos:**
   - Se obtiene el `DataFrame` de tiros llamando a la función `scrape_shots(fotmob_matchId)` con el `matchId` correspondiente.
   - Luego, se realiza una fusión (`merge`) entre los datos de los tiros y los nombres de los equipos. La combinación se basa en la columna `teamId` que está presente en ambos `DataFrames` (`shots_df` y `df_teamNameId`).
   - El `how='left'` asegura que se mantengan todas las filas del `DataFrame` de tiros, incluso si algunos `teamId` no tienen coincidencia en el archivo de equipos.



In [ ]:
fotmob_matchId

In [ ]:
# scrapear los datos de disparos de fotmob
def scrape_shots(mi):
    # params = {'matchId': mi}
    header = {'user-agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Mobile Safari/537.36',
              'sec-fetch-site': 'same-origin',
              'sec-fetch-mode': 'cors',
              'sec-fetch-dest': 'empty',
              'sec-ch-ua-platform': '"Android"',
              'sec-ch-ua-mobile': '?1',
              'sec-ch-ua': '"Chromium";v="130", "Google Chrome";v="130", "Not?A_Brand";v="99"',
              'pragma': 'no-cache',
              'x-fm-req': 'eyJib2R5Ijp7InVybCI6Ii9hcGkvbWF0Y2hEZXRhaWxzP21hdGNoSWQ9NDUwNjI3MSIsImNvZGUiOjE3MzA1ODYyODYzMjN9LCJzaWduYXR1cmUiOiI5NUQwREFGNUQ0QjdEMjk3MEIyRUQ3MkQ3NTFENTlBOCJ9'}
    response = requests.get(f'https://www.fotmob.com/api/matchDetails?matchId={mi}', headers=header)
    # print(response)
    data = response.json()
    shotmap = data['content']['shotmap']['shots']
    shots_df = pd.DataFrame(shotmap)
    shots_df['matchId'] = mi

    return shots_df

# el archivo CSV "teams_name_and_id.csv" contiene la mayoría de los teamId de Fotmob de los equipos.
df_teamNameId = pd.read_csv(f"FData/club_teams_logo/teams_name_and_id.csv")

shots_df = scrape_shots(fotmob_matchId)
shots_df = shots_df.merge(df_teamNameId[['teamId', 'teamName']], on='teamId', how='left')


In [ ]:
df_teamNameId.head()

### Resumen:
Este código procesa los datos de eventos y tiros de un partido de fútbol, incluyendo la identificación del equipo contrario, la normalización de los nombres de jugadores, y el cálculo de estadísticas como goles, xT (expected threat), xG (expected goals) y xGOT (expected goals on target). Finalmente, guarda los resultados en archivos CSV para su posterior análisis.

1. **Función para obtener el equipo contrario:**
   - La función `get_opposite_teamName(team)` toma el nombre de un equipo como argumento y devuelve el nombre del equipo contrario. Si el equipo es el local (`hteamName`), devuelve el visitante (`ateamName`) y viceversa.
   - La función se aplica a la columna `teamName` del `DataFrame` `shots_df` para crear una nueva columna `oppositeTeam`.

2. **Normalización de nombres de jugadores:**
   - Se asegura que los nombres de los jugadores en la columna `playerName` sean cadenas de texto y luego se utiliza la función `unidecode` para convertir cualquier carácter no ASCII en su equivalente en inglés (por ejemplo, convertir letras con acentos).

3. **Filtrado de datos:**
   - Se eliminan las filas correspondientes a la tanda de penales filtrando los eventos donde la columna `period` no sea igual a `'PenaltyShootout'`.

4. **Asignación de colores para los equipos:**
   - `hcol` y `acol` representan los colores del equipo local y visitante, respectivamente, aunque en este fragmento no se especifican los valores de `col1` y `col2`.

5. **Cálculo del total de xT (expected threat):**
   - Se filtran los datos para obtener dos subconjuntos, uno para el equipo local (`homedf`) y otro para el equipo visitante (`awaydf`).
   - Se suma el valor de `xT` para ambos equipos y se redondea a dos decimales.

6. **Cálculo de la cantidad de goles:**
   - Se cuentan los goles de ambos equipos excluyendo los autogoles (`OwnGoal`). Luego se suman los autogoles en la cuenta del equipo contrario.
   - Esto asegura que los goles marcados y los autogoles se contabilicen correctamente para ambos equipos.

7. **Cálculo del total de expected goals (xG) y expected goals on target (xGOT):**
   - Se filtran los datos de tiros (`shots_df`) para obtener los datos del equipo local y visitante.
   - Se calcula el total de `xG` y `xGOT` para ambos equipos y se redondea a dos decimales.

8. **Guardado de los DataFrames:**
   - Se guarda el `DataFrame` de eventos y el `DataFrame` de tiros en archivos CSV con nombres que incluyen el ID del partido de Fotmob y el nombre de los equipos. Los archivos se guardan en la carpeta `"FData"`.



In [ ]:

# Adding new column for opposite team name
def get_opposite_teamName(team):
    if team == hteamName:
        return ateamName
    elif team == ateamName:
        return hteamName
    else:
        return None
# Apply the function to create the new column
shots_df['oppositeTeam'] = shots_df['teamName'].apply(get_opposite_teamName)
shots_df['playerName'] = shots_df['playerName'].astype(str)
shots_df['playerName'] = shots_df['playerName'].apply(unidecode)
shots_df = shots_df[shots_df['period']!='PenaltyShootout']

# Assigning the home and away team's color
hcol= col1 
acol= col2

homedf = df[(df['teamName']==hteamName)]
awaydf = df[(df['teamName']==ateamName)]
hxT = homedf['xT'].sum().round(2)
axT = awaydf['xT'].sum().round(2)

# Getting the Score of the match
hgoal_count = len(homedf[(homedf['teamName']==hteamName) & (homedf['type']=='Goal') & (~homedf['qualifiers'].str.contains('OwnGoal'))])
agoal_count = len(awaydf[(awaydf['teamName']==ateamName) & (awaydf['type']=='Goal') & (~awaydf['qualifiers'].str.contains('OwnGoal'))])
hgoal_count = hgoal_count + len(awaydf[(awaydf['teamName']==ateamName) & (awaydf['type']=='Goal') & (awaydf['qualifiers'].str.contains('OwnGoal'))])
agoal_count = agoal_count + len(homedf[(homedf['teamName']==hteamName) & (homedf['type']=='Goal') & (homedf['qualifiers'].str.contains('OwnGoal'))])

hshots_xgdf = shots_df[shots_df['teamName']==hteamName]
ashots_xgdf = shots_df[shots_df['teamName']==ateamName]
hxg = round(hshots_xgdf['expectedGoals'].sum(), 2)
axg = round(ashots_xgdf['expectedGoals'].sum(), 2)
hxgot = round(hshots_xgdf['expectedGoalsOnTarget'].sum(), 2)
axgot = round(ashots_xgdf['expectedGoalsOnTarget'].sum(), 2)

# Saving the dfs for future use, use your directory file path to save there
file_header = f'{hteamName}_vs_{ateamName}'

df.to_csv(os.path.join("FData", f"{fotmob_matchId}_EventsData_fotmob.csv"), index=False)
shots_df.to_csv(os.path.join("FData", f"{fotmob_matchId}_ShotsData_fotmob.csv"), index=False)

# FUNCIONES DEL REPORTE DE PARTIDO

# Red de pases (Pass Network)

### Resumen:
La función `get_passes_df()` filtra los eventos para obtener solo los pases y añade una columna para identificar al receptor de cada pase. Luego, se devuelve un `DataFrame` con toda la información relevante sobre los pases, como las coordenadas, el equipo, los jugadores involucrados, el tipo de pase, el resultado, y el ángulo del pase. La parte visual con `path_eff` probablemente se utilizará más adelante para representar los pases en un gráfico o visualización.

### Función `get_passes_df(df)`:

1. **Propósito:**
   - Esta función filtra los eventos del `DataFrame` para obtener únicamente los eventos relacionados con pases, eliminando otros tipos de eventos que no son relevantes (como sustituciones, cambios de formación, tarjetas, etc.). También añade una columna para identificar al jugador que recibe el pase.

2. **Filtrado de eventos irrelevantes:**
   - `df1 = df[~df['type'].str.contains('SubstitutionOn|FormationChange|FormationSet|Card')]` filtra el `DataFrame` para eliminar eventos que no son de interés (sustituciones, cambios de formación, y tarjetas).
   - El resultado filtrado se asigna de nuevo a `df`.

3. **Identificación del receptor del pase:**
   - `df.loc[:, "receiver"] = df["playerId"].shift(-1)` crea una nueva columna `receiver`, que asigna el `playerId` del siguiente evento como el receptor del pase. Esto supone que el siguiente evento en el `DataFrame` es un pase al siguiente jugador.

4. **Filtrado de los eventos de pases:**
   - `passes_ids = df.index[df['type'] == 'Pass']` obtiene los índices de los eventos donde el tipo de evento es un pase (`Pass`).
   - Luego, se filtran los datos correspondientes a esos índices para crear `df_passes`, que contiene información relevante para los pases, como las coordenadas iniciales y finales del pase (`x`, `y`, `endX`, `endY`), el equipo, el jugador que realiza el pase (`playerId`), el receptor, el tipo de evento y el resultado del pase (`outcomeType`), además del ángulo del pase (`pass_or_carry_angle`).

5. **Retorno:**
   - La función devuelve un `DataFrame` llamado `df_passes`, que contiene solo los eventos de pases con información adicional como el receptor del pase.

### Código adicional:

- **Variable `passes_df`:** 
  - Se llama a la función `get_passes_df(df)` para obtener el `DataFrame` de pases y almacenarlo en `passes_df`.

- **Variable `path_eff`:**
  - Esta línea parece estar configurando algunos efectos visuales, posiblemente para una futura representación gráfica. El objeto `path_effects.Stroke` parece aplicar un trazo con un ancho de línea de 3, y un color de fondo (`bg_color`), seguido por un efecto normal de `path_effects.Normal()`. Estos efectos probablemente se usarán para trazar las líneas de los pases con estos estilos.



In [ ]:
def get_passes_df(df):
    df1 = df[~df['type'].str.contains('SubstitutionOn|FormationChange|FormationSet|Card')]
    df = df1
    df.loc[:, "receiver"] = df["playerId"].shift(-1)
    passes_ids = df.index[df['type'] == 'Pass']
    df_passes = df.loc[passes_ids, ["index", "x", "y", "endX", "endY", "teamName", "playerId", "receiver", "type", "outcomeType", "pass_or_carry_angle"]]

    return df_passes

passes_df = get_passes_df(df)
path_eff = [path_effects.Stroke(linewidth=3, foreground=bg_color), path_effects.Normal()]

### Resumen:
La función `get_passes_between_df()` extrae los pases realizados entre jugadores de un equipo y calcula las posiciones promedio de los jugadores, así como el número de pases entre cada pareja. Esta información es útil para crear una visualización de la red de pases, mostrando las líneas que conectan a los jugadores que han intercambiado pases, así como las posiciones promedio de estos jugadores en el campo.

### Función `get_passes_between_df(teamName, passes_df, players_df)`:

1. **Propósito:**
   - Esta función analiza los pases entre jugadores de un equipo específico (`teamName`). Calcula las posiciones medias donde los jugadores realizan sus pases y el número de pases entre cada pareja de jugadores. El objetivo es crear un `DataFrame` con la información necesaria para trazar las líneas de los pases entre jugadores en una visualización de red de pases.

2. **Filtrado de pases por equipo:**
   - `passes_df = passes_df[(passes_df["teamName"] == teamName)]`: Filtra el `DataFrame` de pases para obtener solo los eventos de pases del equipo indicado (`teamName`).

3. **Filtrado de eventos relevantes para el equipo:**
   - `dfteam = df[(df['teamName'] == teamName) & (~df['type'].str.contains('SubstitutionOn|FormationChange|FormationSet|Card'))]`: Filtra los eventos del equipo, eliminando aquellos que no son relevantes para el análisis de pases, como sustituciones, cambios de formación y tarjetas.

4. **Unión con la alineación inicial:**
   - `passes_df = passes_df.merge(players_df[["playerId", "isFirstEleven"]], on='playerId', how='left')`: Se une el `DataFrame` de pases con el `DataFrame` de jugadores para agregar la columna `isFirstEleven`, que indica si el jugador estaba en el once inicial.

5. **Cálculo de posiciones promedio de los pases:**
   - `average_locs_and_count_df`: Agrupa los datos de los jugadores del equipo para calcular la posición promedio de los pases (medianas de las coordenadas `x` e `y`) y el número total de pases realizados por cada jugador.
   - Luego, se unen las columnas `name`, `shirtNo`, `position` e `isFirstEleven` desde el `DataFrame` de jugadores para completar la información de cada jugador.

6. **Cálculo del número de pases entre cada pareja de jugadores:**
   - `passes_player_ids_df`: Se crea un `DataFrame` que contiene los IDs de los jugadores que realizan el pase y los que lo reciben.
   - Se añaden las columnas `pos_min` y `pos_max` para identificar cada par de jugadores que participan en un pase, independientemente de quién inició el pase o lo recibió (para contar los pases en ambas direcciones).
   - `passes_between_df`: Se agrupan los pases por cada pareja de jugadores y se cuenta cuántos pases han ocurrido entre ellos. Esto se almacena en la columna `pass_count`.

7. **Unión con las posiciones promedio de los jugadores:**
   - `passes_between_df`: Se realiza una unión con las posiciones promedio (`average_locs_and_count_df`) para obtener las coordenadas iniciales y finales de los jugadores involucrados en los pases, lo que permitirá visualizar las líneas de los pases en una gráfica.

8. **Retorno:**
   - La función devuelve dos `DataFrames`:
     - **`passes_between_df`**: Contiene los pases entre jugadores, junto con las posiciones iniciales y finales para poder dibujar las líneas que representan los pases.
     - **`average_locs_and_count_df`**: Contiene las posiciones promedio de los jugadores y el número de pases que cada uno realizó, útil para mostrar los puntos en una visualización.

### Código adicional:

1. **Aplicación de la función:**
   - **Para el equipo local (`home_passes_between_df` y `home_average_locs_and_count_df`):**
     - Se llama a `get_passes_between_df(hteamName, passes_df, players_df)` para calcular los pases entre los jugadores del equipo local y obtener sus posiciones promedio.
   
   - **Para el equipo visitante (`away_passes_between_df` y `away_average_locs_and_count_df`):**
     - Se llama a `get_passes_between_df(ateamName, passes_df, players_df)` para hacer lo mismo con el equipo visitante.



In [ ]:
def get_passes_between_df(teamName, passes_df, players_df):
    passes_df = passes_df[(passes_df["teamName"] == teamName)]
    # df = pd.DataFrame(events_dict)
    dfteam = df[(df['teamName'] == teamName) & (~df['type'].str.contains('SubstitutionOn|FormationChange|FormationSet|Card'))]
    passes_df = passes_df.merge(players_df[["playerId", "isFirstEleven"]], on='playerId', how='left')
    # calculate median positions for player's passes
    average_locs_and_count_df = (dfteam.groupby('playerId').agg({'x': ['median'], 'y': ['median', 'count']}))
    average_locs_and_count_df.columns = ['pass_avg_x', 'pass_avg_y', 'count']
    average_locs_and_count_df = average_locs_and_count_df.merge(players_df[['playerId', 'name', 'shirtNo', 'position', 'isFirstEleven']], on='playerId', how='left')
    average_locs_and_count_df = average_locs_and_count_df.set_index('playerId')
    average_locs_and_count_df['name'] = average_locs_and_count_df['name'].apply(unidecode)
    # calculate the number of passes between each position (using min/ max so we get passes both ways)
    passes_player_ids_df = passes_df.loc[:, ['index', 'playerId', 'receiver', 'teamName']]
    passes_player_ids_df['pos_max'] = (passes_player_ids_df[['playerId', 'receiver']].max(axis='columns'))
    passes_player_ids_df['pos_min'] = (passes_player_ids_df[['playerId', 'receiver']].min(axis='columns'))
    # get passes between each player
    passes_between_df = passes_player_ids_df.groupby(['pos_min', 'pos_max']).index.count().reset_index()
    passes_between_df.rename({'index': 'pass_count'}, axis='columns', inplace=True)
    # add on the location of each player so we have the start and end positions of the lines
    passes_between_df = passes_between_df.merge(average_locs_and_count_df, left_on='pos_min', right_index=True)
    passes_between_df = passes_between_df.merge(average_locs_and_count_df, left_on='pos_max', right_index=True, suffixes=['', '_end'])

    return passes_between_df, average_locs_and_count_df

# home_team_id = list(teams_dict.keys())[0]
home_passes_between_df, home_average_locs_and_count_df = get_passes_between_df(hteamName, passes_df, players_df)
# away_team_id = list(teams_dict.keys())[1]
away_passes_between_df, away_average_locs_and_count_df = get_passes_between_df(ateamName, passes_df, players_df)


### Resumen:
Esta función dibuja una red de pases que muestra cómo los jugadores de un equipo interactúan entre sí mediante los pases. Los jugadores se representan como nodos en sus posiciones promedio, y las líneas muestran las conexiones entre ellos. La función también calcula estadísticas como la verticalidad del equipo, la altura de la línea defensiva, y la combinación de pases más frecuente, proporcionando información clave sobre el estilo de juego del equipo.

### Propósito:
La función `pass_network_visualization()` crea una visualización de la red de pases de un equipo en un gráfico de campo de fútbol utilizando la biblioteca **mplsoccer** (a través de la clase `Pitch`). La función también calcula estadísticas como la altura de la línea defensiva, la verticalidad del equipo, y la combinación de pases más frecuente entre dos jugadores. La visualización incluye los nodos que representan a los jugadores y las líneas que muestran la conexión entre ellos a través de los pases.

### Parámetros:
- **`ax`**: El eje en el que se va a dibujar el gráfico.
- **`passes_between_df`**: `DataFrame` que contiene los pases entre jugadores, con las coordenadas iniciales y finales de los pases y el conteo de pases entre cada par de jugadores.
- **`average_locs_and_count_df`**: `DataFrame` con las posiciones promedio de los jugadores y el número de pases realizados por cada uno.
- **`col`**: El color que se usará para las líneas y textos del equipo en el gráfico.
- **`teamName`**: El nombre del equipo (local o visitante) para el cual se dibuja la red de pases.
- **`flipped`**: Si es `True`, se invierte el campo (esto es útil para el equipo visitante).

### Funcionamiento de la función:

1. **Configuración inicial:**
   - Se establecen valores máximos para el ancho de las líneas (`MAX_LINE_WIDTH`) y el tamaño de los nodos (`MAX_MARKER_SIZE`), que representan los jugadores y la cantidad de pases.
   - Se ajusta la transparencia de las líneas en función de la cantidad de pases entre cada par de jugadores. Cuanto más pases haya, más opaca será la línea.

2. **Dibujo del campo de fútbol:**
   - Se utiliza la clase `Pitch` de **mplsoccer** para dibujar un campo de fútbol estilo UEFA.
   - Se establecen límites para las coordenadas del gráfico.

3. **Dibujar las líneas de pases:**
   - Se dibujan las líneas que conectan las posiciones de los jugadores (`pass_avg_x`, `pass_avg_y`) en base a la cantidad de pases entre ellos. Estas líneas se ajustan en grosor y color según el número de pases.

4. **Dibujar los nodos de los jugadores:**
   - Se dibujan círculos (`o`) para los jugadores que fueron titulares y cuadrados (`s`) para los suplentes, representando su posición promedio en el campo.
   - El color de fondo y los bordes de los nodos también se ajustan.

5. **Agregar los números de las camisetas:**
   - Se colocan los números de las camisetas de los jugadores en el centro de cada nodo.

6. **Líneas de referencia verticales:**
   - Se trazan líneas verticales para indicar la "altura" promedio de los pases. Esto incluye la **línea defensiva** (mediana de las posiciones de los defensores centrales) y la **línea de los delanteros** (mediana de las posiciones de los dos delanteros más avanzados).

7. **Sombreado de la zona media:**
   - Se colorea la zona entre la línea defensiva y la línea de los delanteros para resaltar el área de juego controlada por el equipo.

8. **Cálculo de la verticalidad:**
   - La verticalidad del equipo se calcula como la mediana del ángulo de los pases entre 0 y 90 grados. Cuanto más cercano a 0 es el ángulo, más directo es el pase hacia adelante. Se convierte en un porcentaje para mostrar cuán directo juega el equipo.

9. **Combinación de pases más frecuente:**
   - Se identifica la combinación de jugadores con el mayor número de pases entre ellos.

10. **Textos y etiquetas adicionales:**
    - Se añaden títulos, leyendas y textos que indican detalles como la verticalidad del equipo y la altura de la línea de pase promedio.
    - Si el equipo es el visitante, se invierte el gráfico (`ax.invert_xaxis()` y `ax.invert_yaxis()`).

### Retorno:
La función devuelve un diccionario con las siguientes estadísticas clave:
- **`Team_Name`**: Nombre del equipo.
- **`Defense_Line_Height`**: Altura de la línea defensiva (posición promedio de los defensores centrales).
- **`Verticality_%`**: Porcentaje que representa la verticalidad del equipo.
- **`Most_pass_combination_from`**: El jugador que inició la combinación de pases más frecuente.
- **`Most_pass_combination_to`**: El jugador que recibió la combinación de pases más frecuente.
- **`Most_passes_in_combination`**: El número de pases en la combinación más frecuente.



In [ ]:
def pass_network_visualization(ax, passes_between_df, average_locs_and_count_df, col, teamName, flipped=False):
    MAX_LINE_WIDTH = 15
    MAX_MARKER_SIZE = 3000
    passes_between_df['width'] = (passes_between_df.pass_count / passes_between_df.pass_count.max() *MAX_LINE_WIDTH)
    # average_locs_and_count_df['marker_size'] = (average_locs_and_count_df['count']/ average_locs_and_count_df['count'].max() * MAX_MARKER_SIZE) #You can plot variable size of each player's node according to their passing volume, in the plot using this
    MIN_TRANSPARENCY = 0.05
    MAX_TRANSPARENCY = 0.85
    color = np.array(to_rgba(col))
    color = np.tile(color, (len(passes_between_df), 1))
    c_transparency = passes_between_df.pass_count / passes_between_df.pass_count.max()
    c_transparency = (c_transparency * (MAX_TRANSPARENCY - MIN_TRANSPARENCY)) + MIN_TRANSPARENCY
    color[:, 3] = c_transparency

    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    # ax.set_ylim(-0.5, 68.5)

    # Plotting those lines between players
    pass_lines = pitch.lines(passes_between_df.pass_avg_x, passes_between_df.pass_avg_y, passes_between_df.pass_avg_x_end, passes_between_df.pass_avg_y_end,
                             lw=passes_between_df.width, color=color, zorder=1, ax=ax)

    # Plotting the player nodes
    for index, row in average_locs_and_count_df.iterrows():
      if row['isFirstEleven'] == True:
        pass_nodes = pitch.scatter(row['pass_avg_x'], row['pass_avg_y'], s=1000, marker='o', color=bg_color, edgecolor=line_color, linewidth=2, alpha=1, ax=ax)
      else:
        pass_nodes = pitch.scatter(row['pass_avg_x'], row['pass_avg_y'], s=1000, marker='s', color=bg_color, edgecolor=line_color, linewidth=2, alpha=0.75, ax=ax)

    # Plotting the shirt no. of each player
    for index, row in average_locs_and_count_df.iterrows():
        player_initials = row["shirtNo"]
        pitch.annotate(player_initials, xy=(row.pass_avg_x, row.pass_avg_y), c=col, ha='center', va='center', size=18, ax=ax)

    # Plotting a vertical line to show the median vertical position of all passes
    avgph = round(average_locs_and_count_df['pass_avg_x'].median(), 2)
    # avgph_show = round((avgph*1.05),2)
    avgph_show = avgph
    ax.axvline(x=avgph, color='gray', linestyle='--', alpha=0.75, linewidth=2)

    # Defense line Passing Height (avg. height of all the passes made by the Center Backs)
    center_backs_height = average_locs_and_count_df[average_locs_and_count_df['position']=='DC']
    def_line_h = round(center_backs_height['pass_avg_x'].median(), 2)
    ax.axvline(x=def_line_h, color='gray', linestyle='dotted', alpha=0.5, linewidth=2)
    
    # Forward line Passing Height (avg. height of all the passes made by the Top 2 avg positoned Forwards)
    Forwards_height = average_locs_and_count_df[average_locs_and_count_df['isFirstEleven']==1]
    Forwards_height = Forwards_height.sort_values(by='pass_avg_x', ascending=False)
    Forwards_height = Forwards_height.head(2)
    fwd_line_h = round(Forwards_height['pass_avg_x'].mean(), 2)
    ax.axvline(x=fwd_line_h, color='gray', linestyle='dotted', alpha=0.5, linewidth=2)
    
    # coloring the middle zone in the pitch
    ymid = [0, 0, 68, 68]
    xmid = [def_line_h, fwd_line_h, fwd_line_h, def_line_h]
    ax.fill(xmid, ymid, col, alpha=0.1)

    # Getting the verticality of a team, (Verticality means how straight forward a team passes while advancing the ball, more the value = more directness in forward passing)
    team_passes_df = passes_df[(passes_df["teamName"] == teamName)]
    team_passes_df['pass_or_carry_angle'] = team_passes_df['pass_or_carry_angle'].abs()
    team_passes_df = team_passes_df[(team_passes_df['pass_or_carry_angle']>=0) & (team_passes_df['pass_or_carry_angle']<=90)]
    med_ang = team_passes_df['pass_or_carry_angle'].median()
    verticality = round((1 - med_ang/90)*100, 2)

    # Getting the top passers combination
    passes_between_df = passes_between_df.sort_values(by='pass_count', ascending=False).head(1).reset_index(drop=True)
    most_pass_from = passes_between_df['name'][0]
    most_pass_to = passes_between_df['name_end'][0]
    most_pass_count = passes_between_df['pass_count'][0]
    
    # Heading and other texts
    if teamName == ateamName:
      # inverting the pitch for away team
      ax.invert_xaxis()
      ax.invert_yaxis()
      ax.text(avgph-1, 73, f"{avgph_show}m", fontsize=15, color=line_color, ha='left')
      ax.text(105, 73,f"verticality: {verticality}%", fontsize=15, color=line_color, ha='left')
    else:
      ax.text(avgph-1, -5, f"{avgph_show}m", fontsize=15, color=line_color, ha='right')
      ax.text(105, -5, f"verticality: {verticality}%", fontsize=15, color=line_color, ha='right')

    # Headlines and other texts
    if teamName == hteamName:
      ax.text(2,66, "circle = starter\nbox = sub", color=hcol, size=12, ha='left', va='top')
      ax.set_title(f"{hteamName}\nPassing Network", color=line_color, size=25, fontweight='bold')

    else:
      ax.text(2,2, "circle = starter\nbox = sub", color=acol, size=12, ha='right', va='top')
      ax.set_title(f"{ateamName}\nPassing Network", color=line_color, size=25, fontweight='bold')

    # returnig the stats for storing those 
    return {
        'Team_Name': teamName,
        'Defense_Line_Height': def_line_h,
        'Vericality_%': verticality,
        'Most_pass_combination_from': most_pass_from,
        'Most_pass_combination_to': most_pass_to,
        'Most_passes_in_combination': most_pass_count,
    } 

Este bloque de código crea una visualización de redes de pases para ambos equipos en un partido de fútbol y almacena estadísticas clave de estas redes en un DataFrame.

In [ ]:
fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
pass_network_stats_home = pass_network_visualization(axs[0], home_passes_between_df, home_average_locs_and_count_df, hcol, hteamName)
pass_network_stats_away = pass_network_visualization(axs[1], away_passes_between_df, away_average_locs_and_count_df, acol, ateamName)
pass_network_stats_list = []
pass_network_stats_list.append(pass_network_stats_home)
pass_network_stats_list.append(pass_network_stats_away)
pass_network_stats_df = pd.DataFrame(pass_network_stats_list)

In [ ]:
pass_network_stats_df.head()

# Bloque Defensivo (Defensive Block)

### Resumen:
La función `get_defensive_action_df()` filtra las acciones defensivas de un partido, como entradas, intercepciones, faltas, despejes, etc., creando un `DataFrame` que contiene únicamente estos eventos junto con información relevante, como el jugador, las coordenadas en el campo, y el resultado de la acción. Esto permite un análisis específico de las acciones defensivas de un equipo o jugador en particular.

### Propósito:
La función `get_defensive_action_df()` extrae y filtra las acciones defensivas de un partido de fútbol a partir de un conjunto de eventos almacenados en un `DataFrame` (presumiblemente en `df`). El objetivo es generar un `DataFrame` que contenga solo los eventos defensivos, para su posterior análisis o visualización.

### Detalles del código:

1. **Filtro de acciones defensivas:**
   - La función filtra los eventos defensivos basándose en los valores de la columna `type` y, en algunos casos, también en la columna `qualifiers`.
   - Los tipos de eventos considerados defensivos son:
     - **Aerial** (solo si incluye la etiqueta `'Defensive'` en `qualifiers`).
     - **BallRecovery** (recuperación del balón).
     - **BlockedPass** (pase bloqueado).
     - **Challenge** (desafío).
     - **Clearance** (despeje).
     - **Error** (error).
     - **Foul** (falta).
     - **Interception** (intercepción).
     - **Tackle** (entrada).
   - Se utiliza `df.index[]` para obtener los índices de las filas donde ocurren estas acciones defensivas. La expresión `&` se usa para combinar múltiples condiciones, como en el caso de los eventos de tipo `'Aerial'` que también contienen `'Defensive'` en la columna `qualifiers`.

2. **Crear el DataFrame de acciones defensivas:**
   - Se seleccionan las filas de `df` que contienen los eventos defensivos usando los índices filtrados.
   - Se seleccionan las siguientes columnas para formar el `DataFrame` de acciones defensivas (`df_defensive_actions`):
     - **index**: El índice original del evento.
     - **x** y **y**: Coordenadas en el campo donde ocurrió la acción defensiva.
     - **teamName**: Nombre del equipo que realizó la acción defensiva.
     - **playerId**: ID del jugador que realizó la acción defensiva.
     - **type**: Tipo de acción defensiva (por ejemplo, `Tackle`, `Foul`, etc.).
     - **outcomeType**: Resultado de la acción (si fue exitosa o no).

3. **Retorno:**
   - La función devuelve el `DataFrame` filtrado que contiene solo las acciones defensivas.

### Aplicación:
- **`defensive_actions_df = get_defensive_action_df(events_dict)`**: 
   - Se llama a la función para crear un `DataFrame` con las acciones defensivas del partido. El resultado se almacena en la variable `defensive_actions_df` para su posterior uso.



In [ ]:
def get_defensive_action_df(events_dict):
    # filter only defensive actions
    defensive_actions_ids = df.index[(df['type'] == 'Aerial') & (df['qualifiers'].str.contains('Defensive')) |
                                     (df['type'] == 'BallRecovery') |
                                     (df['type'] == 'BlockedPass') |
                                     (df['type'] == 'Challenge') |
                                     (df['type'] == 'Clearance') |
                                     (df['type'] == 'Error') |
                                     (df['type'] == 'Foul') |
                                     (df['type'] == 'Interception') |
                                     (df['type'] == 'Tackle')]
    df_defensive_actions = df.loc[defensive_actions_ids, ["index", "x", "y", "teamName", "playerId", "type", "outcomeType"]]

    return df_defensive_actions

defensive_actions_df = get_defensive_action_df(events_dict)

### Resumen:
Este código genera un `DataFrame` que muestra las posiciones promedio y el número de acciones defensivas realizadas por cada jugador de un equipo en un partido, excluyendo a los porteros. Estos datos son útiles para analizar cómo se posicionan y actúan defensivamente los jugadores en el campo.

### Función `get_da_count_df(team_name, defensive_actions_df, players_df)`:

1. **Propósito:**
   - Esta función calcula las posiciones promedio y el número de acciones defensivas realizadas por cada jugador de un equipo específico (`team_name`). También añade información como el nombre del jugador, el número de camiseta, la posición, y si el jugador es parte del once inicial (`isFirstEleven`).

2. **Filtrado de acciones defensivas por equipo:**
   - `defensive_actions_df = defensive_actions_df[defensive_actions_df["teamName"] == team_name]`:
     - Filtra el `DataFrame` de acciones defensivas para obtener únicamente los eventos realizados por el equipo especificado en `team_name`.

3. **Unión con el `DataFrame` de jugadores:**
   - `defensive_actions_df = defensive_actions_df.merge(players_df[["playerId", "isFirstEleven"]], on='playerId', how='left')`:
     - Se une el `DataFrame` de acciones defensivas con el `DataFrame` de jugadores (`players_df`) para agregar la información sobre si el jugador formaba parte del once inicial (`isFirstEleven`).

4. **Cálculo de las posiciones promedio y el número de acciones defensivas:**
   - `average_locs_and_count_df = (defensive_actions_df.groupby('playerId').agg({'x': ['median'], 'y': ['median', 'count']}))`:
     - Agrupa las acciones defensivas por el `playerId` y calcula las coordenadas promedio (`x` e `y`) y el número de acciones defensivas realizadas por cada jugador.
     - Renombra las columnas del `DataFrame` para mayor claridad.

5. **Unión con información adicional del jugador:**
   - `average_locs_and_count_df = average_locs_and_count_df.merge(players_df[['playerId', 'name', 'shirtNo', 'position', 'isFirstEleven']], on='playerId', how='left')`:
     - Se vuelve a unir este `DataFrame` con el de jugadores para agregar más información sobre cada jugador, como su nombre, número de camiseta, posición y si era parte del once inicial.
   - Luego, se establece el `playerId` como índice del `DataFrame`.

6. **Retorno:**
   - La función devuelve el `DataFrame` con las posiciones promedio (`x`, `y`), el número de acciones defensivas (`count`), y los datos adicionales del jugador.

### Aplicación:

1. **Calcular posiciones promedio y acciones defensivas del equipo local:**
   - `defensive_home_average_locs_and_count_df = get_da_count_df(hteamName, defensive_actions_df, players_df)`:
     - Se llama a la función para obtener el `DataFrame` de posiciones promedio y acciones defensivas de los jugadores del equipo local (`hteamName`).

2. **Calcular posiciones promedio y acciones defensivas del equipo visitante:**
   - `defensive_away_average_locs_and_count_df = get_da_count_df(ateamName, defensive_actions_df, players_df)`:
     - Se llama a la función para obtener el `DataFrame` de posiciones promedio y acciones defensivas de los jugadores del equipo visitante (`ateamName`).

3. **Excluir al portero (GK):**
   - `defensive_home_average_locs_and_count_df = defensive_home_average_locs_and_count_df[defensive_home_average_locs_and_count_df['position'] != 'GK']`:
     - Se eliminan los porteros (`GK`) del `DataFrame` de acciones defensivas del equipo local.
   - `defensive_away_average_locs_and_count_df = defensive_away_average_locs_and_count_df[defensive_away_average_locs_and_count_df['position'] != 'GK']`:
     - Se eliminan los porteros del `DataFrame` del equipo visitante.



In [ ]:
def get_da_count_df(team_name, defensive_actions_df, players_df):
    defensive_actions_df = defensive_actions_df[defensive_actions_df["teamName"] == team_name]
    # add column with first eleven players only
    defensive_actions_df = defensive_actions_df.merge(players_df[["playerId", "isFirstEleven"]], on='playerId', how='left')
    # calculate mean positions for players
    average_locs_and_count_df = (defensive_actions_df.groupby('playerId').agg({'x': ['median'], 'y': ['median', 'count']}))
    average_locs_and_count_df.columns = ['x', 'y', 'count']
    average_locs_and_count_df = average_locs_and_count_df.merge(players_df[['playerId', 'name', 'shirtNo', 'position', 'isFirstEleven']], on='playerId', how='left')
    average_locs_and_count_df = average_locs_and_count_df.set_index('playerId')

    return  average_locs_and_count_df

defensive_home_average_locs_and_count_df = get_da_count_df(hteamName, defensive_actions_df, players_df)
defensive_away_average_locs_and_count_df = get_da_count_df(ateamName, defensive_actions_df, players_df)
defensive_home_average_locs_and_count_df = defensive_home_average_locs_and_count_df[defensive_home_average_locs_and_count_df['position'] != 'GK']
defensive_away_average_locs_and_count_df = defensive_away_average_locs_and_count_df[defensive_away_average_locs_and_count_df['position'] != 'GK']

### Resumen:
Este código genera una visualización comparativa del bloque defensivo de ambos equipos en un partido, mostrando la distribución de las acciones defensivas en el campo y calculando métricas clave como la altura promedio de las acciones defensivas, la altura de presión de la línea de delanteros y la compacidad del equipo. Las visualizaciones son útiles para analizar cómo se posiciona y se organiza el equipo en defensa.

### Propósito de la Función `defensive_block()`:

La función `defensive_block()` crea una visualización del bloque defensivo de un equipo en un gráfico de un campo de fútbol utilizando la biblioteca **mplsoccer** (a través de la clase `Pitch`). Además, calcula varias métricas clave como la altura promedio de las acciones defensivas, la altura de presión de la línea de delanteros, y la compacidad del bloque defensivo.

### Parámetros:
- **`ax`**: El eje sobre el cual se dibujará el gráfico.
- **`average_locs_and_count_df`**: `DataFrame` que contiene las posiciones promedio y el número de acciones defensivas por jugador.
- **`team_name`**: El nombre del equipo para el cual se está creando la visualización (local o visitante).
- **`col`**: El color utilizado para la visualización del equipo.

### Descripción paso a paso:

1. **Dibujar el campo de fútbol:**
   - Se crea un campo de fútbol estilo UEFA usando la clase `Pitch`. El campo se dibuja en el eje `ax` con el color de fondo definido por `bg_color`.

2. **Tamaño de los nodos basado en el número de acciones defensivas:**
   - `average_locs_and_count_df['marker_size']` ajusta el tamaño de los nodos para cada jugador según el número de acciones defensivas que realizó. Los nodos más grandes representan jugadores más activos en defensa.

3. **Dibujar el mapa de calor de las acciones defensivas:**
   - Se genera un mapa de calor basado en las posiciones `x` e `y` de las acciones defensivas del equipo usando `kdeplot` de `mplsoccer`. El mapa utiliza una escala de color personalizada para mostrar la intensidad de las acciones defensivas en diferentes áreas del campo.

4. **Dibujar los nodos para los jugadores:**
   - Los jugadores titulares se representan con un círculo (`o`), mientras que los suplentes se muestran como un cuadrado (`s`). Estos nodos se colocan en la posición promedio de las acciones defensivas de cada jugador.

5. **Anotar los números de camiseta:**
   - Se anota el número de camiseta de cada jugador en el centro de su nodo para identificar a cada jugador en el gráfico.

6. **Líneas verticales para la altura de las acciones defensivas:**
   - Se dibuja una línea vertical en el gráfico que muestra la altura promedio de las acciones defensivas del equipo (conocida como **Defensive Actions Height** o **dah**).
   - Se dibujan líneas adicionales para la **línea defensiva** (altura de los defensores centrales) y la **línea de presión de los delanteros** (altura de los dos delanteros más avanzados).

7. **Compacidad del equipo:**
   - La **compacidad** del equipo se calcula como el porcentaje de reducción entre la línea de presión de los delanteros y la línea defensiva, indicando qué tan compacto está el equipo defensivamente.

8. **Textos y títulos:**
   - Se añaden textos para mostrar la altura promedio de las acciones defensivas y la compacidad del equipo.
   - Se añade un título que identifica al equipo (local o visitante) y se muestran leyendas que distinguen entre jugadores titulares y suplentes.

9. **Retorno:**
   - La función devuelve un diccionario con las siguientes estadísticas clave:
     - **`Team_Name`**: Nombre del equipo.
     - **`Average_Defensive_Action_Height`**: Altura promedio de las acciones defensivas.
     - **`Forward_Line_Pressing_Height`**: Altura de presión de la línea de delanteros.

### Código adicional:

1. **Generar visualización del bloque defensivo:**
   - Se crean dos subgráficos (`axs[0]` y `axs[1]`) para mostrar las visualizaciones del bloque defensivo del equipo local y visitante.
   - Se llama a `defensive_block()` para cada equipo, almacenando las estadísticas del bloque defensivo en `defensive_block_stats_home` y `defensive_block_stats_away`.

2. **Almacenar las estadísticas del bloque defensivo:**
   - Las estadísticas calculadas para ambos equipos se almacenan en una lista (`defensive_block_stats_list`) y luego se convierten en un `DataFrame` (`defensive_block_stats_df`).

In [ ]:
def defensive_block(ax, average_locs_and_count_df, team_name, col):
    defensive_actions_team_df = defensive_actions_df[defensive_actions_df["teamName"] == team_name]
    pitch = Pitch(pitch_type='uefa', pitch_color=bg_color, line_color=line_color, linewidth=2, line_zorder=2, corner_arcs=True)
    pitch.draw(ax=ax)
    ax.set_facecolor(bg_color)
    ax.set_xlim(-0.5, 105.5)
    # ax.set_ylim(-0.5, 68.5)

    # using variable marker size for each player according to their defensive engagements
    MAX_MARKER_SIZE = 3500
    average_locs_and_count_df['marker_size'] = (average_locs_and_count_df['count']/ average_locs_and_count_df['count'].max() * MAX_MARKER_SIZE)
    # plotting the heatmap of the team defensive actions
    color = np.array(to_rgba(col))
    flamingo_cmap = LinearSegmentedColormap.from_list("Flamingo - 100 colors", [bg_color, col], N=500)
    kde = pitch.kdeplot(defensive_actions_team_df.x, defensive_actions_team_df.y, ax=ax, fill=True, levels=5000, thresh=0.02, cut=4, cmap=flamingo_cmap)

    # using different node marker for starting and substitute players
    average_locs_and_count_df = average_locs_and_count_df.reset_index(drop=True)
    for index, row in average_locs_and_count_df.iterrows():
        if row['isFirstEleven'] == True:
            da_nodes = pitch.scatter(row['x'], row['y'], s=row['marker_size']+100, marker='o', color=bg_color, edgecolor=line_color, linewidth=1, 
                                 alpha=1, zorder=3, ax=ax)
        else:
            da_nodes = pitch.scatter(row['x'], row['y'], s=row['marker_size']+100, marker='s', color=bg_color, edgecolor=line_color, linewidth=1, 
                                     alpha=1, zorder=3, ax=ax)
    # plotting very tiny scatterings for the defensive actions
    da_scatter = pitch.scatter(defensive_actions_team_df.x, defensive_actions_team_df.y, s=10, marker='x', color='yellow', alpha=0.2, ax=ax)

    # Plotting the shirt no. of each player
    for index, row in average_locs_and_count_df.iterrows():
        player_initials = row["shirtNo"]
        pitch.annotate(player_initials, xy=(row.x, row.y), c=line_color, ha='center', va='center', size=(14), ax=ax)

    # Plotting a vertical line to show the median vertical position of all defensive actions, which is called Defensive Actions Height
    dah = round(average_locs_and_count_df['x'].mean(), 2)
    dah_show = round((dah*1.05), 2)
    ax.axvline(x=dah, color='gray', linestyle='--', alpha=0.75, linewidth=2)

    # Defense line Defensive Actions Height
    center_backs_height = average_locs_and_count_df[average_locs_and_count_df['position']=='DC']
    def_line_h = round(center_backs_height['x'].median(), 2)
    ax.axvline(x=def_line_h, color='gray', linestyle='dotted', alpha=0.5, linewidth=2)
    # Forward line Defensive Actions Height
    Forwards_height = average_locs_and_count_df[average_locs_and_count_df['isFirstEleven']==1]
    Forwards_height = Forwards_height.sort_values(by='x', ascending=False)
    Forwards_height = Forwards_height.head(2)
    fwd_line_h = round(Forwards_height['x'].mean(), 2)
    ax.axvline(x=fwd_line_h, color='gray', linestyle='dotted', alpha=0.5, linewidth=2)

    # Getting the compactness value 
    compactness = round((1 - ((fwd_line_h - def_line_h) / 105)) * 100, 2)

    # Headings and other texts
    if team_name == ateamName:
        # inverting the axis for away team
        ax.invert_xaxis()
        ax.invert_yaxis()
        ax.text(dah-1, 73, f"{dah_show}m", fontsize=15, color=line_color, ha='left', va='center')
    else:
        ax.text(dah-1, -5, f"{dah_show}m", fontsize=15, color=line_color, ha='right', va='center')

    # Headlines and other texts
    if team_name == hteamName:
        ax.text(105, -5, f'Compact:{compactness}%', fontsize=15, color=line_color, ha='right', va='center')
        ax.text(2,66, "circle = starter\nbox = sub", color='gray', size=12, ha='left', va='top')
        ax.set_title(f"{hteamName}\nDefensive Block", color=line_color, fontsize=25, fontweight='bold')
    else:
        ax.text(105, 73, f'Compact:{compactness}%', fontsize=15, color=line_color, ha='left', va='center')
        ax.text(2,2, "circle = starter\nbox = sub", color='gray', size=12, ha='right', va='top')
        ax.set_title(f"{ateamName}\nDefensive Block", color=line_color, fontsize=25, fontweight='bold')

    return {
        'Team_Name': team_name,
        'Average_Defensive_Action_Height': dah,
        'Forward_Line_Pressing_Height': fwd_line_h
    }

fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
defensive_block_stats_home = defensive_block(axs[0], defensive_home_average_locs_and_count_df, hteamName, hcol)
defensive_block_stats_away = defensive_block(axs[1], defensive_away_average_locs_and_count_df, ateamName, acol)
defensive_block_stats_list = []
defensive_block_stats_list.append(defensive_block_stats_home)
defensive_block_stats_list.append(defensive_block_stats_away)
defensive_block_stats_df = pd.DataFrame(defensive_block_stats_list)

In [ ]:
defensive_block_stats_df.head()

# Progresión de pases (Progressive Pass)

### Resumen:
La función `draw_progressive_pass_map()` genera un mapa que visualiza los pases progresivos de un equipo en un partido de fútbol. Se consideran solo aquellos pases que cumplen con condiciones específicas de progresividad y éxito. Además, la función divide el campo en tercios y calcula cuántos pases fueron realizados desde cada tercio, mostrando los datos de manera visual y clara. Al final, también se proporciona un conjunto de estadísticas clave sobre los pases progresivos del equipo.

### Propósito:
La función `draw_progressive_pass_map()` crea una visualización de los **pases progresivos** de un equipo en un campo de fútbol. Un pase progresivo, según la definición en esta función, es un pase que ha reducido la distancia hacia la portería contraria en al menos 10 yardas y no ha comenzado desde el tercio defensivo del equipo. La función también calcula cuántos pases progresivos se realizaron desde la izquierda, el centro y la derecha del campo, y visualiza estas métricas junto con los pases en el gráfico.

### Parámetros:
- **`ax`**: El eje en el que se va a dibujar el gráfico.
- **`team_name`**: Nombre del equipo (local o visitante) para el cual se va a generar el mapa de pases progresivos.
- **`col`**: El color que se utilizará para representar los pases progresivos del equipo en la visualización.

### Descripción paso a paso:

1. **Filtrado de pases progresivos:**
   - **`dfpro`** filtra los eventos de pase que cumplen con los siguientes criterios:
     - **Equipo**: El pase fue realizado por el equipo especificado (`team_name`).
     - **Progresividad**: El pase ha reducido la distancia a la portería contraria en al menos 10 yardas (9.11 metros).
     - **Condiciones adicionales**: No se incluye pases desde jugadas a balón parado como corners o tiros libres, y el pase no debe haber comenzado desde el tercio defensivo del equipo.
     - **Éxito**: El pase fue exitoso (`outcomeType == 'Successful'`).

2. **Dibujo del campo de fútbol:**
   - Se utiliza la clase `Pitch` de **mplsoccer** para dibujar el campo de fútbol estilo UEFA.

3. **Inversión del eje para el equipo visitante:**
   - Si el equipo es el visitante (`ateamName`), se invierten los ejes para que el equipo ataque de izquierda a derecha en el gráfico.

4. **Cálculo de los pases progresivos por tercio del campo:**
   - Se cuentan los pases progresivos realizados desde tres zonas del campo:
     - **Izquierda**: Pases hechos desde el tercio izquierdo del campo (`y >= 45.33`).
     - **Centro**: Pases hechos desde el tercio central del campo (`22.67 <= y < 45.33`).
     - **Derecha**: Pases hechos desde el tercio derecho del campo (`y < 22.67`).
   - Se calculan los porcentajes de los pases progresivos desde cada una de estas zonas.

5. **Añadir líneas horizontales:**
   - Se trazan líneas horizontales que dividen el campo en tercios, para separar las zonas izquierda, central y derecha.

6. **Mostrar las estadísticas en el gráfico:**
   - Se añaden textos en las zonas correspondientes que muestran el número de pases progresivos y su porcentaje en cada tercio del campo.

7. **Dibujo de los pases progresivos:**
   - Se trazan las líneas de los pases progresivos usando `pitch.lines()`, con un estilo de "cometa" que enfatiza la dirección del pase.
   - También se añaden pequeños marcadores (scatter) al final de cada pase para destacar el punto de finalización.

8. **Títulos y textos adicionales:**
   - Se muestra un título para el equipo, que incluye el número total de pases progresivos realizados.
   - Dependiendo de si el equipo es local o visitante, se ajusta el color y el estilo del texto.

9. **Retorno:**
   - La función devuelve un diccionario con las estadísticas clave:
     - **`Team_Name`**: Nombre del equipo.
     - **`Total_Progressive_Passes`**: Número total de pases progresivos realizados.
     - **`Progressive_Passes_From_Left`**: Número de pases progresivos desde la banda izquierda.
     - **`Progressive_Passes_From_Center`**: Número de pases progresivos desde el centro.
     - **`Progressive_Passes_From_Right`**: Número de pases progresivos desde la banda derecha.



In [ ]:
def draw_progressive_pass_map(ax, team_name, col):
    # filtering those passes which has reduced the distance form goal for at least 10yds and not started from defensive third, this is my condition for a progressive pass, which almost similar to opta/statsbomb conditon
    dfpro = df[(df['teamName']==team_name) & (df['prog_pass']>=9.11) & (~df['qualifiers'].str.contains('CornerTaken|Freekick')) & 
               (df['x']>=35) & (df['outcomeType']=='Successful')]
    pitch = Pitch(pitch_type='uefa', pitch_color=bg_color, line_color=line_color, linewidth=2, corner_arcs=True)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    # ax.set_ylim(-0.5, 68.5)

    if team_name == ateamName:
        ax.invert_xaxis()
        ax.invert_yaxis()

    pro_count = len(dfpro)

    # calculating the counts
    left_pro = len(dfpro[dfpro['y']>=45.33])
    mid_pro = len(dfpro[(dfpro['y']>=22.67) & (dfpro['y']<45.33)])
    right_pro = len(dfpro[(dfpro['y']>=0) & (dfpro['y']<22.67)])
    left_percentage = round((left_pro/pro_count)*100)
    mid_percentage = round((mid_pro/pro_count)*100)
    right_percentage = round((right_pro/pro_count)*100)

    ax.hlines(22.67, xmin=0, xmax=105, colors=line_color, linestyle='dashed', alpha=0.35)
    ax.hlines(45.33, xmin=0, xmax=105, colors=line_color, linestyle='dashed', alpha=0.35)

    # showing the texts in the pitch
    bbox_props = dict(boxstyle="round,pad=0.3", edgecolor="None", facecolor=bg_color, alpha=0.75)
    if col == hcol:
        ax.text(8, 11.335, f'{right_pro}\n({right_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 34, f'{mid_pro}\n({mid_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 56.675, f'{left_pro}\n({left_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
    else:
        ax.text(8, 11.335, f'{right_pro}\n({right_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 34, f'{mid_pro}\n({mid_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 56.675, f'{left_pro}\n({left_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)

    # plotting the passes
    pro_pass = pitch.lines(dfpro.x, dfpro.y, dfpro.endX, dfpro.endY, lw=3.5, comet=True, color=col, ax=ax, alpha=0.5)
    # plotting some scatters at the end of each pass
    pro_pass_end = pitch.scatter(dfpro.endX, dfpro.endY, s=35, edgecolor=col, linewidth=1, color=bg_color, zorder=2, ax=ax)

    counttext = f"{pro_count} Progressive Passes"

    # Heading and other texts
    if col == hcol:
        ax.set_title(f"{hteamName}\n{counttext}", color=line_color, fontsize=25, fontweight='bold')
    else:
        ax.set_title(f"{ateamName}\n{counttext}", color=line_color, fontsize=25, fontweight='bold')

    return {
        'Team_Name': team_name,
        'Total_Progressive_Passes': pro_count,
        'Progressive_Passes_From_Left': left_pro,
        'Progressive_Passes_From_Center': mid_pro,
        'Progressive_Passes_From_Right': right_pro
    }

In [ ]:
fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
Progressvie_Passes_Stats_home = draw_progressive_pass_map(axs[0], hteamName, hcol)
Progressvie_Passes_Stats_away = draw_progressive_pass_map(axs[1], ateamName, acol)
Progressvie_Passes_Stats_list = []
Progressvie_Passes_Stats_list.append(Progressvie_Passes_Stats_home)
Progressvie_Passes_Stats_list.append(Progressvie_Passes_Stats_away)
Progressvie_Passes_Stats_df = pd.DataFrame(Progressvie_Passes_Stats_list)

In [ ]:
Progressvie_Passes_Stats_df.head()

# Carriles de Progresión (Progressive Carry)

### Resumen:
La función `draw_progressive_carry_map()` visualiza las conducciones progresivas de un equipo en un campo de fútbol, destacando cómo el equipo avanza con el balón en diferentes áreas del campo (izquierda, centro, derecha). Además de trazar las conducciones, la función calcula cuántas de ellas ocurrieron en cada zona y muestra esta información junto con las flechas de las conducciones, proporcionando una vista clara y detallada de las acciones ofensivas del equipo.

### Propósito:
La función `draw_progressive_carry_map()` crea una visualización de los **arrastres progresivos** o **conducciones progresivas** de un equipo en un campo de fútbol. Un arrastre progresivo, según la definición utilizada aquí, es cuando un jugador lleva el balón más de 10 yardas hacia adelante y termina en una zona más avanzada del campo (excluyendo el tercio defensivo). Además de trazar las conducciones, la función calcula cuántas de estas progresiones ocurrieron por el lado izquierdo, central o derecho del campo, y muestra esta información visualmente en el gráfico.

### Parámetros:
- **`ax`**: El eje sobre el que se dibujará el gráfico.
- **`team_name`**: El nombre del equipo (local o visitante) para el cual se generará el mapa de conducciones progresivas.
- **`col`**: El color utilizado para representar las conducciones progresivas del equipo en la visualización.

### Descripción paso a paso:

1. **Filtrado de las conducciones progresivas:**
   - El `DataFrame` `dfpro` se filtra para obtener solo las conducciones que:
     - **Equipo**: Fueron realizadas por el equipo especificado (`team_name`).
     - **Progresividad**: La conducción ha reducido la distancia hacia la portería contraria en al menos 10 yardas (9.11 metros).
     - **Ubicación final**: La conducción terminó en una zona del campo más allá del tercio defensivo (excluyendo `endX < 35`).

2. **Dibujar el campo de fútbol:**
   - Se utiliza la clase `Pitch` de **mplsoccer** para dibujar un campo de fútbol estilo UEFA.

3. **Inversión del eje para el equipo visitante:**
   - Si el equipo es el visitante (`ateamName`), se invierten los ejes para que el equipo ataque de izquierda a derecha en el gráfico.

4. **Cálculo de las conducciones progresivas por tercio del campo:**
   - Se cuentan las conducciones progresivas realizadas desde tres zonas del campo:
     - **Izquierda**: Conducciones realizadas desde el tercio izquierdo del campo (`y >= 45.33`).
     - **Centro**: Conducciones realizadas desde el tercio central (`22.67 <= y < 45.33`).
     - **Derecha**: Conducciones realizadas desde el tercio derecho (`y < 22.67`).
   - Se calculan los porcentajes de las conducciones progresivas desde cada una de estas zonas.

5. **Añadir líneas horizontales:**
   - Se dibujan líneas horizontales en las posiciones `22.67` y `45.33` para dividir el campo en tercios, separando visualmente las zonas izquierda, central y derecha.

6. **Mostrar las estadísticas en el gráfico:**
   - Se añaden textos en las zonas correspondientes del campo que muestran el número de conducciones progresivas y su porcentaje desde cada tercio.

7. **Dibujar las conducciones progresivas:**
   - Las conducciones progresivas se representan como flechas desde las coordenadas de inicio (`x`, `y`) hasta las coordenadas de finalización (`endX`, `endY`) utilizando la clase `FancyArrowPatch` de `matplotlib.patches`.
   - Las flechas están estilizadas con un color y grosor específicos para mejorar su visibilidad.

8. **Títulos y textos adicionales:**
   - Se muestra un título para el equipo, que incluye el número total de conducciones progresivas realizadas.
   - Dependiendo de si el equipo es local o visitante, se ajusta el color y el estilo del texto.

9. **Retorno:**
   - La función devuelve un diccionario con las siguientes estadísticas clave:
     - **`Team_Name`**: Nombre del equipo.
     - **`Total_Progressive_Carries`**: Número total de conducciones progresivas realizadas.
     - **`Progressive_Carries_From_Left`**: Número de conducciones progresivas desde la banda izquierda.
     - **`Progressive_Carries_From_Center`**: Número de conducciones progresivas desde el centro del campo.
     - **`Progressive_Carries_From_Right`**: Número de conducciones progresivas desde la banda derecha.


In [ ]:
def draw_progressive_carry_map(ax, team_name, col):
    # filtering those carries which has reduced the distance form goal for at least 10yds and not ended at defensive third, this is my condition for a progressive pass, which almost similar to opta/statsbomb conditon
    dfpro = df[(df['teamName']==team_name) & (df['prog_carry']>=9.11) & (df['endX']>=35)]
    pitch = Pitch(pitch_type='uefa', pitch_color=bg_color, line_color=line_color, linewidth=2,
                          corner_arcs=True)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    # ax.set_ylim(-5, 68.5)

    if team_name == ateamName:
        ax.invert_xaxis()
        ax.invert_yaxis()

    pro_count = len(dfpro)

    # calculating the counts
    left_pro = len(dfpro[dfpro['y']>=45.33])
    mid_pro = len(dfpro[(dfpro['y']>=22.67) & (dfpro['y']<45.33)])
    right_pro = len(dfpro[(dfpro['y']>=0) & (dfpro['y']<22.67)])
    left_percentage = round((left_pro/pro_count)*100)
    mid_percentage = round((mid_pro/pro_count)*100)
    right_percentage = round((right_pro/pro_count)*100)

    ax.hlines(22.67, xmin=0, xmax=105, colors=line_color, linestyle='dashed', alpha=0.35)
    ax.hlines(45.33, xmin=0, xmax=105, colors=line_color, linestyle='dashed', alpha=0.35)

    # showing the texts in the pitch
    bbox_props = dict(boxstyle="round,pad=0.3", edgecolor="None", facecolor=bg_color, alpha=0.75)
    if col == hcol:
        ax.text(8, 11.335, f'{right_pro}\n({right_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 34, f'{mid_pro}\n({mid_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 56.675, f'{left_pro}\n({left_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
    else:
        ax.text(8, 11.335, f'{right_pro}\n({right_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 34, f'{mid_pro}\n({mid_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 56.675, f'{left_pro}\n({left_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)

    # plotting the carries
    for index, row in dfpro.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), arrowstyle='->', color=col, zorder=4, mutation_scale=20, 
                                        alpha=0.9, linewidth=2, linestyle='--')
        ax.add_patch(arrow)

    counttext = f"{pro_count} Progressive Carries"

    # Heading and other texts
    if col == hcol:
        ax.set_title(f"{hteamName}\n{counttext}", color=line_color, fontsize=25, fontweight='bold')
    else:
        ax.set_title(f"{ateamName}\n{counttext}", color=line_color, fontsize=25, fontweight='bold')

    return {
        'Team_Name': team_name,
        'Total_Progressive_Carries': pro_count,
        'Progressive_Carries_From_Left': left_pro,
        'Progressive_Carries_From_Center': mid_pro,
        'Progressive_Carries_From_Right': right_pro
    }

In [ ]:
fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
Progressvie_Carries_Stats_home = draw_progressive_carry_map(axs[0], hteamName, hcol)
Progressvie_Carries_Stats_away = draw_progressive_carry_map(axs[1], ateamName, acol)
Progressvie_Carries_Stats_list = []
Progressvie_Carries_Stats_list.append(Progressvie_Carries_Stats_home)
Progressvie_Carries_Stats_list.append(Progressvie_Carries_Stats_away)
Progressvie_Carries_Stats_df = pd.DataFrame(Progressvie_Carries_Stats_list)

In [ ]:
Progressvie_Carries_Stats_df.head()

# Mapa de disparos (ShotMap)

### Resumen:
Este código procesa y analiza los datos de tiros de un partido de fútbol, proporcionando estadísticas clave para ambos equipos, como el número total de tiros, tiros a puerta, el promedio de goles esperados por tiro (xG por tiro), y la distancia promedio de los tiros desde la portería.

### Propósito:
Este bloque de código filtra y analiza los **datos de tiros** en un partido de fútbol. Filtra los tiros realizados por ambos equipos y genera estadísticas sobre los tiros, como el número total de tiros, los tiros a puerta, el valor esperado de goles por tiro, y la distancia promedio de los tiros.

### Descripción paso a paso:

1. **Filtrar los eventos relacionados con tiros:**
   - **`mask4`**: Crea una máscara que selecciona solo los eventos de tipo `Goal`, `MissedShots`, `SavedShot` y `ShotOnPost`.
   - **`Shotsdf`**: Aplica la máscara para filtrar el `DataFrame` original `df` y obtener un `DataFrame` que contenga solo los tiros. Luego, se reinicia el índice del `DataFrame` resultante para asegurar que sea consecutivo.

2. **Filtrar tiros por equipo:**
   - **`hShotsdf`**: Filtra los tiros realizados por el equipo local (`hteamName`).
   - **`aShotsdf`**: Filtra los tiros realizados por el equipo visitante (`ateamName`).

3. **Filtrar los tiros atajados (SavedShots) y excluir autogoles:**
   - **`hSavedf`** y **`aSavedf`**: Filtran los tiros atajados por ambos equipos, excluyendo aquellos que contienen una cadena que indica que el tiro fue un autogol (`': 82,'`).
   - **`hogdf`** y **`aogdf`**: Filtran los autogoles (OwnGoals) de ambos equipos.

4. **Estadísticas de tiro:**
   - **`hTotalShots`** y **`aTotalShots`**: Calcula el número total de tiros realizados por el equipo local y visitante.
   - **`hShotsOnT`** y **`aShotsOnT`**: Calcula el número de tiros a puerta combinando los tiros atajados (`SavedShots`) y los goles marcados. Los autogoles no se incluyen en estas cuentas.
   - **`hxGpSh`** y **`axGpSh`**: Calculan el valor esperado de goles por tiro (`xG per shot`) para el equipo local y visitante. Esto se obtiene dividiendo el valor total de goles esperados (`hxg` y `axg`) por el número total de tiros realizados.

5. **Distancia promedio de los tiros:**
   - **`given_point`**: Define el centro de la portería, que se encuentra en las coordenadas `(105, 34)` en el campo de fútbol.
   - **`home_shot_distances`** y **`away_shot_distances`**: Calculan la distancia de cada tiro desde su posición en el campo (`x`, `y`) hasta el centro de la portería.
   - **`home_average_shot_distance`** y **`away_average_shot_distance`**: Calculan la distancia promedio de los tiros para ambos equipos y redondean el valor a dos decimales.



In [ ]:
# filtering the shots only
mask4 = (df['type'] == 'Goal') | (df['type'] == 'MissedShots') | (df['type'] == 'SavedShot') | (df['type'] == 'ShotOnPost')
Shotsdf = df[mask4]
Shotsdf.reset_index(drop=True, inplace=True)

# filtering according to the types of shots
hShotsdf = Shotsdf[Shotsdf['teamName']==hteamName]
aShotsdf = Shotsdf[Shotsdf['teamName']==ateamName]
hSavedf = hShotsdf[(hShotsdf['type']=='SavedShot') & (~hShotsdf['qualifiers'].str.contains(': 82,'))]
aSavedf = aShotsdf[(aShotsdf['type']=='SavedShot') & (~aShotsdf['qualifiers'].str.contains(': 82,'))]
hogdf = hShotsdf[(hShotsdf['teamName']==hteamName) & (hShotsdf['qualifiers'].str.contains('OwnGoal'))]
aogdf = aShotsdf[(aShotsdf['teamName']==ateamName) & (aShotsdf['qualifiers'].str.contains('OwnGoal'))]

#shooting stats
hTotalShots = len(hShotsdf)
aTotalShots = len(aShotsdf)
hShotsOnT = len(hSavedf) + hgoal_count
aShotsOnT = len(aSavedf) + agoal_count
hxGpSh = round(hxg/hTotalShots, 2)
axGpSh = round(axg/hTotalShots, 2)
# Center Goal point
given_point = (105, 34)
# Calculate shot distances
home_shot_distances = np.sqrt((hShotsdf['x'] - given_point[0])**2 + (hShotsdf['y'] - given_point[1])**2)
home_average_shot_distance = round(home_shot_distances.mean(),2)
away_shot_distances = np.sqrt((aShotsdf['x'] - given_point[0])**2 + (aShotsdf['y'] - given_point[1])**2)
away_average_shot_distance = round(away_shot_distances.mean(),2)


### Resumen:
La función `plot_shotmap()` no solo dibuja un mapa detallado de los tiros en un partido de fútbol, sino que también genera una serie de estadísticas clave que comparan el rendimiento de ambos equipos en términos de goles, xG, tiros, Big Chances, y más. La visualización incluye un gráfico de barras horizontal que facilita la comparación de estas métricas entre los dos equipos. Los datos de estas estadísticas se devuelven como diccionarios y se almacenan en un `DataFrame` para análisis posteriores.

### Propósito:
La función `plot_shotmap()` crea una **visualización del mapa de tiros** para un partido de fútbol. Incluye diferentes tipos de tiros (goles, tiros al poste, tiros atajados, y tiros fallidos) para ambos equipos (local y visitante), diferenciando entre oportunidades normales y grandes oportunidades (**Big Chances**). También genera un gráfico de barras que compara varias estadísticas clave de ambos equipos, como goles, xG (goles esperados), tiros totales, y más.

### Descripción paso a paso:

1. **Dibujo del campo de fútbol:**
   - Utiliza la clase `Pitch` de **mplsoccer** para dibujar el campo de fútbol estilo UEFA. El campo se dibuja en el eje `ax`.

2. **Filtrado de los datos de tiros:**
   - Los datos de tiros (`Shotsdf`) se filtran en varios subconjuntos:
     - Tiros del equipo local y visitante que resultaron en goles, tiros al poste, tiros atajados, y tiros fallidos, tanto con como sin **Big Chances**.
     - También se filtran los autogoles para ambos equipos.

3. **Dibujo de los tiros en el campo:**
   - Se utilizan las posiciones `x` e `y` de los tiros para crear diferentes tipos de marcas en el campo, usando `scatter()` de **mplsoccer**:
     - **Goles** se representan con una marca de fútbol de color verde.
     - **Autogoles** se representan con una marca de fútbol de color naranja.
     - **Tiros al poste**, **tiros atajados**, y **tiros fallidos** tienen diferentes marcas y colores.
     - Los tiros considerados **Big Chances** se dibujan con marcas más grandes para destacar su importancia.

4. **Generación de estadísticas clave:**
   - Se calculan varias estadísticas clave para ambos equipos, incluyendo:
     - **Goles anotados**.
     - **xG (goles esperados)**.
     - **xGOT (goles esperados en tiros a puerta)**.
     - **Total de tiros**.
     - **Tiros a puerta**.
     - **Big Chances (grandes oportunidades)** y **Big Chances falladas**.
     - **xG por tiro**.
     - **Distancia promedio de los tiros**.

5. **Normalización de estadísticas para el gráfico de barras:**
   - Se normalizan las estadísticas de ambos equipos para asegurar que las barras estén en la misma escala, incluso si el partido terminó 0-0 o si alguna métrica tiene un valor 0.
   - Se crea un gráfico de barras horizontal comparando las estadísticas de tiro de ambos equipos. Las barras para el equipo local y visitante se dibujan una al lado de la otra para facilitar la comparación.

6. **Etiquetas de estadísticas en el gráfico:**
   - Se añaden etiquetas de texto para cada estadística en el gráfico de barras, mostrando tanto los valores de las estadísticas como el título de cada métrica (goles, xG, tiros, etc.).
   - Los tiros de ambos equipos también se describen visualmente con etiquetas que indican la dirección de los tiros en el gráfico de campo.

7. **Retorno de los datos:**
   - La función devuelve dos diccionarios, uno para cada equipo, que contienen todas las estadísticas clave calculadas en el gráfico. Estos datos se pueden utilizar para análisis adicionales o reportes.

8. **Visualización del gráfico:**
   - Finalmente, se genera el gráfico de barras y el mapa de tiros para ambos equipos usando `matplotlib`.



In [ ]:
def plot_shotmap(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, linewidth=2, line_color=line_color)
    pitch.draw(ax=ax)
    ax.set_ylim(-0.5,68.5)
    ax.set_xlim(-0.5,105.5)
    # without big chances for home team
    hGoalData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'Goal') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    hPostData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'ShotOnPost') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    hSaveData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'SavedShot') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    hMissData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'MissedShots') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    # only big chances of home team
    Big_C_hGoalData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'Goal') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    Big_C_hPostData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'ShotOnPost') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    Big_C_hSaveData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'SavedShot') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    Big_C_hMissData = Shotsdf[(Shotsdf['teamName'] == hteamName) & (Shotsdf['type'] == 'MissedShots') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    total_bigC_home = len(Big_C_hGoalData) + len(Big_C_hPostData) + len(Big_C_hSaveData) + len(Big_C_hMissData)
    bigC_miss_home = len(Big_C_hPostData) + len(Big_C_hSaveData) + len(Big_C_hMissData)
    # normal shots scatter of home team
    sc2 = pitch.scatter((105-hPostData.x), (68-hPostData.y), s=200, edgecolors=hcol, c=hcol, marker='o', ax=ax)
    sc3 = pitch.scatter((105-hSaveData.x), (68-hSaveData.y), s=200, edgecolors=hcol, c='None', hatch='///////', marker='o', ax=ax)
    sc4 = pitch.scatter((105-hMissData.x), (68-hMissData.y), s=200, edgecolors=hcol, c='None', marker='o', ax=ax)
    sc1 = pitch.scatter((105-hGoalData.x), (68-hGoalData.y), s=350, edgecolors='green', linewidths=0.6, c='None', marker='football', zorder=3, ax=ax)
    sc1_og = pitch.scatter((105-hogdf.x), (68-hogdf.y), s=350, edgecolors='orange', linewidths=0.6, c='None', marker='football', zorder=3, ax=ax)
    # big chances bigger scatter of home team
    bc_sc2 = pitch.scatter((105-Big_C_hPostData.x), (68-Big_C_hPostData.y), s=500, edgecolors=hcol, c=hcol, marker='o', ax=ax)
    bc_sc3 = pitch.scatter((105-Big_C_hSaveData.x), (68-Big_C_hSaveData.y), s=500, edgecolors=hcol, c='None', hatch='///////', marker='o', ax=ax)
    bc_sc4 = pitch.scatter((105-Big_C_hMissData.x), (68-Big_C_hMissData.y), s=500, edgecolors=hcol, c='None', marker='o', ax=ax)
    bc_sc1 = pitch.scatter((105-Big_C_hGoalData.x), (68-Big_C_hGoalData.y), s=650, edgecolors='green', linewidths=0.6, c='None', marker='football', ax=ax)

    # without big chances for away team
    aGoalData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'Goal') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    aPostData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'ShotOnPost') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    aSaveData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'SavedShot') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    aMissData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'MissedShots') & (~Shotsdf['qualifiers'].str.contains('BigChance'))]
    # only big chances of away team
    Big_C_aGoalData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'Goal') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    Big_C_aPostData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'ShotOnPost') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    Big_C_aSaveData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'SavedShot') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    Big_C_aMissData = Shotsdf[(Shotsdf['teamName'] == ateamName) & (Shotsdf['type'] == 'MissedShots') & (Shotsdf['qualifiers'].str.contains('BigChance'))]
    total_bigC_away = len(Big_C_aGoalData) + len(Big_C_aPostData) + len(Big_C_aSaveData) + len(Big_C_aMissData)
    bigC_miss_away = len(Big_C_aPostData) + len(Big_C_aSaveData) + len(Big_C_aMissData)
    # normal shots scatter of away team
    sc6 = pitch.scatter(aPostData.x, aPostData.y, s=200, edgecolors=acol, c=acol, marker='o', ax=ax)
    sc7 = pitch.scatter(aSaveData.x, aSaveData.y, s=200, edgecolors=acol, c='None', hatch='///////', marker='o', ax=ax)
    sc8 = pitch.scatter(aMissData.x, aMissData.y, s=200, edgecolors=acol, c='None', marker='o', ax=ax)
    sc5 = pitch.scatter(aGoalData.x, aGoalData.y, s=350, edgecolors='green', linewidths=0.6, c='None', marker='football', zorder=3, ax=ax)
    sc5_og = pitch.scatter((aogdf.x), (aogdf.y), s=350, edgecolors='orange', linewidths=0.6, c='None', marker='football', zorder=3, ax=ax)
    # big chances bigger scatter of away team
    bc_sc6 = pitch.scatter(Big_C_aPostData.x, Big_C_aPostData.y, s=700, edgecolors=acol, c=acol, marker='o', ax=ax)
    bc_sc7 = pitch.scatter(Big_C_aSaveData.x, Big_C_aSaveData.y, s=700, edgecolors=acol, c='None', hatch='///////', marker='o', ax=ax)
    bc_sc8 = pitch.scatter(Big_C_aMissData.x, Big_C_aMissData.y, s=700, edgecolors=acol, c='None', marker='o', ax=ax)
    bc_sc5 = pitch.scatter(Big_C_aGoalData.x, Big_C_aGoalData.y, s=850, edgecolors='green', linewidths=0.6, c='None', marker='football', ax=ax)

    # Stats bar diagram
    shooting_stats_title = [62, 62-(1*7), 62-(2*7), 62-(3*7), 62-(4*7), 62-(5*7), 62-(6*7), 62-(7*7), 62-(8*7)]
    shooting_stats_home = [hgoal_count, hxg, hxgot, hTotalShots, hShotsOnT, hxGpSh, total_bigC_home, bigC_miss_home, home_average_shot_distance]
    shooting_stats_away = [agoal_count, axg, axgot, aTotalShots, aShotsOnT, axGpSh, total_bigC_away, bigC_miss_away, away_average_shot_distance]

    # sometimes the both teams ends the match 0-0, then normalizing the data becomes problem, thats why this part of the code
    if hgoal_count+agoal_count == 0:
       hgoal = 10
       agoal = 10
    else:
       hgoal = (hgoal_count/(hgoal_count+agoal_count))*20
       agoal = (agoal_count/(hgoal_count+agoal_count))*20
        
    if total_bigC_home+total_bigC_away == 0:
       total_bigC_home = 10
       total_bigC_away = 10
        
    if bigC_miss_home+bigC_miss_away == 0:
       bigC_miss_home = 10
       bigC_miss_away = 10

    # normalizing the stats
    shooting_stats_normalized_home = [hgoal, (hxg/(hxg+axg))*20, (hxgot/(hxgot+axgot))*20,
                                      (hTotalShots/(hTotalShots+aTotalShots))*20, (hShotsOnT/(hShotsOnT+aShotsOnT))*20,
                                      (total_bigC_home/(total_bigC_home+total_bigC_away))*20, (bigC_miss_home/(bigC_miss_home+bigC_miss_away))*20,
                                      (hxGpSh/(hxGpSh+axGpSh))*20, 
                                      (home_average_shot_distance/(home_average_shot_distance+away_average_shot_distance))*20]
    shooting_stats_normalized_away = [agoal, (axg/(hxg+axg))*20, (axgot/(hxgot+axgot))*20,
                                      (aTotalShots/(hTotalShots+aTotalShots))*20, (aShotsOnT/(hShotsOnT+aShotsOnT))*20,
                                      (total_bigC_away/(total_bigC_home+total_bigC_away))*20, (bigC_miss_away/(bigC_miss_home+bigC_miss_away))*20,
                                      (axGpSh/(hxGpSh+axGpSh))*20,
                                      (away_average_shot_distance/(home_average_shot_distance+away_average_shot_distance))*20]

    # definig the start point
    start_x = 42.5
    start_x_for_away = [x + 42.5 for x in shooting_stats_normalized_home]
    ax.barh(shooting_stats_title, shooting_stats_normalized_home, height=5, color=hcol, left=start_x)
    ax.barh(shooting_stats_title, shooting_stats_normalized_away, height=5, left=start_x_for_away, color=acol)
    # Turn off axis-related elements
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)
    ax.set_xticks([])
    ax.set_yticks([])

    # plotting the texts
    ax.text(52.5, 62, "Goals", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(1*7), "xG", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(2*7), "xGOT", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(3*7), "Shots", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(4*7), "On Target", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(5*7), "BigChance", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(6*7), "BigC.Miss", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(7*7), "xG/Shot", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')
    ax.text(52.5, 62-(8*7), "Avg.Dist.", color=bg_color, fontsize=18, ha='center', va='center', fontweight='bold')

    ax.text(41.5, 62, f"{hgoal_count}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(1*7), f"{hxg}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(2*7), f"{hxgot}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(3*7), f"{hTotalShots}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(4*7), f"{hShotsOnT}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(5*7), f"{total_bigC_home}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(6*7), f"{bigC_miss_home}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(7*7), f"{hxGpSh}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')
    ax.text(41.5, 62-(8*7), f"{home_average_shot_distance}", color=line_color, fontsize=18, ha='right', va='center', fontweight='bold')

    ax.text(63.5, 62, f"{agoal_count}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(1*7), f"{axg}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(2*7), f"{axgot}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(3*7), f"{aTotalShots}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(4*7), f"{aShotsOnT}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(5*7), f"{total_bigC_away}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(6*7), f"{bigC_miss_away}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(7*7), f"{axGpSh}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')
    ax.text(63.5, 62-(8*7), f"{away_average_shot_distance}", color=line_color, fontsize=18, ha='left', va='center', fontweight='bold')

    # Heading and other texts
    ax.text(0, 70, f"{hteamName}\n<---shots", color=hcol, size=25, ha='left', fontweight='bold')
    ax.text(105, 70, f"{ateamName}\nshots--->", color=acol, size=25, ha='right', fontweight='bold')

    home_data = {
        'Team_Name': hteamName,
        'Goals_Scored': hgoal_count,
        'xG': hxg,
        'xGOT': hxgot,
        'Total_Shots': hTotalShots,
        'Shots_On_Target': hShotsOnT,
        'BigChances': total_bigC_home,
        'BigChances_Missed': bigC_miss_home,
        'xG_per_Shot': hxGpSh,
        'Average_Shot_Distance': home_average_shot_distance
    }
    
    away_data = {
        'Team_Name': ateamName,
        'Goals_Scored': agoal_count,
        'xG': axg,
        'xGOT': axgot,
        'Total_Shots': aTotalShots,
        'Shots_On_Target': aShotsOnT,
        'BigChances': total_bigC_away,
        'BigChances_Missed': bigC_miss_away,
        'xG_per_Shot': axGpSh,
        'Average_Shot_Distance': away_average_shot_distance
    }
    
    return [home_data, away_data]

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
shooting_stats = plot_shotmap(ax)
shooting_stats_df = pd.DataFrame(shooting_stats)

In [ ]:
shooting_stats_df.head()

# Mapas de disparos en la portería (GoalPost)

### Resumen:
La función `plot_goalPost()` crea una visualización detallada de los tiros a puerta, destacando las atajadas, los goles y los tiros al poste, tanto para el equipo local como para el visitante. Además, calcula estadísticas importantes sobre el rendimiento de los arqueros, como las atajadas y los goles prevenidos, proporcionando una vista clara y útil para analizar su desempeño durante el partido. Los datos generados se devuelven en un formato adecuado para su posterior análisis en un `DataFrame`.

### Propósito:
La función `plot_goalPost()` genera una **visualización de las atajadas de los arqueros** en un partido de fútbol. Dibuja los postes de ambas porterías, superpone los tiros realizados a la portería (incluyendo goles, tiros atajados y tiros al poste), y también calcula y muestra estadísticas clave relacionadas con el desempeño de los arqueros, como las atajadas y los goles evitados.

### Descripción paso a paso:

1. **Transformación de los datos de tiros según las dimensiones de la portería:**
   - Las posiciones de los tiros hacia la portería (`goalMouthY` y `goalMouthZ`) se ajustan para adaptarse a las dimensiones del campo y de la portería. 
   - Se escala y transforma la ubicación de los tiros para que se ajusten correctamente a la visualización.

2. **Dibujo del campo de fútbol y los postes de la portería:**
   - Se utiliza **mplsoccer** para dibujar el campo de fútbol (sin líneas visibles, ya que los postes están dentro de las dimensiones del campo).
   - Los postes de la portería del equipo local y visitante se dibujan manualmente con `ax.plot()`, representando el marco y las líneas de la red.

3. **Filtrado de los tiros:**
   - Se filtran los datos de tiros (`Shotsdf`) para cada equipo (local y visitante) y se dividen en varias categorías:
     - **Tiros atajados (`SavedShot`)**: Los tiros que fueron atajados por el arquero.
     - **Goles (`Goal`)**: Tiros que resultaron en un gol.
     - **Tiros al poste (`ShotOnPost`)**: Tiros que pegaron en el poste.
     - **Big Chances**: Se incluyen tiros con grandes oportunidades (Big Chances), distinguidos de los tiros normales.

4. **Dibujo de los tiros en las porterías:**
   - Los tiros se representan en las porterías usando `scatter()`, con diferentes estilos y tamaños dependiendo del tipo de tiro:
     - **Tiros atajados**: Se representan con marcas circulares (`o`) y un patrón de rayas.
     - **Goles**: Se representan con marcas de fútbol (`football`) de color verde.
     - **Tiros al poste**: Se representan con marcas circulares de color naranja.
     - **Big Chances**: Estos tiros tienen marcas más grandes para resaltar su importancia.

5. **Cálculo de estadísticas clave de los arqueros:**
   - Se calculan varias estadísticas relacionadas con el rendimiento del arquero, incluyendo:
     - **Shots Saved**: El número de tiros atajados.
     - **Big Chance Saved**: El número de grandes oportunidades que fueron atajadas.
     - **Goals Prevented**: El número de goles evitados, calculado como la diferencia entre los goles esperados en tiros a puerta (`xGOT`) y los goles realmente encajados.

6. **Mostrar las estadísticas en el gráfico:**
   - Se añaden textos en el gráfico que muestran el número de atajadas realizadas por cada arquero, el xGOT que enfrentaron y los goles prevenidos. Estas estadísticas se colocan cerca de cada portería para facilitar la visualización.

7. **Retorno de los datos:**
   - La función devuelve dos diccionarios, uno para cada equipo, que contienen las estadísticas clave calculadas (atajadas, Big Chances atajadas, y goles prevenidos). Estos datos se almacenan en un `DataFrame` para análisis posteriores.



In [ ]:
def plot_goalPost(ax):
    hShotsdf = Shotsdf[Shotsdf['teamName']==hteamName]
    aShotsdf = Shotsdf[Shotsdf['teamName']==ateamName]
    # converting the datapoints according to the pitch dimension, because the goalposts are being plotted inside the pitch using pitch's dimension
    hShotsdf['goalMouthZ'] = hShotsdf['goalMouthZ']*0.75
    aShotsdf['goalMouthZ'] = (aShotsdf['goalMouthZ']*0.75) + 38

    hShotsdf['goalMouthY'] = ((37.66 - hShotsdf['goalMouthY'])*12.295) + 7.5
    aShotsdf['goalMouthY'] = ((37.66 - aShotsdf['goalMouthY'])*12.295) + 7.5

    # plotting an invisible pitch using the pitch color and line color same color, because the goalposts are being plotted inside the pitch using pitch's dimension
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=bg_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_ylim(-0.5,68.5)
    ax.set_xlim(-0.5,105.5)

    # away goalpost bars
    ax.plot([7.5, 7.5], [0, 30], color=line_color, linewidth=5)
    ax.plot([7.5, 97.5], [30, 30], color=line_color, linewidth=5)
    ax.plot([97.5, 97.5], [30, 0], color=line_color, linewidth=5)
    ax.plot([0, 105], [0, 0], color=line_color, linewidth=3)
    # plotting the away net
    y_values = np.arange(0, 6) * 6
    for y in y_values:
        ax.plot([7.5, 97.5], [y, y], color=line_color, linewidth=2, alpha=0.2)
    x_values = (np.arange(0, 11) * 9) + 7.5
    for x in x_values:
        ax.plot([x, x], [0, 30], color=line_color, linewidth=2, alpha=0.2)
    # home goalpost bars
    ax.plot([7.5, 7.5], [38, 68], color=line_color, linewidth=5)
    ax.plot([7.5, 97.5], [68, 68], color=line_color, linewidth=5)
    ax.plot([97.5, 97.5], [68, 38], color=line_color, linewidth=5)
    ax.plot([0, 105], [38, 38], color=line_color, linewidth=3)
    # plotting the home net
    y_values = (np.arange(0, 6) * 6) + 38
    for y in y_values:
        ax.plot([7.5, 97.5], [y, y], color=line_color, linewidth=2, alpha=0.2)
    x_values = (np.arange(0, 11) * 9) + 7.5
    for x in x_values:
        ax.plot([x, x], [38, 68], color=line_color, linewidth=2, alpha=0.2)

    # filtering different types of shots without BigChance
    hSavedf = hShotsdf[(hShotsdf['type']=='SavedShot') & (~hShotsdf['qualifiers'].str.contains(': 82,')) & (~hShotsdf['qualifiers'].str.contains('BigChance'))]
    hGoaldf = hShotsdf[(hShotsdf['type']=='Goal') & (~hShotsdf['qualifiers'].str.contains('OwnGoal')) & (~hShotsdf['qualifiers'].str.contains('BigChance'))]
    hPostdf = hShotsdf[(hShotsdf['type']=='ShotOnPost') & (~hShotsdf['qualifiers'].str.contains('BigChance'))]
    aSavedf = aShotsdf[(aShotsdf['type']=='SavedShot') & (~aShotsdf['qualifiers'].str.contains(': 82,')) & (~aShotsdf['qualifiers'].str.contains('BigChance'))]
    aGoaldf = aShotsdf[(aShotsdf['type']=='Goal') & (~aShotsdf['qualifiers'].str.contains('OwnGoal')) & (~aShotsdf['qualifiers'].str.contains('BigChance'))]
    aPostdf = aShotsdf[(aShotsdf['type']=='ShotOnPost') & (~aShotsdf['qualifiers'].str.contains('BigChance'))]
    # filtering different types of shots with BigChance
    hSavedf_bc = hShotsdf[(hShotsdf['type']=='SavedShot') & (~hShotsdf['qualifiers'].str.contains(': 82,')) & (hShotsdf['qualifiers'].str.contains('BigChance'))]
    hGoaldf_bc = hShotsdf[(hShotsdf['type']=='Goal') & (~hShotsdf['qualifiers'].str.contains('OwnGoal')) & (hShotsdf['qualifiers'].str.contains('BigChance'))]
    hPostdf_bc = hShotsdf[(hShotsdf['type']=='ShotOnPost') & (hShotsdf['qualifiers'].str.contains('BigChance'))]
    aSavedf_bc = aShotsdf[(aShotsdf['type']=='SavedShot') & (~aShotsdf['qualifiers'].str.contains(': 82,')) & (aShotsdf['qualifiers'].str.contains('BigChance'))]
    aGoaldf_bc = aShotsdf[(aShotsdf['type']=='Goal') & (~aShotsdf['qualifiers'].str.contains('OwnGoal')) & (aShotsdf['qualifiers'].str.contains('BigChance'))]
    aPostdf_bc = aShotsdf[(aShotsdf['type']=='ShotOnPost') & (aShotsdf['qualifiers'].str.contains('BigChance'))]

    # scattering those shots without BigChance
    sc1 = pitch.scatter(hSavedf.goalMouthY, hSavedf.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolor=acol, hatch='/////', s=350, ax=ax)
    sc2 = pitch.scatter(hGoaldf.goalMouthY, hGoaldf.goalMouthZ, marker='football', c=bg_color, zorder=3, edgecolors='green', s=350, ax=ax)
    sc3 = pitch.scatter(hPostdf.goalMouthY, hPostdf.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolors='orange', hatch='/////', s=350, ax=ax)
    sc4 = pitch.scatter(aSavedf.goalMouthY, aSavedf.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolor=hcol, hatch='/////', s=350, ax=ax)
    sc5 = pitch.scatter(aGoaldf.goalMouthY, aGoaldf.goalMouthZ, marker='football', c=bg_color, zorder=3, edgecolors='green', s=350, ax=ax)
    sc6 = pitch.scatter(aPostdf.goalMouthY, aPostdf.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolors='orange', hatch='/////', s=350, ax=ax)
    # scattering those shots with BigChance
    sc1_bc = pitch.scatter(hSavedf_bc.goalMouthY, hSavedf_bc.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolor=acol, hatch='/////', s=1000, ax=ax)
    sc2_bc = pitch.scatter(hGoaldf_bc.goalMouthY, hGoaldf_bc.goalMouthZ, marker='football', c=bg_color, zorder=3, edgecolors='green', s=1000, ax=ax)
    sc3_bc = pitch.scatter(hPostdf_bc.goalMouthY, hPostdf_bc.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolors='orange', hatch='/////', s=1000, ax=ax)
    sc4_bc = pitch.scatter(aSavedf_bc.goalMouthY, aSavedf_bc.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolor=hcol, hatch='/////', s=1000, ax=ax)
    sc5_bc = pitch.scatter(aGoaldf_bc.goalMouthY, aGoaldf_bc.goalMouthZ, marker='football', c=bg_color, zorder=3, edgecolors='green', s=1000, ax=ax)
    sc6_bc = pitch.scatter(aPostdf_bc.goalMouthY, aPostdf_bc.goalMouthZ, marker='o', c=bg_color, zorder=3, edgecolors='orange', hatch='/////', s=1000, ax=ax)

    # Headlines and other texts
    ax.text(52.5, 70, f"{hteamName} GK saves", color=hcol, fontsize=30, ha='center', fontweight='bold')
    ax.text(52.5, -2, f"{ateamName} GK saves", color=acol, fontsize=30, ha='center', va='top', fontweight='bold')

    ax.text(100, 68, f"Saves = {len(aSavedf)+len(aSavedf_bc)}\n\nxGOT faced:\n{axgot}\n\nGoals Prevented:\n{round(axgot - len(aGoaldf) - len(aGoaldf_bc),2)}",
                    color=hcol, fontsize=16, va='top', ha='left')
    ax.text(100, 2, f"Saves = {len(hSavedf)+len(hSavedf_bc)}\n\nxGOT faced:\n{hxgot}\n\nGoals Prevented:\n{round(hxgot - len(hGoaldf) - len(hGoaldf_bc),2)}",
                    color=acol, fontsize=16, va='bottom', ha='left')

    home_data = {
        'Team_Name': hteamName,
        'Shots_Saved': len(hSavedf)+len(hSavedf_bc),
        'Big_Chance_Saved': len(hSavedf_bc),
        'Goals_Prevented': round(hxgot - len(hGoaldf) - len(hGoaldf_bc),2)
    }
    
    away_data = {
        'Team_Name': ateamName,
        'Shots_Saved': len(aSavedf)+len(aSavedf_bc),
        'Big_Chance_Saved': len(aSavedf_bc),
        'Goals_Prevented': round(axgot - len(aGoaldf) - len(aGoaldf_bc),2)
    }
    
    return [home_data, away_data]

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
goalkeeping_stats = plot_goalPost(ax)
goalkeeping_stats_df = pd.DataFrame(goalkeeping_stats)

In [ ]:
goalkeeping_stats_df.head()

# Momentos del Partido (Match Momentum)

### Resumen:
Este código prepara los datos para visualizar el **momentum** del partido basado en el xT promedio por minuto. Invirtiendo los valores de xT del equipo visitante, se facilita la creación de un gráfico donde se comparan las amenazas ofensivas de ambos equipos a lo largo del tiempo. Los valores de xT se suavizan opcionalmente usando una mediana móvil para obtener una representación más clara del flujo del partido.

### Descripción paso a paso:

1. **Copia del `DataFrame`:**
   - **`Momentumdf = df.copy()`**: Se crea una copia del `DataFrame` original `df` (que contiene los eventos del partido), de modo que los cambios realizados no afecten los datos originales.

2. **Inversión de los valores xT del equipo visitante:**
   - **`Momentumdf.loc[Momentumdf['teamName'] == ateamName, 'end_zone_value_xT'] *= -1`**: Los valores de xT del equipo visitante (`ateamName`) se multiplican por -1. Esto se hace para poder trazar el momentum de ambos equipos en el mismo gráfico, colocando los valores del equipo visitante en el lado opuesto (negativo) del equipo local.

3. **Cálculo del xT promedio por minuto:**
   - **`Momentumdf.groupby('minute')['end_zone_value_xT'].mean()`**: Se agrupan los datos por minuto, y se calcula el promedio de los valores xT de cada equipo en cada minuto del partido. Esto proporciona una visión general de cómo fue cambiando la amenaza ofensiva durante el transcurso del partido, minuto a minuto.
   - **`Momentumdf.reset_index()`**: Se reinicia el índice del `DataFrame` resultante después del agrupamiento para mantener un formato adecuado de los datos.

4. **Renombrar columnas:**
   - **`Momentumdf.columns = ['minute', 'average_xT']`**: Se renombran las columnas del `DataFrame` a `minute` y `average_xT` para facilitar la comprensión de qué datos contienen.

5. **Rellenar valores nulos (NaN):**
   - **`Momentumdf['average_xT'].fillna(0, inplace=True)`**: Si algún minuto no tiene datos de xT (por ejemplo, si no hubo acciones significativas), los valores nulos (NaN) se rellenan con 0, lo que implica que no hubo amenaza ofensiva en ese minuto específico.

6. **Mediana móvil (comentado):**
   - **`Momentumdf['average_xT'] = Momentumdf['average_xT'].rolling(window=2, min_periods=1).median()`**: Esta línea, que está comentada, aplica una mediana móvil con una ventana de 2 minutos para suavizar los datos del xT promedio. Esto ayudaría a reducir fluctuaciones bruscas en el gráfico y ofrecer una visualización más fluida del momentum. 

   Si esta línea se descomenta, el código aplicará la mediana móvil, lo que significa que el valor de cada minuto sería ajustado en base al valor promedio de ese minuto y el anterior, suavizando los cambios bruscos.

In [ ]:
Momentumdf = df.copy()
# multiplying the away teams xT values with -1 so that I can plot them in the opposite of home teams
Momentumdf.loc[Momentumdf['teamName'] == ateamName, 'end_zone_value_xT'] *= -1
# taking average xT per minute
Momentumdf = Momentumdf.groupby('minute')['end_zone_value_xT'].mean()
Momentumdf = Momentumdf.reset_index()
Momentumdf.columns = ['minute', 'average_xT']
Momentumdf['average_xT'].fillna(0, inplace=True)
# Momentumdf['average_xT'] = Momentumdf['average_xT'].rolling(window=2, min_periods=1).median()

### Resumen:
La función `plot_Momentum()` visualiza cómo se desarrolló el control del partido en función del **Expected Threat (xT)** de cada equipo. Utiliza un gráfico de barras que representa el xT promedio por minuto, con diferentes colores para indicar qué equipo tuvo el control en cada momento. Además, marca eventos clave como goles, autogoles y tarjetas rojas para proporcionar una visión más detallada de cómo estos eventos afectaron el momentum del partido.

### Propósito:
La función `plot_Momentum()` genera una visualización del **momentum del partido**, basada en los valores promedio de **Expected Threat (xT)** por minuto para ambos equipos. También marca eventos clave como goles, autogoles y tarjetas rojas, mostrando cómo evolucionó el control del partido a lo largo del tiempo.

### Descripción paso a paso:

1. **Asignación de colores a las barras según el xT:**
   - **`colors = [hcol if x > 0 else acol for x in Momentumdf['average_xT']]`**: Se asignan colores a las barras del gráfico según el valor de xT. Si el valor es positivo (control del equipo local), se utiliza el color `hcol`, y si es negativo (control del equipo visitante), se utiliza el color `acol`.

2. **Listas de minutos clave:**
   - Se crean varias listas que contienen los minutos en los que ocurrieron eventos clave en el partido:
     - **`hgoal_list`**: Minutos en los que el equipo local anotó un gol.
     - **`agoal_list`**: Minutos en los que el equipo visitante anotó un gol.
     - **`hog_list`** y **`aog_list`**: Minutos en los que ocurrieron autogoles para el equipo local y visitante, respectivamente.
     - **`hred_list`** y **`ared_list`**: Minutos en los que el equipo local o visitante recibió una tarjeta roja.

3. **Variables auxiliares para el gráfico:**
   - Se calculan los valores más altos y más bajos de xT (`highest_xT` y `lowest_xT`), así como el minuto más alto (`highest_minute`), que se utilizarán para posicionar las anotaciones en el gráfico.

4. **Añadir anotaciones de goles, autogoles y tarjetas rojas:**
   - Usando `scatter()`, se colocan marcadores circulares y cuadrados para señalar cuándo ocurrieron goles, autogoles y tarjetas rojas:
     - Goles se marcan con círculos verdes.
     - Autogoles se marcan con círculos naranjas.
     - Tarjetas rojas se marcan con cuadrados rojos.

5. **Gráfico de barras del momentum:**
   - **`ax.bar(Momentumdf['minute'], Momentumdf['average_xT'], color=colors)`**: Se crea el gráfico de barras, donde cada barra representa el xT promedio de un minuto del partido.
   - El eje X muestra los minutos del partido, con una línea vertical punteada que marca el final de la primera mitad (minuto 45).

6. **Estilización del gráfico:**
   - Se configuran varios ajustes para mejorar la visualización:
     - Se eliminan las espinas del gráfico (`spines`) y las marcas de los ejes (`tick_params`).
     - Se añaden etiquetas para los ejes X e Y.
     - Una línea horizontal en `y=0` separa el xT positivo (equipo local) del negativo (equipo visitante).

7. **Anotaciones adicionales:**
   - Se añaden textos que indican los valores totales de xT para ambos equipos, colocándolos en la parte superior e inferior del gráfico, cerca del valor máximo y mínimo de xT.

8. **Título del gráfico:**
   - El gráfico recibe el título **"Match Momentum by xT"**.

9. **Retorno de las estadísticas:**
   - La función devuelve un diccionario con los valores totales de xT para ambos equipos, los cuales se almacenan en un `DataFrame` (`xT_stats_df`) para análisis posteriores.



In [ ]:
def plot_Momentum(ax):
    # Set colors based on positive or negative values
    colors = [hcol if x > 0 else acol for x in Momentumdf['average_xT']]

    # making a list of munutes when goals are scored
    hgoal_list = homedf[(homedf['type'] == 'Goal') & (~homedf['qualifiers'].str.contains('OwnGoal'))]['minute'].tolist()
    agoal_list = awaydf[(awaydf['type'] == 'Goal') & (~awaydf['qualifiers'].str.contains('OwnGoal'))]['minute'].tolist()
    hog_list = homedf[(homedf['type'] == 'Goal') & (homedf['qualifiers'].str.contains('OwnGoal'))]['minute'].tolist()
    aog_list = awaydf[(awaydf['type'] == 'Goal') & (awaydf['qualifiers'].str.contains('OwnGoal'))]['minute'].tolist()
    hred_list = homedf[homedf['qualifiers'].str.contains('Red|SecondYellow')]['minute'].tolist()
    ared_list = awaydf[awaydf['qualifiers'].str.contains('Red|SecondYellow')]['minute'].tolist()

    # plotting scatters when goals are scored
    highest_xT = Momentumdf['average_xT'].max()
    lowest_xT = Momentumdf['average_xT'].min()
    highest_minute = Momentumdf['minute'].max()
    hscatter_y = [highest_xT]*len(hgoal_list)
    ascatter_y = [lowest_xT]*len(agoal_list)
    hogscatter_y = [highest_xT]*len(aog_list)
    aogscatter_y = [lowest_xT]*len(hog_list)
    hred_y = [highest_xT]*len(hred_list)
    ared_y = [lowest_xT]*len(ared_list)

    ax.text((45/2), lowest_xT, 'First Half', color='gray', fontsize=20, alpha=0.25, va='center', ha='center')
    ax.text((45+(45/2)), lowest_xT, 'Second Half', color='gray', fontsize=20, alpha=0.25, va='center', ha='center')

    ax.scatter(hgoal_list, hscatter_y, s=250, c='None', edgecolor='green', hatch='////', marker='o')
    ax.scatter(agoal_list, ascatter_y, s=250, c='None', edgecolor='green', hatch='////', marker='o')
    ax.scatter(hog_list, aogscatter_y, s=250, c='None', edgecolor='orange', hatch='////', marker='o')
    ax.scatter(aog_list, hogscatter_y, s=250, c='None', edgecolor='orange', hatch='////', marker='o')
    ax.scatter(hred_list, hred_y, s=250, c='None', edgecolor='red', hatch='////', marker='s')
    ax.scatter(ared_list, ared_y, s=250, c='None', edgecolor='red', hatch='////', marker='s')

    # Creating the bar plot
    ax.bar(Momentumdf['minute'], Momentumdf['average_xT'], color=colors)
    ax.set_xticks(range(0, len(Momentumdf['minute']), 5))
    ax.axvline(45, color='gray', linewidth=2, linestyle='dotted')
    # ax.axvline(90, color='gray', linewidth=2, linestyle='dotted')
    ax.set_facecolor(bg_color)
    # Hide spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    # # Hide ticks
    ax.tick_params(axis='both', which='both', length=0)
    ax.tick_params(axis='x', colors=line_color)
    ax.tick_params(axis='y', colors=line_color)
    # Add labels and title
    ax.set_xlabel('Minute', color=line_color, fontsize=20)
    ax.set_ylabel('Avg. xT per minute', color=line_color, fontsize=20)
    ax.axhline(y=0, color=line_color, alpha=1, linewidth=2)

    ax.text(highest_minute+1,highest_xT, f"{hteamName}\nxT: {hxT}", color=hcol, fontsize=20, va='bottom', ha='left')
    ax.text(highest_minute+1,lowest_xT,  f"{ateamName}\nxT: {axT}", color=acol, fontsize=20, va='top', ha='left')

    ax.set_title('Match Momentum by xT', color=line_color, fontsize=30, fontweight='bold')

    home_data = {
        'Team_Name': hteamName,
        'xT': hxT
    }
    
    away_data = {
        'Team_Name': ateamName,
        'xT': axT
    }
    
    return [home_data, away_data]

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
plot_Momentum(ax)
xT_stats = plot_Momentum(ax)
xT_stats_df = pd.DataFrame(xT_stats)

In [ ]:
xT_stats_df.head()

# Estadísticas de partido (Match Stats)

## Passing Stats

El código está diseñado para calcular diversas estadísticas de pases y acciones de dribbling en un partido de fútbol. Se están considerando varios aspectos como la precisión de los pases, los centros, los saques de esquina, los saques de puerta, y los intentos de regate (TakeOns). Aquí te explico cada parte del código y cómo se está utilizando.

### Explicación del código:

1. **Posesión (%)**:
   - Se calculan los pases realizados por el equipo local y visitante.
   - La posesión se define como el porcentaje de pases de cada equipo sobre el total de pases en el partido.

2. **Field Tilt (%)**:
   - Se calcula la posesión avanzada de cada equipo (toques en la zona ofensiva, más allá de los 70 metros del campo).

3. **Total de Pases y Pases Acertados**:
   - Se calculan los pases totales y los pases precisos (pases completados con éxito).

4. **Pases Acertados (sin incluir el tercio defensivo)**:
   - Filtra los pases exitosos excluyendo aquellos que terminan en el tercio defensivo del campo, enfocándose en pases más avanzados o con mayor valor ofensivo.

5. **Pases Largos y Pases Largos Acertados**:
   - Se identifican los pases largos, excluyendo centros y saques de esquina.
   - Luego se calculan cuántos de esos pases largos fueron completados con éxito.

6. **Centros y Centros Acertados**:
   - Se contabilizan los centros realizados y los centros completados con éxito.

7. **Freekick (Tiros libres), Saques de Esquina y Saques de Banda**:
   - Se calculan el número de pases derivados de tiros libres, saques de esquina y saques de banda.

8. **Dribblings (Take-Ons) y Dribblings Exitosos**:
   - Calcula el número de intentos de regate y aquellos que fueron exitosos.

9. **Longitud Media de los Saques de Puerta**:
   - Se usa una función para extraer la longitud de los saques de puerta desde la columna de calificadores (`qualifiers`), calculando el promedio de la longitud de los saques.

### Detalles adicionales:
- **Extracción de la longitud del saque de puerta**:
   - Se convierte la columna `qualifiers` que contiene listas de diccionarios en cada fila y se extrae el valor de longitud (si existe) de cada saque de puerta.
   - Este valor es importante para entender cómo un equipo utiliza sus saques de puerta, ya sea jugando en corto o lanzando balones largos hacia áreas más avanzadas.

Este enfoque está bien optimizado para obtener métricas detalladas sobre el comportamiento de los equipos en términos de pases y acciones de juego. Estas estadísticas pueden ser muy útiles para análisis de rendimiento y estrategias dentro de un partido de fútbol.

In [ ]:
# Passing Stats

#Possession%
hpossdf = df[(df['teamName']==hteamName) & (df['type']=='Pass')]
apossdf = df[(df['teamName']==ateamName) & (df['type']=='Pass')]
hposs = round((len(hpossdf)/(len(hpossdf)+len(apossdf)))*100,2)
aposs = round((len(apossdf)/(len(hpossdf)+len(apossdf)))*100,2)
#Field Tilt%
hftdf = df[(df['teamName']==hteamName) & (df['isTouch']==1) & (df['x']>=70)]
aftdf = df[(df['teamName']==ateamName) & (df['isTouch']==1) & (df['x']>=70)]
hft = round((len(hftdf)/(len(hftdf)+len(aftdf)))*100,2)
aft = round((len(aftdf)/(len(hftdf)+len(aftdf)))*100,2)
#Total Passes
htotalPass = len(df[(df['teamName']==hteamName) & (df['type']=='Pass')])
atotalPass = len(df[(df['teamName']==ateamName) & (df['type']=='Pass')])
#Accurate Pass
hAccPass = len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['outcomeType']=='Successful')])
aAccPass = len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['outcomeType']=='Successful')])
#Accurate Pass (without defensive third)
hAccPasswdt = len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['outcomeType']=='Successful') & (df['endX']>35)])
aAccPasswdt = len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['outcomeType']=='Successful') & (df['endX']>35)])
#LongBall
hLongB = len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Longball')) & (~df['qualifiers'].str.contains('Corner')) & (~df['qualifiers'].str.contains('Cross'))])
aLongB = len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Longball')) & (~df['qualifiers'].str.contains('Corner')) & (~df['qualifiers'].str.contains('Cross'))])
#Accurate LongBall
hAccLongB = len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Longball')) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('Corner')) & (~df['qualifiers'].str.contains('Cross'))])
aAccLongB = len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Longball')) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('Corner')) & (~df['qualifiers'].str.contains('Cross'))])
#Crosses
hCrss= len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Cross'))])
aCrss= len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Cross'))])
#Accurate Crosses
hAccCrss= len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Cross')) & (df['outcomeType']=='Successful')])
aAccCrss= len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Cross')) & (df['outcomeType']=='Successful')])
#Freekick
hfk= len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Freekick'))])
afk= len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Freekick'))])
#Corner
hCor= len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Corner'))])
aCor= len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Corner'))])
#ThrowIn
htins= len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('ThrowIn'))])
atins= len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('ThrowIn'))])
#GoalKicks
hglkk= len(df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('GoalKick'))])
aglkk= len(df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('GoalKick'))])
#Dribbling
htotalDrb = len(df[(df['teamName']==hteamName) & (df['type']=='TakeOn') & (df['qualifiers'].str.contains('Offensive'))])
atotalDrb = len(df[(df['teamName']==ateamName) & (df['type']=='TakeOn') & (df['qualifiers'].str.contains('Offensive'))])
#Accurate TakeOn
hAccDrb = len(df[(df['teamName']==hteamName) & (df['type']=='TakeOn') & (df['qualifiers'].str.contains('Offensive')) & (df['outcomeType']=='Successful')])
aAccDrb = len(df[(df['teamName']==ateamName) & (df['type']=='TakeOn') & (df['qualifiers'].str.contains('Offensive')) & (df['outcomeType']=='Successful')])
#GoalKick Length
home_goalkick = df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('GoalKick'))]
away_goalkick = df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('GoalKick'))]
import ast
# Convert 'qualifiers' column from string to list of dictionaries
home_goalkick['qualifiers'] = home_goalkick['qualifiers'].apply(ast.literal_eval)
away_goalkick['qualifiers'] = away_goalkick['qualifiers'].apply(ast.literal_eval)
# Function to extract value of 'Length'
def extract_length(qualifiers):
    for item in qualifiers:
        if 'displayName' in item['type'] and item['type']['displayName'] == 'Length':
            return float(item['value'])
    return None
# Apply the function to the 'qualifiers' column
home_goalkick['length'] = home_goalkick['qualifiers'].apply(extract_length).astype(float)
away_goalkick['length'] = away_goalkick['qualifiers'].apply(extract_length).astype(float)
hglkl = round(home_goalkick['length'].mean(),2)
aglkl = round(away_goalkick['length'].mean(),2)


## Defensive Stats

Este código calcula estadísticas defensivas clave para ambos equipos en un partido de fútbol. Aquí te explico cada métrica defensiva y su relevancia para el análisis:

### Explicación de las métricas defensivas:

1. **Tackles (Entradas)**:
   - **htkl / atkl**: Cuenta cuántas veces los jugadores de cada equipo intentaron recuperar la posesión del balón haciendo una entrada.
   
2. **Tackles Ganados (Entradas Exitosas)**:
   - **htklw / atklw**: Cuenta las entradas que resultaron en una recuperación exitosa del balón, un buen indicador de la efectividad defensiva de los jugadores.

3. **Interceptions (Intercepciones)**:
   - **hintc / aintc**: Número de veces que un jugador cortó un pase del oponente. Las intercepciones reflejan la capacidad de un equipo para anticipar los movimientos del rival.

4. **Clearances (Despejes)**:
   - **hclr / aclr**: Cantidad de veces que un jugador despejó el balón para alejar el peligro del área propia. Los despejes son importantes cuando un equipo está bajo presión.

5. **Aerial Duels (Duelos Aéreos)**:
   - **harl / aarl**: Cuenta los duelos aéreos en los que un jugador saltó para competir por el balón.
   
6. **Aerial Duels Ganados (Duelos Aéreos Ganados)**:
   - **harlw / aarlw**: De esos duelos aéreos, cuántos fueron ganados por un jugador de cada equipo. Esta métrica mide la capacidad de los jugadores para imponerse en el juego aéreo.

7. **Ball Recoveries (Recuperaciones de Balón)**:
   - **hblrc / ablrc**: Cuenta las veces que un jugador recuperó el balón tras un mal control o pérdida de posesión del equipo contrario.

8. **Blocked Passes (Pases Bloqueados)**:
   - **hblkp / ablkp**: Número de veces que un pase del equipo contrario fue bloqueado por un defensor.

9. **Offside Given (Fueras de Juego)**:
   - **hofs / aofs**: Cuenta las veces que un jugador fue sorprendido en fuera de juego, una indicación de la disciplina defensiva del rival.

10. **Fouls (Faltas Cometidas)**:
    - **hfoul / afoul**: Número de faltas cometidas por los jugadores de cada equipo, lo que puede ser un indicador del nivel de agresividad defensiva.

Estas estadísticas te dan una visión clara de cómo cada equipo se desempeñó defensivamente durante el partido, proporcionando datos valiosos para evaluar su solidez y eficiencia a la hora de frenar los ataques del rival.

In [ ]:
# Defensive Stats

#Tackles
htkl = len(df[(df['teamName']==hteamName) & (df['type']=='Tackle')])
atkl = len(df[(df['teamName']==ateamName) & (df['type']=='Tackle')])
#Tackles Won
htklw = len(df[(df['teamName']==hteamName) & (df['type']=='Tackle') & (df['outcomeType']=='Successful')])
atklw = len(df[(df['teamName']==ateamName) & (df['type']=='Tackle') & (df['outcomeType']=='Successful')])
#Interceptions
hintc= len(df[(df['teamName']==hteamName) & (df['type']=='Interception')])
aintc= len(df[(df['teamName']==ateamName) & (df['type']=='Interception')])
#Clearances
hclr= len(df[(df['teamName']==hteamName) & (df['type']=='Clearance')])
aclr= len(df[(df['teamName']==ateamName) & (df['type']=='Clearance')])
#Aerials
harl= len(df[(df['teamName']==hteamName) & (df['type']=='Aerial')])
aarl= len(df[(df['teamName']==ateamName) & (df['type']=='Aerial')])
#Aerials Wins
harlw= len(df[(df['teamName']==hteamName) & (df['type']=='Aerial') & (df['outcomeType']=='Successful')])
aarlw= len(df[(df['teamName']==ateamName) & (df['type']=='Aerial') & (df['outcomeType']=='Successful')])
#BallRecovery
hblrc= len(df[(df['teamName']==hteamName) & (df['type']=='BallRecovery')])
ablrc= len(df[(df['teamName']==ateamName) & (df['type']=='BallRecovery')])
#BlockedPass
hblkp= len(df[(df['teamName']==hteamName) & (df['type']=='BlockedPass')])
ablkp= len(df[(df['teamName']==ateamName) & (df['type']=='BlockedPass')])
#OffsideGiven
hofs= len(df[(df['teamName']==hteamName) & (df['type']=='OffsideGiven')])
aofs= len(df[(df['teamName']==ateamName) & (df['type']=='OffsideGiven')])
#Fouls
hfoul= len(df[(df['teamName']==hteamName) & (df['type']=='Foul')])
afoul= len(df[(df['teamName']==ateamName) & (df['type']=='Foul')])

El código aquí calcula varias métricas clave relacionadas con la presión y las secuencias de pases durante el partido. Aquí te explico las métricas que estás calculando:

### PPDA (Passes per Defensive Action)
El **PPDA** mide la presión que un equipo aplica sobre su rival. Es la cantidad de pases permitidos por un equipo antes de realizar una acción defensiva en el tercio ofensivo.

1. **home_ppda / away_ppda**:
   - Calcula el PPDA para el equipo local y visitante.
   - **PPDA = (Total de pases exitosos del equipo contrario en la zona defensiva) / (Total de acciones defensivas en la zona de ataque)**.
   - Un PPDA bajo indica que un equipo está aplicando mucha presión, mientras que un PPDA alto indica que el equipo permite que el rival juegue con más libertad antes de intervenir defensivamente.

### Secuencias de Pases

2. **Promedio de Pases por Secuencia (PPS)**:
   - **PPS_home / PPS_away**: Esta métrica calcula el promedio de pases por secuencia de posesión.
   - Se agrupan los pases de cada equipo por su posesión y se calcula el promedio de cuántos pases hubo en cada secuencia.
   - Esto puede mostrar si un equipo tiende a mantener más posesión con muchos pases o si juega de manera más directa con menos pases por secuencia.

3. **Secuencias con 10 o más pases**:
   - **pass_seq_10_more_home / pass_seq_10_more_away**: Este valor indica cuántas veces un equipo tuvo una secuencia de posesión con al menos 10 pases.
   - Una alta cantidad de secuencias de 10 o más pases indica que un equipo mantuvo posesiones largas y controladas.

Estas métricas son útiles para evaluar el estilo de juego de los equipos, su nivel de presión y control en el partido.

In [ ]:
# PPDA
home_def_acts = df[(df['teamName']==hteamName) & (df['type'].str.contains('Interception|Foul|Challenge|BlockedPass|Tackle')) & (df['x']>35)]
away_def_acts = df[(df['teamName']==ateamName) & (df['type'].str.contains('Interception|Foul|Challenge|BlockedPass|Tackle')) & (df['x']>35)]
home_pass = df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['outcomeType']=='Successful') & (df['x']<70)]
away_pass = df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['outcomeType']=='Successful') & (df['x']<70)]
home_ppda = round((len(away_pass)/len(home_def_acts)), 2)
away_ppda = round((len(home_pass)/len(away_def_acts)), 2)

# Average Passes per Sequence
pass_df_home = df[(df['type'] == 'Pass') & (df['teamName']==hteamName)]
pass_counts_home = pass_df_home.groupby('possession_id').size()
PPS_home = pass_counts_home.mean().round()
pass_df_away = df[(df['type'] == 'Pass') & (df['teamName']==ateamName)]
pass_counts_away = pass_df_away.groupby('possession_id').size()
PPS_away = pass_counts_away.mean().round()

# Number of Sequence with 10+ Passes
possessions_with_10_or_more_passes = pass_counts_home[pass_counts_home >= 10]
pass_seq_10_more_home = possessions_with_10_or_more_passes.count()
possessions_with_10_or_more_passes = pass_counts_away[pass_counts_away >= 10]
pass_seq_10_more_away = possessions_with_10_or_more_passes.count()

path_eff1 = [path_effects.Stroke(linewidth=1.5, foreground=line_color), path_effects.Normal()]

La función `plotting_match_stats` que hemos implementado genera un gráfico con barras horizontales para visualizar diversas estadísticas del partido, dividiendo las métricas entre los dos equipos (local y visitante). A continuación, te explico cómo se visualiza cada parte y qué representa:

1. **Posicionamiento del campo y cuadro de título**:
   - Se dibuja un campo de fútbol con `Pitch` y se posiciona el área de las estadísticas, donde el encabezado "Match Stats" aparece destacado.
   
2. **Barras de estadísticas**:
   - Se grafican las estadísticas normalizadas para ambos equipos (local y visitante) en formato horizontal, lo que permite una fácil comparación. El uso de barras hacia la izquierda para el equipo local y hacia la derecha para el equipo visitante permite una visualización clara.
   
3. **Texto de las estadísticas**:
   - A cada lado de las barras, se muestran los valores exactos de las estadísticas como porcentaje de posesión, pases, tackles, etc., tanto para el equipo local como para el visitante.

4. **Métricas calculadas**:
   - **Posesión** (%), **Field Tilt** (%), **Total de Pases**, **Longballs** (balones largos), **Corners**, **Tackles** (entradas), **Intercepciones**, **Clearances** (despejes), **Aerial Duels** (duelos aéreos), y **PPDA** son algunas de las métricas clave representadas.
   - Estas estadísticas ofrecen una imagen completa de cómo se desempeñaron ambos equipos en distintas áreas del juego.

5. **Datos Devueltos**:
   - Después de crear el gráfico, se genera un DataFrame (`general_match_stats_df`) que contiene todas las estadísticas calculadas para ambos equipos, lo que puede ser útil para posteriores análisis o reportes.

Este enfoque visual te permite comparar rápidamente el desempeño de los equipos en distintas áreas claves del partido, y además el DataFrame generado te permite utilizar estos datos de forma estructurada.

In [ ]:
def plotting_match_stats(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=bg_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-5, 68.5)

    # plotting the headline box
    head_y = [62,68,68,62]
    head_x = [0,0,105,105]
    ax.fill(head_x, head_y, 'orange')
    ax.text(52.5,64.5, "Match Stats", ha='center', va='center', color=line_color, fontsize=25, fontweight='bold', path_effects=path_eff)

    # Stats bar diagram
    stats_title = [58, 58-(1*6), 58-(2*6), 58-(3*6), 58-(4*6), 58-(5*6), 58-(6*6), 58-(7*6), 58-(8*6), 58-(9*6), 58-(10*6)] # y co-ordinate values of the bars
    stats_home = [hposs, hft, htotalPass, hLongB, hCor, hglkl, htkl, hintc, hclr, harl, home_ppda]
    stats_away = [aposs, aft, atotalPass, aLongB, aCor, aglkl, atkl, aintc, aclr, aarl, away_ppda]

    stats_normalized_home = [-(hposs/(hposs+aposs))*50, -(hft/(hft+aft))*50, -(htotalPass/(htotalPass+atotalPass))*50,
                                        -(hLongB/(hLongB+aLongB))*50, -(hCor/(hCor+aCor))*50, -(hglkl/(hglkl+aglkl))*50, -(htkl/(htkl+atkl))*50,       # put a (-) sign before each value so that the
                                        -(hintc/(hintc+aintc))*50, -(hclr/(hclr+aclr))*50, -(harl/(harl+aarl))*50, -(home_ppda/(home_ppda+away_ppda))*50]          # home stats bar shows in the opposite of away
    stats_normalized_away = [(aposs/(hposs+aposs))*50, (aft/(hft+aft))*50, (atotalPass/(htotalPass+atotalPass))*50,
                                        (aLongB/(hLongB+aLongB))*50, (aCor/(hCor+aCor))*50, (aglkl/(hglkl+aglkl))*50, (atkl/(htkl+atkl))*50,
                                        (aintc/(hintc+aintc))*50, (aclr/(hclr+aclr))*50, (aarl/(harl+aarl))*50, (away_ppda/(home_ppda+away_ppda))*50]

    start_x = 52.5
    ax.barh(stats_title, stats_normalized_home, height=4, color=hcol, left=start_x)
    ax.barh(stats_title, stats_normalized_away, height=4, left=start_x, color=acol)
    # Turn off axis-related elements
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)
    ax.set_xticks([])
    ax.set_yticks([])

    # Plotting the texts
    ax.text(52.5, 58, "Possession", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(1*6), "Field Tilt", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(2*6), "Passes (Acc.)", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(3*6), "LongBalls (Acc.)", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(4*6), "Corners", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(5*6), "Avg. Goalkick len.", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(6*6), "Tackles (Wins)", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(7*6), "Interceptions", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(8*6), "Clearence", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(9*6), "Aerials (Wins)", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)
    ax.text(52.5, 58-(10*6), "PPDA", color=bg_color, fontsize=17, ha='center', va='center', fontweight='bold', path_effects=path_eff1)

    ax.text(0, 58, f"{round(hposs)}%", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(1*6), f"{round(hft)}%", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(2*6), f"{htotalPass}({hAccPass})", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(3*6), f"{hLongB}({hAccLongB})", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(4*6), f"{hCor}", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(5*6), f"{hglkl} m", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(6*6), f"{htkl}({htklw})", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(7*6), f"{hintc}", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(8*6), f"{hclr}", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(9*6), f"{harl}({harlw})", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')
    ax.text(0, 58-(10*6), f"{home_ppda}", color=line_color, fontsize=20, ha='right', va='center', fontweight='bold')

    ax.text(105, 58, f"{round(aposs)}%", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(1*6), f"{round(aft)}%", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(2*6), f"{atotalPass}({aAccPass})", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(3*6), f"{aLongB}({aAccLongB})", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(4*6), f"{aCor}", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(5*6), f"{aglkl} m", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(6*6), f"{atkl}({atklw})", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(7*6), f"{aintc}", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(8*6), f"{aclr}", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(9*6), f"{aarl}({aarlw})", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')
    ax.text(105, 58-(10*6), f"{away_ppda}", color=line_color, fontsize=20, ha='left', va='center', fontweight='bold')

    home_data = {
        'Team_Name': hteamName,
        'Possession_%': hposs,
        'Field_Tilt_%': hft,
        'Total_Passes': htotalPass,
        'Accurate_Passes': hAccPass,
        'Longballs': hLongB,
        'Accurate_Longballs': hAccLongB,
        'Corners': hCor,
        'Avg.GoalKick_Length': hglkl,
        'Tackles': htkl,
        'Tackles_Won': htklw,
        'Interceptions': hintc,
        'Clearances': hclr,
        'Aerial_Duels': harl,
        'Aerial_Duels_Won': harlw,
        'Passes_Per_Defensive_Actions(PPDA)': home_ppda,
        'Average_Passes_Per_Sequences': PPS_home,
        '10+_Passing_Sequences': pass_seq_10_more_home
    }
    
    away_data = {
        'Team_Name': ateamName,
        'Possession_%': aposs,
        'Field_Tilt_%': aft,
        'Total_Passes': atotalPass,
        'Accurate_Passes': aAccPass,
        'Longballs': aLongB,
        'Accurate_Longballs': aAccLongB,
        'Corners': aCor,
        'Avg.GoalKick_Length': aglkl,
        'Tackles': atkl,
        'Tackles_Won': atklw,
        'Interceptions': aintc,
        'Clearances': aclr,
        'Aerial_Duels': aarl,
        'Aerial_Duels_Won': aarlw,
        'Passes_Per_Defensive_Actions(PPDA)': away_ppda,
        'Average_Passes_Per_Sequences': PPS_away,
        '10+_Passing_Sequences': pass_seq_10_more_away
    }
    
    return [home_data, away_data]

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
general_match_stats = plotting_match_stats(ax)
general_match_stats_df = pd.DataFrame(general_match_stats)

In [ ]:
general_match_stats_df.head()

## Entradas en el último tercio (Final Third Entry)

The function `Final_third_entry` generates a visual representation and summary of the final third entries made by a team during a football match. A final third entry is defined as a successful pass or carry that starts outside the final third of the pitch (before the 70-yard mark) and ends within it.

### How the function works:
1. **Data Filtering**:
   - It filters the data for both **passes** and **carries** that meet the condition of starting outside and ending inside the final third, excluding free kicks.
   
2. **Visualization Setup**:
   - A pitch is drawn, with specific regions for the left, middle, and right zones of the final third.
   - The number of entries is divided into left, center, and right zones, and their percentages are calculated.

3. **Plotting Passes and Carries**:
   - **Passes** are shown with lines from the start to the end point, while **carries** are displayed with dashed arrows.
   - It distinguishes between passes and carries to the final third with visual markers and text annotations.

4. **Text Annotations**:
   - The function dynamically adds the counts and percentages for each zone (left, center, right) and displays additional information such as the total number of entries, entries by pass, and entries by carry.

5. **Return Values**:
   - The function returns a dictionary with detailed statistics about the final third entries, including the team name, total entries, breakdown of entries by zone, and method of entry (pass or carry).

This visualization effectively communicates the team's attacking patterns, showing how often and from which areas of the pitch they successfully entered the final third, making it a valuable tool for match analysis.

In [ ]:
def Final_third_entry(ax, team_name, col):
    # Final third Entry means passes or carries which has started outside the Final third and ended inside the final third
    dfpass = df[(df['teamName']==team_name) & (df['type']=='Pass') & (df['x']<70) & (df['endX']>=70) & (df['outcomeType']=='Successful') &
                (~df['qualifiers'].str.contains('Freekick'))]
    dfcarry = df[(df['teamName']==team_name) & (df['type']=='Carry') & (df['x']<70) & (df['endX']>=70)]
    pitch = Pitch(pitch_type='uefa', pitch_color=bg_color, line_color=line_color, linewidth=2,
                          corner_arcs=True)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    # ax.set_ylim(-0.5, 68.5)

    if team_name == ateamName:
        ax.invert_xaxis()
        ax.invert_yaxis()

    pass_count = len(dfpass) + len(dfcarry)

    # calculating the counts
    left_entry = len(dfpass[dfpass['y']>=45.33]) + len(dfcarry[dfcarry['y']>=45.33])
    mid_entry = len(dfpass[(dfpass['y']>=22.67) & (dfpass['y']<45.33)]) + len(dfcarry[(dfcarry['y']>=22.67) & (dfcarry['y']<45.33)])
    right_entry = len(dfpass[(dfpass['y']>=0) & (dfpass['y']<22.67)]) + len(dfcarry[(dfcarry['y']>=0) & (dfcarry['y']<22.67)])
    left_percentage = round((left_entry/pass_count)*100)
    mid_percentage = round((mid_entry/pass_count)*100)
    right_percentage = round((right_entry/pass_count)*100)

    ax.hlines(22.67, xmin=0, xmax=70, colors=line_color, linestyle='dashed', alpha=0.35)
    ax.hlines(45.33, xmin=0, xmax=70, colors=line_color, linestyle='dashed', alpha=0.35)
    ax.vlines(70, ymin=-2, ymax=70, colors=line_color, linestyle='dashed', alpha=0.55)

    # showing the texts in the pitch
    bbox_props = dict(boxstyle="round,pad=0.3", edgecolor="None", facecolor=bg_color, alpha=0.75)
    if col == hcol:
        ax.text(8, 11.335, f'{right_entry}\n({right_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 34, f'{mid_entry}\n({mid_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 56.675, f'{left_entry}\n({left_percentage}%)', color=hcol, fontsize=24, va='center', ha='center', bbox=bbox_props)
    else:
        ax.text(8, 11.335, f'{right_entry}\n({right_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 34, f'{mid_entry}\n({mid_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)
        ax.text(8, 56.675, f'{left_entry}\n({left_percentage}%)', color=acol, fontsize=24, va='center', ha='center', bbox=bbox_props)

    # plotting the passes
    pro_pass = pitch.lines(dfpass.x, dfpass.y, dfpass.endX, dfpass.endY, lw=3.5, comet=True, color=col, ax=ax, alpha=0.5)
    # plotting some scatters at the end of each pass
    pro_pass_end = pitch.scatter(dfpass.endX, dfpass.endY, s=35, edgecolor=col, linewidth=1, color=bg_color, zorder=2, ax=ax)
    # plotting carries
    for index, row in dfcarry.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), arrowstyle='->', color=col, zorder=4, mutation_scale=20, 
                                        alpha=1, linewidth=2, linestyle='--')
        ax.add_patch(arrow)

    counttext = f"{pass_count} Final Third Entries"

    # Heading and other texts
    if col == hcol:
        ax.set_title(f"{hteamName}\n{counttext}", color=line_color, fontsize=25, fontweight='bold', path_effects=path_eff)
        ax.text(87.5, 70, '<--------------- Final third --------------->', color=line_color, ha='center', va='center')
        pitch.lines(53, -2, 73, -2, lw=3, transparent=True, comet=True, color=col, ax=ax, alpha=0.5)
        ax.scatter(73,-2, s=35, edgecolor=col, linewidth=1, color=bg_color, zorder=2)
        arrow = patches.FancyArrowPatch((83, -2), (103, -2), arrowstyle='->', color=col, zorder=4, mutation_scale=20, 
                                        alpha=1, linewidth=2, linestyle='--')
        ax.add_patch(arrow)
        ax.text(63, -5, f'Entry by Pass: {len(dfpass)}', fontsize=15, color=line_color, ha='center', va='center')
        ax.text(93, -5, f'Entry by Carry: {len(dfcarry)}', fontsize=15, color=line_color, ha='center', va='center')
        
    else:
        ax.set_title(f"{ateamName}\n{counttext}", color=line_color, fontsize=25, fontweight='bold', path_effects=path_eff)
        ax.text(87.5, -2, '<--------------- Final third --------------->', color=line_color, ha='center', va='center')
        pitch.lines(53, 70, 73, 70, lw=3, transparent=True, comet=True, color=col, ax=ax, alpha=0.5)
        ax.scatter(73,70, s=35, edgecolor=col, linewidth=1, color=bg_color, zorder=2)
        arrow = patches.FancyArrowPatch((83, 70), (103, 70), arrowstyle='->', color=col, zorder=4, mutation_scale=20, 
                                        alpha=1, linewidth=2, linestyle='--')
        ax.add_patch(arrow)
        ax.text(63, 73, f'Entry by Pass: {len(dfpass)}', fontsize=15, color=line_color, ha='center', va='center')
        ax.text(93, 73, f'Entry by Carry: {len(dfcarry)}', fontsize=15, color=line_color, ha='center', va='center')

    return {
        'Team_Name': team_name,
        'Total_Final_Third_Entries': pass_count,
        'Final_Third_Entries_From_Left': left_entry,
        'Final_Third_Entries_From_Center': mid_entry,
        'Final_Third_Entries_From_Right': right_entry,
        'Entry_By_Pass': len(dfpass),
        'Entry_By_Carry': len(dfcarry)
    }

In [ ]:
fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
final_third_entry_stats_home = Final_third_entry(axs[0], hteamName, hcol)
final_third_entry_stats_away = Final_third_entry(axs[1], ateamName, acol)
final_third_entry_stats_list = []
final_third_entry_stats_list.append(final_third_entry_stats_home)
final_third_entry_stats_list.append(final_third_entry_stats_away)
final_third_entry_stats_df = pd.DataFrame(final_third_entry_stats_list)

In [ ]:
final_third_entry_stats_df.head()

## 14 Zonas y sus pases (Zone14 & Half-Space Passes)

La función `zone14hs` visualiza los pases exitosos en áreas específicas de alto valor en el campo: **Zona 14** y los **Espacios Interiores** (Half-Spaces). Aquí te explico cómo funciona:

### Elementos clave:
1. **Filtrado de datos**:
   - La función filtra los pases exitosos que no provienen de jugadas a balón parado (como córners o tiros libres) y que pertenecen al equipo especificado.

2. **Configuración de la visualización**:
   - Se dibuja el campo utilizando un estilo predefinido, y si es el equipo visitante, se invierte el eje para mantener la consistencia con la dirección de ataque de ese equipo.

3. **Detección de pases**:
   - **Zona 14** es el área central justo fuera del área penal, crucial para generar oportunidades de gol.
   - Los **Espacios Interiores** son zonas a la derecha e izquierda, fuera del área penal, a menudo utilizadas para pases clave y ataques.
   - Para cada pase que termina en estas áreas, se dibuja una línea que representa el pase y se coloca un marcador en el final del pase.
   - **Naranja** representa los pases en la Zona 14, mientras que el color del equipo (ya sea `hcol` o `acol`) se usa para los pases en los Espacios Interiores.

4. **Resaltado visual**:
   - Se sombrea el área de la **Zona 14** y los **Espacios Interiores** en el campo, lo que facilita identificar dónde se están realizando los pases.
   - Se utilizan formas y colores visualmente distintos para destacar estas áreas clave.

5. **Visualización de estadísticas**:
   - En la parte inferior izquierda del campo, se utilizan marcadores hexagonales grandes para mostrar la cantidad de pases hacia la **Zona 14** y los **Espacios Interiores**. Cada conteo se muestra con texto en negrita, junto con etiquetas como "Zone14" y "HalfSp."

6. **Anotaciones de texto**:
   - El título de la gráfica se ajusta dinámicamente para mostrar el nombre del equipo e indicar el tipo de pases que se están visualizando.

7. **Valores de retorno**:
   - La función devuelve un diccionario con estadísticas detalladas sobre los pases hacia la **Zona 14** y los **Espacios Interiores**, que incluye:
     - El nombre del equipo.
     - La cantidad total de pases en la **Zona 14**.
     - La cantidad de pases en los **Espacios Interiores**, y un desglose entre los pases hacia el lado izquierdo y derecho.

Esta función es especialmente útil para comprender los patrones de ataque de un equipo y cómo utilizan las áreas clave en el tercio final del campo.

In [ ]:
def zone14hs(ax, team_name, col):
    dfhp = df[(df['teamName']==team_name) & (df['type']=='Pass') & (df['outcomeType']=='Successful') & 
              (~df['qualifiers'].str.contains('CornerTaken|Freekick'))]
    
    pitch = Pitch(pitch_type='uefa', pitch_color=bg_color, line_color=line_color,  linewidth=2,
                          corner_arcs=True)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_facecolor(bg_color)
    if team_name == ateamName:
      ax.invert_xaxis()
      ax.invert_yaxis()

    # setting the count varibale
    z14 = 0
    hs = 0
    lhs = 0
    rhs = 0

    path_eff = [path_effects.Stroke(linewidth=3, foreground=bg_color), path_effects.Normal()]
    # iterating ecah pass and according to the conditions plotting only zone14 and half spaces passes
    for index, row in dfhp.iterrows():
        if row['endX'] >= 70 and row['endX'] <= 88.54 and row['endY'] >= 22.66 and row['endY'] <= 45.32:
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color='orange', comet=True, lw=3, zorder=3, ax=ax, alpha=0.75)
            ax.scatter(row['endX'], row['endY'], s=35, linewidth=1, color=bg_color, edgecolor='orange', zorder=4)
            z14 += 1
        if row['endX'] >= 70 and row['endY'] >= 11.33 and row['endY'] <= 22.66:
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=col, comet=True, lw=3, zorder=3, ax=ax, alpha=0.75)
            ax.scatter(row['endX'], row['endY'], s=35, linewidth=1, color=bg_color, edgecolor=col, zorder=4)
            hs += 1
            rhs += 1
        if row['endX'] >= 70 and row['endY'] >= 45.32 and row['endY'] <= 56.95:
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=col, comet=True, lw=3, zorder=3, ax=ax, alpha=0.75)
            ax.scatter(row['endX'], row['endY'], s=35, linewidth=1, color=bg_color, edgecolor=col, zorder=4)
            hs += 1
            lhs += 1

    # coloring those zones in the pitch
    y_z14 = [22.66, 22.66, 45.32, 45.32]
    x_z14 = [70, 88.54, 88.54, 70]
    ax.fill(x_z14, y_z14, 'orange', alpha=0.2, label='Zone14')

    y_rhs = [11.33, 11.33, 22.66, 22.66]
    x_rhs = [70, 105, 105, 70]
    ax.fill(x_rhs, y_rhs, col, alpha=0.2, label='HalfSpaces')

    y_lhs = [45.32, 45.32, 56.95, 56.95]
    x_lhs = [70, 105, 105, 70]
    ax.fill(x_lhs, y_lhs, col, alpha=0.2, label='HalfSpaces')

    # showing the counts in an attractive way
    z14name = "Zone14"
    hsname = "HalfSp"
    z14count = f"{z14}"
    hscount = f"{hs}"
    ax.scatter(16.46, 13.85, color=col, s=15000, edgecolor=line_color, linewidth=2, alpha=1, marker='h')
    ax.scatter(16.46, 54.15, color='orange', s=15000, edgecolor=line_color, linewidth=2, alpha=1, marker='h')
    ax.text(16.46, 13.85-4, hsname, fontsize=20, color=line_color, ha='center', va='center', path_effects=path_eff)
    ax.text(16.46, 54.15-4, z14name, fontsize=20, color=line_color, ha='center', va='center', path_effects=path_eff)
    ax.text(16.46, 13.85+2, hscount, fontsize=40, color=line_color, ha='center', va='center', path_effects=path_eff)
    ax.text(16.46, 54.15+2, z14count, fontsize=40, color=line_color, ha='center', va='center', path_effects=path_eff)

    # Headings and other texts
    if col == hcol:
      ax.set_title(f"{hteamName}\nZone14 & Halfsp. Pass", color=line_color, fontsize=25, fontweight='bold')
    else:
      ax.set_title(f"{ateamName}\nZone14 & Halfsp. Pass", color=line_color, fontsize=25, fontweight='bold')

    return {
        'Team_Name': team_name,
        'Total_Passes_Into_Zone14': z14,
        'Passes_Into_Halfspaces': hs,
        'Passes_Into_Left_Halfspaces': lhs,
        'Passes_Into_Right_Halfspaces': rhs
    }

In [ ]:
fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
zonal_passing_stats_home = zone14hs(axs[0], hteamName, hcol)
zonal_passing_stats_away = zone14hs(axs[1], ateamName, acol)
zonal_passing_stats_list = []
zonal_passing_stats_list.append(zonal_passing_stats_home)
zonal_passing_stats_list.append(zonal_passing_stats_away)
zonal_passing_stats_df = pd.DataFrame(zonal_passing_stats_list)

In [ ]:
zonal_passing_stats_df.head()

## Zonas de finalizacion de los pases (Pass Ending Zone)

La función `Pass_end_zone` visualiza un mapa de calor de los puntos finales de los pases exitosos para un equipo específico en un partido de fútbol. Aquí te explico en español cómo funciona y su propósito:

### 1. **Configuración del mapa de color personalizado**:
   - **Colores personalizados**: Los mapas de calor se generan con un color degradado entre el color de fondo (representando el campo) y el color del equipo (ya sea el color del equipo local o visitante).
   - Se utiliza una escala personalizada llamada "Pearl Earring" para crear una transición suave entre los colores, dándole un efecto visual atractivo.

### 2. **Filtrado de los datos**:
   - Se filtran los pases exitosos (`outcomeType` = 'Successful') del equipo en cuestión para obtener las coordenadas finales de cada pase exitoso.

### 3. **Visualización de la cancha**:
   - La cancha de fútbol se dibuja utilizando una visualización detallada con arcos de esquina y porterías, lo que facilita ubicar las zonas clave de los pases en el contexto del campo.

### 4. **Generación del mapa de calor**:
   - Los puntos finales de los pases se agrupan en una cuadrícula (de 6x5) para calcular la densidad de los pases en cada área. Esto permite ver qué zonas del campo se han usado más durante el juego.
   - El mapa de calor utiliza el colormap personalizado para mostrar la concentración de los pases, con zonas más oscuras o saturadas representando áreas con mayor densidad de pases.

### 5. **Detalles adicionales**:
   - Además del mapa de calor, se añade una capa de dispersión que muestra cada punto final de pase como un pequeño marcador gris, proporcionando más contexto sobre la distribución de los pases.
   - También se agregan etiquetas sobre el mapa de calor, mostrando los porcentajes de concentración en cada celda de la cuadrícula, lo que facilita la interpretación visual de los datos.

### 6. **Título dinámico**:
   - El título del gráfico se ajusta dinámicamente para reflejar el nombre del equipo y resaltar que se trata de las "Zonas de finalización de los pases".

### 7. **Visualización comparativa**:
   - En este caso, la función genera dos subgráficos, uno para cada equipo, lo que permite comparar visualmente las zonas donde cada equipo completa más pases exitosos.

Este tipo de visualización es útil para los analistas deportivos, ya que proporciona información sobre las zonas del campo donde un equipo es más efectivo con sus pases, lo que puede ayudar a identificar patrones de juego o áreas de ataque clave.

In [ ]:
# setting the custom colormap
pearl_earring_cmaph = LinearSegmentedColormap.from_list("Pearl Earring - 10 colors",  [bg_color, hcol], N=20)
pearl_earring_cmapa = LinearSegmentedColormap.from_list("Pearl Earring - 10 colors",  [bg_color, acol], N=20)

path_eff = [path_effects.Stroke(linewidth=3, foreground=bg_color), path_effects.Normal()]

# Getting heatmap of all the end point of the successful Passes
def Pass_end_zone(ax, team_name, cm):
    pez = df[(df['teamName'] == team_name) & (df['type'] == 'Pass') & (df['outcomeType'] == 'Successful')]
    pitch = Pitch(pitch_type='uefa', line_color=line_color, goal_type='box', goal_alpha=.5, corner_arcs=True, line_zorder=2, pitch_color=bg_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    if team_name == ateamName:
      ax.invert_xaxis()
      ax.invert_yaxis()

    pearl_earring_cmap = cm
    # binning the data points
    bin_statistic = pitch.bin_statistic(pez.endX, pez.endY, bins=(6, 5), normalize=True)
    pitch.heatmap(bin_statistic, ax=ax, cmap=pearl_earring_cmap, edgecolors=bg_color)
    pitch.scatter(df.endX, df.endY, c='gray', s=5, ax=ax)
    labels = pitch.label_heatmap(bin_statistic, color=line_color, fontsize=25, ax=ax, ha='center', va='center', str_format='{:.0%}', path_effects=path_eff)

    # Headings and other texts
    if team_name == hteamName:
      ax.set_title(f"{hteamName}\nPass End Zone", color=line_color, fontsize=25, fontweight='bold', path_effects=path_eff)
    else:
      ax.set_title(f"{ateamName}\nPass End Zone", color=line_color, fontsize=25, fontweight='bold', path_effects=path_eff)

fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
Pass_end_zone(axs[0], hteamName, pearl_earring_cmaph)
Pass_end_zone(axs[1], ateamName, pearl_earring_cmapa)

## Zona de creación de oportunidades (Chances Creating Zone)

La función `Chance_creating_zone` tiene como objetivo visualizar las zonas del campo donde se han creado oportunidades clave (pases clave y asistencias) en un partido de fútbol. Aquí te lo explico en español:

### 1. **Configuración del mapa de color**:
   - Se utiliza un colormap personalizado llamado "Pearl Earring", que cambia entre el color de fondo del campo y el color del equipo seleccionado (local o visitante) para mostrar el mapa de calor de las zonas donde se originan las oportunidades.

### 2. **Filtrado de datos**:
   - La función filtra los pases clave que hayan resultado en una oportunidad de gol (`KeyPass`) para el equipo seleccionado. Se incluye una distinción entre las asistencias y los pases clave que no resultaron en un gol.

### 3. **Visualización de la cancha**:
   - Se dibuja la cancha de fútbol completa, y los pases clave se trazan como líneas. Los puntos de finalización de los pases se marcan con diferentes colores:
     - **Verde**: Si el pase resultó en una asistencia de gol.
     - **Violeta**: Si el pase fue clave pero no resultó en un gol.
   
### 4. **Generación del mapa de calor**:
   - Se crea un mapa de calor que muestra la concentración de los pases clave desde las distintas zonas del campo. Las áreas más usadas para generar oportunidades se destacan según la densidad de pases clave realizados desde esas zonas.

### 5. **Conteo de oportunidades creadas**:
   - La función cuenta el total de pases clave realizados por el equipo y lo muestra en el gráfico, lo que permite tener una idea del volumen de oportunidades generadas.

### 6. **Etiquetas y textos**:
   - El gráfico incluye leyendas que explican los colores de los pases clave (verde para asistencias y violeta para pases clave).
   - Se añaden títulos y etiquetas descriptivas para destacar el número total de oportunidades creadas y el nombre del equipo que se analiza.

### 7. **Estructura de retorno**:
   - La función devuelve un diccionario con el nombre del equipo y el total de oportunidades creadas durante el partido, proporcionando una salida útil para informes posteriores o análisis adicionales.

Esta visualización es especialmente útil para comprender las zonas del campo desde donde los equipos crean más oportunidades de gol y cómo distribuyen sus pases clave, lo que puede ser clave para identificar patrones ofensivos de juego.

In [ ]:
# setting the custom colormap
pearl_earring_cmaph = LinearSegmentedColormap.from_list("Pearl Earring - 10 colors", [bg_color, hcol], N=20)
pearl_earring_cmapa = LinearSegmentedColormap.from_list("Pearl Earring - 10 colors", [bg_color, acol], N=20)

path_eff = [path_effects.Stroke(linewidth=3, foreground=bg_color), path_effects.Normal()]

def Chance_creating_zone(ax, team_name, cm, col):
    ccp = df[(df['qualifiers'].str.contains('KeyPass')) & (df['teamName']==team_name)]
    pitch = Pitch(pitch_type='uefa', line_color=line_color, corner_arcs=True, line_zorder=2, pitch_color=bg_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    if team_name == ateamName:
      ax.invert_xaxis()
      ax.invert_yaxis()

    cc = 0
    pearl_earring_cmap = cm
    # bin_statistic = pitch.bin_statistic_positional(df.x, df.y, statistic='count', positional='full', normalize=False)
    bin_statistic = pitch.bin_statistic(ccp.x, ccp.y, bins=(6,5), statistic='count', normalize=False)
    pitch.heatmap(bin_statistic, ax=ax, cmap=pearl_earring_cmap, edgecolors='#f8f8f8')
    # pitch.scatter(ccp.x, ccp.y, c='gray', s=5, ax=ax)
    for index, row in ccp.iterrows():
      if 'IntentionalGoalAssist' in row['qualifiers']:
        pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=green, comet=True, lw=3, zorder=3, ax=ax)
        ax.scatter(row['endX'], row['endY'], s=35, linewidth=1, color=bg_color, edgecolor=green, zorder=4)
        cc += 1
      else :
        pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=violet, comet=True, lw=3, zorder=3, ax=ax)
        ax.scatter(row['endX'], row['endY'], s=35, linewidth=1, color=bg_color, edgecolor=violet, zorder=4)
        cc += 1
    labels = pitch.label_heatmap(bin_statistic, color=line_color, fontsize=25, ax=ax, ha='center', va='center', str_format='{:.0f}', path_effects=path_eff)
    teamName = team_name

    # Headings and other texts
    if col == hcol:
      ax.text(105,-3.5, "violet = key pass\ngreen = assist", color=hcol, size=15, ha='right', va='center')
      ax.text(52.5,70, f"Total Chances Created = {cc}", color=col, fontsize=15, ha='center', va='center')
      ax.set_title(f"{hteamName}\nChance Creating Zone", color=line_color, fontsize=25, fontweight='bold', path_effects=path_eff)
    else:
      ax.text(105,71.5, "violet = key pass\ngreen = assist", color=acol, size=15, ha='left', va='center')
      ax.text(52.5,-2, f"Total Chances Created = {cc}", color=col, fontsize=15, ha='center', va='center')
      ax.set_title(f"{ateamName}\nChance Creating Zone", color=line_color, fontsize=25, fontweight='bold', path_effects=path_eff)

    return {
        'Team_Name': team_name,
        'Total_Chances_Created': cc
    }

fig,axs=plt.subplots(1,2, figsize=(20,10), facecolor=bg_color)
chance_creating_stats_home = Chance_creating_zone(axs[0], hteamName, pearl_earring_cmaph, hcol)
chance_creating_stats_away = Chance_creating_zone(axs[1], ateamName, pearl_earring_cmapa, acol)
chance_creating_stats_list = []
chance_creating_stats_list.append(chance_creating_stats_home)
chance_creating_stats_list.append(chance_creating_stats_away)
chance_creating_stats_df = pd.DataFrame(chance_creating_stats_list)

In [ ]:
chance_creating_stats_df.head()

## Dentro del área (Box Entry)

La función `box_entry` tiene como propósito visualizar las entradas al área del oponente (penalty box) de ambos equipos, ya sea mediante pases o conducciones. A continuación te explico la funcionalidad en detalle:

### **1. Filtro de datos para identificar las entradas al área:**
   - Se seleccionan todas las jugadas que resultan en una entrada al área rival, ya sea por un pase o una conducción (carry). 
   - La condición es que la jugada comience fuera del área y termine dentro de la zona del área penal del oponente.

### **2. División por equipo:**
   - Se separan los datos de cada equipo para analizar y visualizar las entradas de ambos equipos (local y visitante).

### **3. Clasificación por zonas:**
   - Las entradas se clasifican según si ocurrieron por el lado izquierdo, el centro o el lado derecho del campo, lo que permite un análisis más detallado de las áreas donde cada equipo tiene mayor actividad ofensiva.

### **4. Visualización en la cancha:**
   - Se dibuja el campo de fútbol y se trazan las líneas para representar los pases y las flechas para las conducciones que resultaron en una entrada al área.
   - Las entradas de un equipo se invierten visualmente para ajustarse al lado de la cancha donde atacan, lo que permite una comparación directa entre los dos equipos.

### **5. Estadísticas visuales:**
   - En la parte superior se muestra el nombre de cada equipo junto con el total de entradas al área.
   - Se utilizan cuadrados grandes para representar las tres zonas (izquierda, centro y derecha), y dentro de cada cuadrado se muestra el número de entradas que provienen de esa zona específica para cada equipo.

### **6. Estructura de retorno:**
   - La función devuelve un diccionario con los datos de cada equipo, incluyendo el total de entradas al área y cuántas de estas entradas provienen de la izquierda, el centro o la derecha.

### **Salida gráfica:**
   - El gráfico resultante te permitirá visualizar las entradas al área de cada equipo y sus puntos de origen en el campo, proporcionando una perspectiva clara sobre cómo los equipos logran penetrar la defensa rival.

Este tipo de visualización es muy útil para identificar las áreas del campo donde los equipos tienden a concentrar su ataque y cómo logran acceder al área rival, ya sea mediante pases o conducciones.

In [ ]:
def box_entry(ax):
    # Box Entry means passes or carries which has started outside the Opponent Penalty Box and ended inside the Opponent Penalty Box 
    bentry = df[((df['type']=='Pass')|(df['type']=='Carry')) & (df['outcomeType']=='Successful') & (df['endX']>=88.5) &
                 ~((df['x']>=88.5) & (df['y']>=13.6) & (df['y']<=54.6)) & (df['endY']>=13.6) & (df['endY']<=54.4) &
            (~df['qualifiers'].str.contains('CornerTaken|Freekick|ThrowIn'))]
    hbentry = bentry[bentry['teamName']==hteamName]
    abentry = bentry[bentry['teamName']==ateamName]

    hrigt = hbentry[hbentry['y']<68/3]
    hcent = hbentry[(hbentry['y']>=68/3) & (hbentry['y']<=136/3)]
    hleft = hbentry[hbentry['y']>136/3]

    arigt = abentry[(abentry['y']<68/3)]
    acent = abentry[(abentry['y']>=68/3) & (abentry['y']<=136/3)]
    aleft = abentry[(abentry['y']>136/3)]

    pitch = Pitch(pitch_type='uefa', line_color=line_color, corner_arcs=True, line_zorder=2, pitch_color=bg_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)

    for index, row in bentry.iterrows():
        if row['teamName'] == ateamName:
            color = acol
            x, y, endX, endY = row['x'], row['y'], row['endX'], row['endY']
        elif row['teamName'] == hteamName:
            color = hcol
            x, y, endX, endY = 105 - row['x'], 68 - row['y'], 105 - row['endX'], 68 - row['endY']
        else:
            continue  # Skip rows that don't match either team name

        if row['type'] == 'Pass':
            pitch.lines(x, y, endX, endY, lw=3.5, comet=True, color=color, ax=ax, alpha=0.5)
            pitch.scatter(endX, endY, s=35, edgecolor=color, linewidth=1, color=bg_color, zorder=2, ax=ax)
        elif row['type'] == 'Carry':
            arrow = patches.FancyArrowPatch((x, y), (endX, endY), arrowstyle='->', color=color, zorder=4, mutation_scale=20, 
                                            alpha=1, linewidth=2, linestyle='--')
            ax.add_patch(arrow)

    
    ax.text(0, 69, f'{hteamName}\nBox Entries: {len(hbentry)}', color=hcol, fontsize=25, fontweight='bold', ha='left', va='bottom')
    ax.text(105, 69, f'{ateamName}\nBox Entries: {len(abentry)}', color=acol, fontsize=25, fontweight='bold', ha='right', va='bottom')

    ax.scatter(46, 6, s=2000, marker='s', color=hcol, zorder=3)
    ax.scatter(46, 34, s=2000, marker='s', color=hcol, zorder=3)
    ax.scatter(46, 62, s=2000, marker='s', color=hcol, zorder=3)
    ax.text(46, 6, f'{len(hleft)}', fontsize=30, fontweight='bold', color=bg_color, ha='center', va='center')
    ax.text(46, 34, f'{len(hcent)}', fontsize=30, fontweight='bold', color=bg_color, ha='center', va='center')
    ax.text(46, 62, f'{len(hrigt)}', fontsize=30, fontweight='bold', color=bg_color, ha='center', va='center')

    ax.scatter(59.5, 6, s=2000, marker='s', color=acol, zorder=3)
    ax.scatter(59.5, 34, s=2000, marker='s', color=acol, zorder=3)
    ax.scatter(59.5, 62, s=2000, marker='s', color=acol, zorder=3)
    ax.text(59.5, 6, f'{len(arigt)}', fontsize=30, fontweight='bold', color=bg_color, ha='center', va='center')
    ax.text(59.5, 34, f'{len(acent)}', fontsize=30, fontweight='bold', color=bg_color, ha='center', va='center')
    ax.text(59.5, 62, f'{len(aleft)}', fontsize=30, fontweight='bold', color=bg_color, ha='center', va='center')

    home_data = {
        'Team_Name': hteamName,
        'Total_Box_Entries': len(hbentry),
        'Box_Entry_From_Left': len(hleft),
        'Box_Entry_From_Center': len(hcent),
        'Box_Entry_From_Right': len(hrigt)
    }
    
    away_data = {
        'Team_Name': ateamName,
        'Total_Box_Entries': len(abentry),
        'Box_Entry_From_Left': len(aleft),
        'Box_Entry_From_Center': len(acent),
        'Box_Entry_From_Right': len(arigt)
    }
    
    return [home_data, away_data]

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
box_entry_stats = box_entry(ax)
box_entry_stats_df = pd.DataFrame(box_entry_stats)

In [ ]:
box_entry_stats_df.head()

## Centros (Cross)

La función `Crosses` que has proporcionado tiene como objetivo visualizar y analizar los centros (crosses) realizados por ambos equipos en un partido, distinguiendo entre los centros exitosos y fallidos. A continuación te doy una explicación detallada en español sobre cómo funciona:

### **1. Visualización del campo:**
   - Se dibuja un campo de fútbol utilizando la librería `mplsoccer.Pitch`, donde se representarán los centros de ambos equipos.

### **2. Filtrado de los centros:**
   - La función filtra todos los pases que son centros (crosses) y excluye los corners. 
   - Los centros se dividen entre los realizados por el equipo local (`home_cross`) y el equipo visitante (`away_cross`).

### **3. Distinción entre centros exitosos y fallidos:**
   - Para cada equipo, se traza una flecha desde el punto de origen hasta el destino del centro.
   - Si el centro fue exitoso, se dibuja una flecha de mayor tamaño y con mayor opacidad, mientras que si fue fallido, se dibuja una flecha más pequeña y menos visible.

### **4. Resumen de estadísticas:**
   - El código calcula el número total de centros exitosos y fallidos para ambos equipos.
   - También cuenta cuántos centros provienen de la banda izquierda y cuántos de la banda derecha para cada equipo.

### **5. Visualización de estadísticas:**
   - En el gráfico, se muestran las estadísticas relacionadas con los centros, como los centros exitosos y fallidos, así como la distribución de centros desde la izquierda y derecha.
   - Además, se incluyen títulos con el nombre del equipo y el lado del campo en el que realizaron los centros.

### **6. Datos de salida:**
   - La función retorna un diccionario con estadísticas detalladas de ambos equipos, que incluyen el número total de centros, los exitosos, fallidos y la cantidad de centros desde cada banda.

Este análisis es muy útil para entender cómo los equipos atacan desde las bandas y qué tan efectivos son sus centros. El uso de visualizaciones como esta puede ayudar a identificar patrones ofensivos y mejorar la estrategia en futuros partidos.

Este tipo de análisis y visualización es perfecto para quienes están trabajando en análisis de rendimiento de equipos en el fútbol, proporcionando una forma clara de interpretar los datos en relación con las jugadas desde las bandas.

In [ ]:
def Crosses(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_ylim(-0.5,68.5)
    ax.set_xlim(-0.5,105.5)

    home_cross = df[(df['teamName']==hteamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Cross')) & (~df['qualifiers'].str.contains('Corner'))]
    away_cross = df[(df['teamName']==ateamName) & (df['type']=='Pass') & (df['qualifiers'].str.contains('Cross')) & (~df['qualifiers'].str.contains('Corner'))]

    hsuc = 0
    hunsuc = 0
    asuc = 0
    aunsuc = 0

    # iterating through each pass and coloring according to successful or not
    for index, row in home_cross.iterrows():
        if row['outcomeType'] == 'Successful':
            arrow = patches.FancyArrowPatch((105-row['x'], 68-row['y']), (105-row['endX'], 68-row['endY']), arrowstyle='->', mutation_scale=15, color=hcol, linewidth=1.5, alpha=1)
            ax.add_patch(arrow)
            hsuc += 1
        else:
            arrow = patches.FancyArrowPatch((105-row['x'], 68-row['y']), (105-row['endX'], 68-row['endY']), arrowstyle='->', mutation_scale=10, color=line_color, linewidth=1.5, alpha=.25)
            ax.add_patch(arrow)
            hunsuc += 1

    for index, row in away_cross.iterrows():
        if row['outcomeType'] == 'Successful':
            arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), arrowstyle='->', mutation_scale=15, color=acol, linewidth=1.5, alpha=1)
            ax.add_patch(arrow)
            asuc += 1
        else:
            arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), arrowstyle='->', mutation_scale=10, color=line_color, linewidth=1.5, alpha=.25)
            ax.add_patch(arrow)
            aunsuc += 1

    # Headlines and other texts
    home_left = len(home_cross[home_cross['y']>=34])
    home_right = len(home_cross[home_cross['y']<34])
    away_left = len(away_cross[away_cross['y']>=34])
    away_right = len(away_cross[away_cross['y']<34])

    ax.text(51, 2, f"Crosses from\nLeftwing: {home_left}", color=hcol, fontsize=15, va='bottom', ha='right')
    ax.text(51, 66, f"Crosses from\nRightwing: {home_right}", color=hcol, fontsize=15, va='top', ha='right')
    ax.text(54, 66, f"Crosses from\nLeftwing: {away_left}", color=acol, fontsize=15, va='top', ha='left')
    ax.text(54, 2, f"Crosses from\nRightwing: {away_right}", color=acol, fontsize=15, va='bottom', ha='left')

    ax.text(0,-2, f"Successful: {hsuc}", color=hcol, fontsize=20, ha='left', va='top')
    ax.text(0,-5.5, f"Unsuccessful: {hunsuc}", color=line_color, fontsize=20, ha='left', va='top')
    ax.text(105,-2, f"Successful: {asuc}", color=acol, fontsize=20, ha='right', va='top')
    ax.text(105,-5.5, f"Unsuccessful: {aunsuc}", color=line_color, fontsize=20, ha='right', va='top')

    ax.text(0, 70, f"{hteamName}\n<---Crosses", color=hcol, size=25, ha='left', fontweight='bold')
    ax.text(105, 70, f"{ateamName}\nCrosses--->", color=acol, size=25, ha='right', fontweight='bold')

    home_data = {
        'Team_Name': hteamName,
        'Total_Cross': hsuc + hunsuc,
        'Successful_Cross': hsuc,
        'Unsuccessful_Cross': hunsuc,
        'Cross_From_LeftWing': home_left,
        'Cross_From_RightWing': home_right
    }
    
    away_data = {
        'Team_Name': ateamName,
        'Total_Cross': asuc + aunsuc,
        'Successful_Cross': asuc,
        'Unsuccessful_Cross': aunsuc,
        'Cross_From_LeftWing': away_left,
        'Cross_From_RightWing': away_right
    }
    
    return [home_data, away_data]

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
cross_stats = Crosses(ax)
cross_stats_df = pd.DataFrame(cross_stats)

In [ ]:
cross_stats_df

## Perdidas de Balón (High Turnover)

La función `HighTO` que has proporcionado tiene como objetivo visualizar y analizar las pérdidas de balón (high turnovers) que ocurren a menos de 40 metros de la portería del equipo contrario, y que resultan en tiros o goles. A continuación te doy una explicación detallada en español:

### **1. Visualización del campo:**
   - Utiliza la librería `mplsoccer.Pitch` para dibujar el campo de fútbol, donde se visualizarán las pérdidas de balón de ambos equipos (tanto las que terminan en tiros o goles como otras pérdidas cercanas a la portería rival).

### **2. Cálculo de la distancia desde la portería:**
   - Para cada evento en el dataframe `df`, se calcula la distancia entre la ubicación del evento y la portería contraria. Aquellos eventos que ocurren dentro de los 40 metros de la portería se consideran "high turnovers".

### **3. Identificación de las pérdidas de balón clave:**
   - **Pérdidas que resultan en gol:** El código busca secuencias donde un equipo pierde el balón en una posición avanzada (dentro de los 40 metros), lo recupera y luego termina la posesión con un gol.
   - **Pérdidas que resultan en tiro:** Similar a las pérdidas que terminan en gol, pero en este caso busca secuencias que terminan con un tiro, sin importar si fue gol o no.
   - **Otras pérdidas:** El código también rastrea las pérdidas de balón cercanas a la portería que no terminan en tiros ni en goles.

### **4. Visualización de las pérdidas de balón:**
   - Se dibujan diferentes símbolos y colores para representar las pérdidas de balón que resultaron en goles, tiros y las demás pérdidas que ocurrieron en el área avanzada.
   - Las pérdidas que resultan en gol son representadas con una estrella verde, mientras que las que resultan en tiros son círculos más pequeños en color.

### **5. Círculos de influencia:**
   - Se dibujan dos círculos grandes (de radio 40 metros) en cada mitad del campo para representar el área donde se consideran las "high turnovers" para ambos equipos.

### **6. Datos de salida:**
   - La función devuelve un diccionario con estadísticas detalladas para ambos equipos, que incluyen el número total de "high turnovers", cuántos de ellos terminaron en tiros y cuántos en goles.

Este tipo de análisis es muy útil para equipos y analistas que buscan evaluar el impacto de la presión alta y la recuperación de balón en zonas peligrosas del campo. Los datos visualizados pueden ayudar a entender mejor cómo las pérdidas de balón en zonas cercanas a la portería pueden influir en el rendimiento ofensivo de un equipo.

In [ ]:
def HighTO(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_ylim(-0.5,68.5)
    ax.set_xlim(-0.5,105.5)

    # High Turnover means any sequence which starts in open play and within 40 metres of the opponent's goal 
    highTO = df
    highTO['Distance'] = ((highTO['x'] - 105)**2 + (highTO['y'] - 34)**2)**0.5

    # HTO which led to Goal for away team
    agoal_count = 0
    # Iterate through the DataFrame
    for i in range(len(highTO)):
        if ((highTO.loc[i, 'type'] in ['BallRecovery', 'Interception']) and 
            (highTO.loc[i, 'teamName'] == ateamName) and 
            (highTO.loc[i, 'Distance'] <= 40)):
            
            possession_id = highTO.loc[i, 'possession_id']
            
            # Check the following rows within the same possession
            j = i + 1
            while j < len(highTO) and highTO.loc[j, 'possession_id'] == possession_id and highTO.loc[j, 'teamName']==ateamName:
                if highTO.loc[j, 'type'] == 'Goal' and highTO.loc[j, 'teamName']==ateamName:
                    ax.scatter(highTO.loc[i, 'x'],highTO.loc[i, 'y'], s=600, marker='*', color='green', edgecolor='k', zorder=3)
                    agoal_count += 1
                    break
                j += 1

    # HTO which led to Shot for away team
    ashot_count = 0
    # Iterate through the DataFrame
    for i in range(len(highTO)):
        if ((highTO.loc[i, 'type'] in ['BallRecovery', 'Interception']) and 
            (highTO.loc[i, 'teamName'] == ateamName) and 
            (highTO.loc[i, 'Distance'] <= 40)):
            
            possession_id = highTO.loc[i, 'possession_id']
            
            # Check the following rows within the same possession
            j = i + 1
            while j < len(highTO) and highTO.loc[j, 'possession_id'] == possession_id and highTO.loc[j, 'teamName']==ateamName:
                if ('Shot' in highTO.loc[j, 'type']) and (highTO.loc[j, 'teamName']==ateamName):
                    ax.scatter(highTO.loc[i, 'x'],highTO.loc[i, 'y'], s=150, color=acol, edgecolor=bg_color, zorder=2)
                    ashot_count += 1
                    break
                j += 1
    
    # other HTO for away team
    aht_count = 0
    p_list = []
    # Iterate through the DataFrame
    for i in range(len(highTO)):
        if ((highTO.loc[i, 'type'] in ['BallRecovery', 'Interception']) and 
            (highTO.loc[i, 'teamName'] == ateamName) and 
            (highTO.loc[i, 'Distance'] <= 40)):
            
            # Check the following rows
            j = i + 1
            if ((highTO.loc[j, 'teamName']==ateamName) and
                (highTO.loc[j, 'type']!='Dispossessed') and (highTO.loc[j, 'type']!='OffsidePass')):
                ax.scatter(highTO.loc[i, 'x'],highTO.loc[i, 'y'], s=100, color='None', edgecolor=acol)
                aht_count += 1
                p_list.append(highTO.loc[i, 'shortName'])

    # HTO which led to Goal for home team
    hgoal_count = 0
    # Iterate through the DataFrame
    for i in range(len(highTO)):
        if ((highTO.loc[i, 'type'] in ['BallRecovery', 'Interception']) and 
            (highTO.loc[i, 'teamName'] == hteamName) and 
            (highTO.loc[i, 'Distance'] <= 40)):
            
            possession_id = highTO.loc[i, 'possession_id']
            
            # Check the following rows within the same possession
            j = i + 1
            while j < len(highTO) and highTO.loc[j, 'possession_id'] == possession_id and highTO.loc[j, 'teamName']==hteamName:
                if highTO.loc[j, 'type'] == 'Goal' and highTO.loc[j, 'teamName']==hteamName:
                    ax.scatter(105-highTO.loc[i, 'x'],68-highTO.loc[i, 'y'], s=600, marker='*', color='green', edgecolor='k', zorder=3)
                    hgoal_count += 1
                    break
                j += 1

    # HTO which led to Shot for home team
    hshot_count = 0
    # Iterate through the DataFrame
    for i in range(len(highTO)):
        if ((highTO.loc[i, 'type'] in ['BallRecovery', 'Interception']) and 
            (highTO.loc[i, 'teamName'] == hteamName) and 
            (highTO.loc[i, 'Distance'] <= 40)):
            
            possession_id = highTO.loc[i, 'possession_id']
            
            # Check the following rows within the same possession
            j = i + 1
            while j < len(highTO) and highTO.loc[j, 'possession_id'] == possession_id and highTO.loc[j, 'teamName']==hteamName:
                if ('Shot' in highTO.loc[j, 'type']) and (highTO.loc[j, 'teamName']==hteamName):
                    ax.scatter(105-highTO.loc[i, 'x'],68-highTO.loc[i, 'y'], s=150, color=hcol, edgecolor=bg_color, zorder=2)
                    hshot_count += 1
                    break
                j += 1

    # other HTO for home team
    hht_count = 0
    p_list = []
    # Iterate through the DataFrame
    for i in range(len(highTO)):
        if ((highTO.loc[i, 'type'] in ['BallRecovery', 'Interception']) and 
            (highTO.loc[i, 'teamName'] == hteamName) and 
            (highTO.loc[i, 'Distance'] <= 40)):
            
            # Check the following rows
            j = i + 1
            if ((highTO.loc[j, 'teamName']==hteamName) and
                (highTO.loc[j, 'type']!='Dispossessed') and (highTO.loc[j, 'type']!='OffsidePass')):
                ax.scatter(105-highTO.loc[i, 'x'],68-highTO.loc[i, 'y'], s=100, color='None', edgecolor=hcol)
                hht_count += 1
                p_list.append(highTO.loc[i, 'shortName'])

    # Plotting the half circle
    left_circle = plt.Circle((0,34), 40, color=hcol, fill=True, alpha=0.25, linestyle='dashed')
    ax.add_artist(left_circle)
    right_circle = plt.Circle((105,34), 40, color=acol, fill=True, alpha=0.25, linestyle='dashed')
    ax.add_artist(right_circle)
    # Set the aspect ratio to be equal
    ax.set_aspect('equal', adjustable='box')
    # Headlines and other texts
    ax.text(0, 70, f"{hteamName}\nHigh Turnover: {hht_count}", color=hcol, size=25, ha='left', fontweight='bold')
    ax.text(105, 70, f"{ateamName}\nHigh Turnover: {aht_count}", color=acol, size=25, ha='right', fontweight='bold')
    ax.text(0,  -3, '<---Attacking Direction', color=hcol, fontsize=13, ha='left', va='center')
    ax.text(105,-3, 'Attacking Direction--->', color=acol, fontsize=13, ha='right', va='center')

    home_data = {
        'Team_Name': hteamName,
        'Total_High_Turnovers': hht_count,
        'Shot_Ending_High_Turnovers': hshot_count,
        'Goal_Ending_High_Turnovers': hgoal_count,
        'Opponent_Team_Name': ateamName
    }
    
    away_data = {
        'Team_Name': ateamName,
        'Total_High_Turnovers': aht_count,
        'Shot_Ending_High_Turnovers': ashot_count,
        'Goal_Ending_High_Turnovers': agoal_count,
        'Opponent_Team_Name': hteamName
    }
    
    return [home_data, away_data]

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
high_turnover_stats = HighTO(ax)
high_turnover_stats_df = pd.DataFrame(high_turnover_stats)

In [ ]:
high_turnover_stats_df

## Congestion (Congestion)

La función `plot_congestion` tiene como objetivo visualizar las zonas del campo donde un equipo domina las jugadas abiertas o donde la posesión está más disputada. Para hacerlo, compara los toques en juego abierto de ambos equipos en varias zonas del campo, y asigna colores según el equipo que tenga más del 55% de los toques en cada zona.

### **Explicación del proceso en español:**

1. **Filtrado de datos:**
   - Se filtran los datos (`df`) para obtener los toques en juego abierto (es decir, aquellos que no provienen de saques de esquina, tiros libres o saques de banda) de ambos equipos.
   - Para el equipo visitante, se invierten las coordenadas (`x` e `y`) para asegurar que ambos equipos estén representados atacando en la misma dirección en el gráfico.

2. **Cálculo de las zonas de dominio:**
   - Se dividen el campo en una cuadrícula de 6 columnas por 5 filas.
   - Se calcula la cantidad de toques de cada equipo en cada zona utilizando la función `bin_statistic` de la librería `mplsoccer.Pitch`.
   - Se compara la proporción de toques en cada zona: 
     - Si más del 55% de los toques corresponden a un equipo, esa zona se colorea con el color del equipo.
     - Si menos del 45% de los toques pertenecen a un equipo, se colorea con el color del equipo contrario.
     - Si ninguna de las proporciones alcanza esos valores, la zona se marca como "disputada" (gris).

3. **Visualización:**
   - Se crea un gráfico de calor (`heatmap`) que muestra qué equipo domina cada zona del campo o si la posesión está disputada.
   - Las líneas de la cuadrícula se representan con líneas discontinuas para dividir visualmente el campo en las 30 zonas.

4. **Leyendas y títulos:**
   - El gráfico incluye un título que explica que las zonas del campo están dominadas por uno de los equipos o están disputadas.
   - También se incluyen indicaciones de la dirección de ataque para ambos equipos.

Este tipo de visualización puede ser muy útil para analizar cómo se distribuye el juego en el campo y qué áreas están controladas por cada equipo, lo que puede influir en decisiones tácticas y análisis de rendimiento.

In [ ]:
def plot_congestion(ax):
    # Comparing open play touches of both teams in each zones of the pitch, if more than 55% touches for a team it will be coloured of that team, otherwise gray to represent contested
    pcmap = LinearSegmentedColormap.from_list("Pearl Earring - 10 colors",  [acol, 'gray', hcol], N=20)
    df1 = df[(df['teamName']==hteamName) & (df['isTouch']==1) & (~df['qualifiers'].str.contains('CornerTaken|Freekick|ThrowIn'))]
    df2 = df[(df['teamName']==ateamName) & (df['isTouch']==1) & (~df['qualifiers'].str.contains('CornerTaken|Freekick|ThrowIn'))]
    df2['x'] = 105-df2['x']
    df2['y'] =  68-df2['y']
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2, line_zorder=6)
    pitch.draw(ax=ax)
    ax.set_ylim(-0.5,68.5)
    ax.set_xlim(-0.5,105.5)

    bin_statistic1 = pitch.bin_statistic(df1.x, df1.y, bins=(6,5), statistic='count', normalize=False)
    bin_statistic2 = pitch.bin_statistic(df2.x, df2.y, bins=(6,5), statistic='count', normalize=False)

    # Assuming 'cx' and 'cy' are as follows:
    cx = np.array([[ 8.75, 26.25, 43.75, 61.25, 78.75, 96.25],
               [ 8.75, 26.25, 43.75, 61.25, 78.75, 96.25],
               [ 8.75, 26.25, 43.75, 61.25, 78.75, 96.25],
               [ 8.75, 26.25, 43.75, 61.25, 78.75, 96.25],
               [ 8.75, 26.25, 43.75, 61.25, 78.75, 96.25]])

    cy = np.array([[61.2, 61.2, 61.2, 61.2, 61.2, 61.2],
               [47.6, 47.6, 47.6, 47.6, 47.6, 47.6],
               [34.0, 34.0, 34.0, 34.0, 34.0, 34.0],
               [20.4, 20.4, 20.4, 20.4, 20.4, 20.4],
               [ 6.8,  6.8,  6.8,  6.8,  6.8,  6.8]])

    # Flatten the arrays
    cx_flat = cx.flatten()
    cy_flat = cy.flatten()

    # Create a DataFrame
    df_cong = pd.DataFrame({'cx': cx_flat, 'cy': cy_flat})

    hd_values = []
    # Loop through the 2D arrays
    for i in range(bin_statistic1['statistic'].shape[0]):
        for j in range(bin_statistic1['statistic'].shape[1]):
            stat1 = bin_statistic1['statistic'][i, j]
            stat2 = bin_statistic2['statistic'][i, j]
        
            if (stat1 / (stat1 + stat2)) > 0.55:
                hd_values.append(1)
            elif (stat1 / (stat1 + stat2)) < 0.45:
                hd_values.append(0)
            else:
                hd_values.append(0.5)

    df_cong['hd']=hd_values
    bin_stat = pitch.bin_statistic(df_cong.cx, df_cong.cy, bins=(6,5), values=df_cong['hd'], statistic='sum', normalize=False)
    pitch.heatmap(bin_stat, ax=ax, cmap=pcmap, edgecolors='#000000', lw=0, zorder=3, alpha=0.85)

    ax_text(52.5, 71, s=f"<{hteamName}>  |  Contested  |  <{ateamName}>", highlight_textprops=[{'color':hcol}, {'color':acol}],
            color='gray', fontsize=18, ha='center', va='center', ax=ax)
    ax.set_title("Team's Dominating Zone", color=line_color, fontsize=30, fontweight='bold', y=1.075)
    ax.text(0,  -3, 'Attacking Direction--->', color=hcol, fontsize=13, ha='left', va='center')
    ax.text(105,-3, '<---Attacking Direction', color=acol, fontsize=13, ha='right', va='center')

    ax.vlines(1*(105/6), ymin=0, ymax=68, color=bg_color, lw=2, ls='--', zorder=5)
    ax.vlines(2*(105/6), ymin=0, ymax=68, color=bg_color, lw=2, ls='--', zorder=5)
    ax.vlines(3*(105/6), ymin=0, ymax=68, color=bg_color, lw=2, ls='--', zorder=5)
    ax.vlines(4*(105/6), ymin=0, ymax=68, color=bg_color, lw=2, ls='--', zorder=5)
    ax.vlines(5*(105/6), ymin=0, ymax=68, color=bg_color, lw=2, ls='--', zorder=5)

    ax.hlines(1*(68/5), xmin=0, xmax=105, color=bg_color, lw=2, ls='--', zorder=5)
    ax.hlines(2*(68/5), xmin=0, xmax=105, color=bg_color, lw=2, ls='--', zorder=5)
    ax.hlines(3*(68/5), xmin=0, xmax=105, color=bg_color, lw=2, ls='--', zorder=5)
    ax.hlines(4*(68/5), xmin=0, xmax=105, color=bg_color, lw=2, ls='--', zorder=5)
    
    return

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
plot_congestion(ax)

# Team Dashboard

In [ ]:
hteamName

In [ ]:
ateamName

El código crea un informe posterior al partido con múltiples visualizaciones clave, organizadas en una cuadrícula de 4 filas y 3 columnas. Aquí tienes una explicación paso a paso de lo que se está logrando:

1. **Subgráficos en una cuadrícula (4x3)**: 
   Se crea una cuadrícula de 4 filas y 3 columnas, donde cada gráfico ocupa una posición específica dentro de esta cuadrícula. El tamaño total de la figura es grande (35x35 pulgadas) para permitir espacio para todos los elementos visuales.

2. **Gráficos de redes de pases, mapa de disparos y mapa defensivo**:
   - **Primera fila**: 
     - Gráficos de las redes de pases tanto para el equipo local como para el visitante, y en el centro un mapa de disparos (`plot_shotmap`). 
   - **Segunda fila**: 
     - Mapa defensivo del equipo local (`defensive_block`), en el centro las estadísticas de porteros (`plot_goalPost`), y un mapa defensivo similar para el equipo visitante.

3. **Pases progresivos y estadísticas de momentum (Expected Threat - xT)**:
   - **Tercera fila**: 
     - Mapa de pases progresivos del equipo local, en el centro el gráfico de momentum (`plot_Momentum`), y el mapa de pases progresivos del equipo visitante.

4. **Portes progresivos y estadísticas generales del partido**:
   - **Cuarta fila**: 
     - Mapa de conducciones progresivas (`Progressvie_Carries_Stats_home` y `Progressvie_Carries_Stats_away`) para ambos equipos, y en el centro las estadísticas generales del partido (`plotting_match_stats`).

5. **Encabezados y textos adicionales**:
   - **Resultado del partido**: En la parte superior se muestra el nombre de ambos equipos con el resultado del partido en el formato "<Nombre equipo local> - <Nombre equipo visitante>". Los nombres de los equipos están resaltados con sus respectivos colores.
   - **Subtítulos**: Información sobre la jornada (GW 1), liga (Premier League 2024-25), y un reconocimiento a la fuente de datos (Opta) y el autor del código.
   - **Indicadores de dirección de ataque**: Texto en la parte inferior que indica las direcciones de ataque para ambos equipos en colores específicos.

Este código genera una visualización compleja y detallada del partido, ideal para informes post-partido con análisis visuales detallados tanto de los equipos como de sus desempeños ofensivos y defensivos.

In [ ]:
fig, axs = plt.subplots(4,3, figsize=(35,35), facecolor=bg_color)

pass_network_stats_home = pass_network_visualization(axs[0,0], home_passes_between_df, home_average_locs_and_count_df, hcol, hteamName)
shooting_stats = plot_shotmap(axs[0,1])
pass_network_stats_away = pass_network_visualization(axs[0,2], away_passes_between_df, away_average_locs_and_count_df, acol, ateamName)

defensive_block_stats_home = defensive_block(axs[1,0], defensive_home_average_locs_and_count_df, hteamName, hcol)
goalkeeping_stats = plot_goalPost(axs[1,1])
defensive_block_stats_away = defensive_block(axs[1,2], defensive_away_average_locs_and_count_df, ateamName, acol)

Progressvie_Passes_Stats_home = draw_progressive_pass_map(axs[2,0], hteamName, hcol)
xT_stats = plot_Momentum(axs[2,1])
Progressvie_Passes_Stats_away = draw_progressive_pass_map(axs[2,2], ateamName, acol)

Progressvie_Carries_Stats_home = draw_progressive_carry_map(axs[3,0], hteamName, hcol)
general_match_stats = plotting_match_stats(axs[3,1])
Progressvie_Carries_Stats_away = draw_progressive_carry_map(axs[3,2], ateamName, acol)

# Heading
highlight_text = [{'color':hcol}, {'color':acol}]
fig_text(0.5, 0.98, f"<{hteamName} {hgoal_count}> - <{agoal_count} {ateamName}>", color=line_color, fontsize=70, fontweight='bold',
            highlight_textprops=highlight_text, ha='center', va='center', ax=fig)

# Subtitles
fig.text(0.5, 0.95, f"GW 1 , EPL 2024-25 | Post Match Report-1", color=line_color, fontsize=30, ha='center', va='center')
fig.text(0.5, 0.93, f"Data from: Opta | code by: @adnaaan433", color=line_color, fontsize=22.5, ha='center', va='center')

fig.text(0.125,0.1, 'Attacking Direction ------->', color=hcol, fontsize=25, ha='left', va='center')
fig.text(0.9,0.1, '<------- Attacking Direction', color=acol, fontsize=25, ha='right', va='center')

In [ ]:
awayteamId = df_teamNameId[df_teamNameId['teamName']==ateamName]
hometeamId = df_teamNameId[df_teamNameId['teamName']==hteamName]

In [ ]:
# Plotting Team's Logo
himage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{hometeamId.iloc[0]['teamId']}.png")
himage = Image.open(himage)
ax_himage = add_image(himage, fig, left=0.125, bottom=0.94, width=0.05, height=0.05)

aimage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{awayteamId.iloc[0]['teamId']}.png")
aimage = Image.open(aimage)
ax_aimage = add_image(aimage, fig, left=0.85, bottom=0.94, width=0.05, height=0.05)

# Saving the final figure
# fig.savefig(f"{match_id}_Match_Report_1.png", bbox_inches='tight')

El siguiente código genera un informe visual completo del partido con una variedad de métricas y estadísticas, organizadas en una cuadrícula de 4 filas y 3 columnas. A continuación, desgloso lo que hace cada sección del código:

1. **Creación de la cuadrícula de subgráficos**:
   Se establece una cuadrícula de subgráficos de 4 filas y 3 columnas con un tamaño total de 35x35 pulgadas y un color de fondo personalizado.

2. **Estadísticas de entradas en el último tercio**:
   - **Primera fila**:
     - Se visualizan las estadísticas de las entradas en el último tercio tanto para el equipo local (`Final_third_entry`) como para el visitante.
     - Se incluye un gráfico que muestra las entradas en el área del penal (`box_entry`).

3. **Estadísticas de pases zonales y cruces**:
   - **Segunda fila**:
     - Se muestran las estadísticas de los pases en la zona 14 tanto para el equipo local como para el visitante (`zone14hs`).
     - Se presenta un gráfico que ilustra los cruces realizados por ambos equipos (`Crosses`).

4. **Estadísticas de pases finales y turnovers altos**:
   - **Tercera fila**:
     - Se visualizan los pases finales que llevan a situaciones de gol para ambos equipos (`Pass_end_zone`).
     - Se muestran los turnovers altos que resultan en goles o disparos (`HighTO`).

5. **Estadísticas de creación de oportunidades y congestión**:
   - **Cuarta fila**:
     - Se visualizan las estadísticas de creación de oportunidades en el último tercio para ambos equipos (`Chance_creating_zone`).
     - Se muestra un gráfico que representa la congestión en el campo (`plot_congestion`).

6. **Encabezados y textos adicionales**:
   - **Resultado del partido**: En la parte superior, se muestra el resultado del partido con los nombres de ambos equipos resaltados en sus respectivos colores.
   - **Subtítulos**: Se incluye información sobre la jornada, la liga y la fuente de datos, así como el autor del código.
   - **Direcciones de ataque**: Texto en la parte inferior que indica las direcciones de ataque de ambos equipos.

7. **Logo del equipo**:
   - Se añaden los logotipos de ambos equipos en la parte superior de la figura, utilizando enlaces a las imágenes almacenadas en FotMob.

8. **Guardado de la figura**:
   - Se incluye un comentario que indica cómo guardar la figura final en un archivo PNG, aunque esta línea está comentada en el código.

Este conjunto de visualizaciones proporciona un análisis detallado del rendimiento de ambos equipos durante el partido, ofreciendo tanto estadísticas numéricas como representaciones gráficas que facilitan la comprensión del juego.

In [ ]:
fig, axs = plt.subplots(4,3, figsize=(35,35), facecolor=bg_color)

final_third_entry_stats_home = Final_third_entry(axs[0,0], hteamName, hcol)
box_entry_stats = box_entry(axs[0,1])
final_third_entry_stats_away = Final_third_entry(axs[0,2], ateamName, acol)

zonal_passing_stats_home = zone14hs(axs[1,0], hteamName, hcol)
cross_stats = Crosses(axs[1,1])
zonal_passing_stats_away = zone14hs(axs[1,2], ateamName, acol)

Pass_end_zone(axs[2,0], hteamName, pearl_earring_cmaph)
high_turnover_stats = HighTO(axs[2,1])
Pass_end_zone(axs[2,2], ateamName, pearl_earring_cmapa)

chance_creating_stats_home = Chance_creating_zone(axs[3,0], hteamName, pearl_earring_cmaph, hcol)
plot_congestion(axs[3,1])
chance_creating_stats_away = Chance_creating_zone(axs[3,2], ateamName, pearl_earring_cmapa, acol)

# Heading
highlight_text = [{'color':hcol}, {'color':acol}]
fig_text(0.5, 0.98, f"<{hteamName} {hgoal_count}> - <{agoal_count} {ateamName}>", color=line_color, fontsize=70, fontweight='bold',
            highlight_textprops=highlight_text, ha='center', va='center', ax=fig)

# Subtitles
fig.text(0.5, 0.95, f"GW 1 , EPL 2024-25 | Post Match Report-2", color=line_color, fontsize=30, ha='center', va='center')
fig.text(0.5, 0.93, f"Data from: Opta | code by: @adnaaan433", color=line_color, fontsize=22.5, ha='center', va='center')

fig.text(0.125,0.1, 'Attacking Direction ------->', color=hcol, fontsize=25, ha='left', va='center')
fig.text(0.9,0.1, '<------- Attacking Direction', color=acol, fontsize=25, ha='right', va='center')

# Plotting Team's Logo
# himage_url = himage_url.replace(' ', '%20')
himage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{hometeamId.iloc[0]['teamId']}.png")
himage = Image.open(himage)
ax_himage = add_image(himage, fig, left=0.125, bottom=0.94, width=0.05, height=0.05)

# aimage_url = aimage_url.replace(' ', '%20')
aimage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{awayteamId.iloc[0]['teamId']}.png")
aimage = Image.open(aimage)
ax_aimage = add_image(aimage, fig, left=0.85, bottom=0.94, width=0.05, height=0.05)

# Saving the final Figure
# fig.savefig(f"{match_id}_Match_Report.png", bbox_inches='tight')

El suiguiente código está diseñado para combinar múltiples DataFrames en un único DataFrame principal llamado `team_stats_df`, utilizando la columna `Team_Name` como clave para la fusión. Aquí te doy un breve desglose de cada parte:

1. **Lista de DataFrames a fusionar**:
   - Se crea una lista llamada `dfs_to_merge` que contiene todos los DataFrames que deseas fusionar. Estos DataFrames parecen estar relacionados con diferentes estadísticas de equipos, como tiros, estadísticas generales del partido, redes de pases, bloques defensivos, pases progresivos, estadísticas de portería, y más.

2. **Inicialización del DataFrame principal**:
   - El DataFrame principal `team_stats_df` se inicializa con `shooting_stats_df`, que contiene las estadísticas de tiros.

3. **Fusión de DataFrames**:
   - Se itera a través de la lista de DataFrames, comenzando desde el segundo DataFrame (índice 1) en adelante. En cada iteración, se fusiona el DataFrame actual (`dfs`) con `team_stats_df` utilizando el método `.merge()`.
   - La fusión se realiza en la columna `Team_Name` con el método de unión `left`, lo que significa que se conservarán todos los registros de `team_stats_df`, y se agregarán las columnas de los otros DataFrames donde haya coincidencias en `Team_Name`.

### Consejos Adicionales:
- **Verifica la existencia de la columna `Team_Name`**: Asegúrate de que todos los DataFrames en la lista `dfs_to_merge` contengan la columna `Team_Name`. De lo contrario, se generará un error durante la fusión.
- **Manejo de valores nulos**: Después de la fusión, es posible que desees manejar los valores nulos resultantes de la unión. Puedes usar `fillna()` o eliminar filas con `dropna()`, dependiendo de tus necesidades.
- **Verifica el resultado final**: Después de realizar las fusiones, sería útil revisar el DataFrame final `team_stats_df` para asegurarte de que todas las columnas se hayan fusionado correctamente y que los datos tengan sentido.


In [ ]:
# List of DataFrames to merge
dfs_to_merge = [shooting_stats_df, general_match_stats_df, pass_network_stats_df,defensive_block_stats_df, Progressvie_Passes_Stats_df, 
                Progressvie_Carries_Stats_df,goalkeeping_stats_df, xT_stats_df, final_third_entry_stats_df,zonal_passing_stats_df,
                chance_creating_stats_df, box_entry_stats_df, cross_stats_df, high_turnover_stats_df]

# Initialize the main DataFrame
team_stats_df = shooting_stats_df

# Merge each DataFrame in the list
for dfs in dfs_to_merge[1:]:
    team_stats_df = team_stats_df.merge(dfs, on='Team_Name', how='left')

In [ ]:
team_stats_df

In [ ]:
team_stats_df.to_csv(os.path.join("FData", f"{fotmob_matchId}_team_stats_fotmob.csv"))

# Top Players Dashboard Function

## Recuento de estadísticas de jugadores

El siguiente código está diseñado para calcular diversas estadísticas de jugadores en un partido de fútbol, centrándose en dos equipos: uno de local y otro de visitante. A continuación, te doy un resumen de las diferentes secciones de tu código y algunos consejos para mejorar la claridad y la eficiencia:

### Resumen del Código

1. **Jugadores Únicos**:
   - Se obtiene una lista de jugadores únicos de los DataFrames `homedf` y `awaydf`.

2. **Top Ball Progressors**:
   - Se inicializan diccionarios para contar el número de pases progresivos, carries progresivos y pases que rompen líneas para los jugadores de ambos equipos.
   - Se itera sobre los jugadores, contando el número de cada tipo de acción y almacenando los resultados en un DataFrame.

3. **Top Threat Creators**:
   - Se realizan conteos de Expected Threat (xT) a partir de pases y carries para los jugadores de ambos equipos. 
   - Se utiliza un enfoque similar al anterior para almacenar los resultados.

4. **Shot Sequence Involvement**:
   - Se inicializan diccionarios para contar los tiros, asistencias de tiro y acciones que contribuyen a un tiro para los jugadores de ambos equipos.
   - Se procesan los datos y se almacenan en un DataFrame.

5. **Top Defenders**:
   - Se cuentan las acciones defensivas (tackles, interceptions y clearances) para los jugadores de ambos equipos.
   - Los resultados se almacenan en DataFrames.


In [ ]:
# Get unique players
home_unique_players = homedf['name'].unique()
away_unique_players = awaydf['name'].unique()


# Top Ball Progressor
# Initialize an empty dictionary to store home players different type of pass counts
home_progressor_counts = {'name': home_unique_players, 'Progressive Passes': [], 'Progressive Carries': [], 'LineBreaking Pass': []}
for name in home_unique_players:
    home_progressor_counts['Progressive Passes'].append(len(df[(df['name'] == name) & (df['prog_pass'] >= 9.144) & (df['x']>=35) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('CornerTaken|Freekick'))]))
    home_progressor_counts['Progressive Carries'].append(len(df[(df['name'] == name) & (df['prog_carry'] >= 9.144) & (df['endX']>=35)]))
    home_progressor_counts['LineBreaking Pass'].append(len(df[(df['name'] == name) & (df['type'] == 'Pass') & (df['qualifiers'].str.contains('Throughball'))]))
home_progressor_df = pd.DataFrame(home_progressor_counts)
home_progressor_df['total'] = home_progressor_df['Progressive Passes']+home_progressor_df['Progressive Carries']+home_progressor_df['LineBreaking Pass']
home_progressor_df = home_progressor_df.sort_values(by='total', ascending=False)
home_progressor_df.reset_index(drop=True, inplace=True)
home_progressor_df = home_progressor_df.head(5)
home_progressor_df['shortName'] = home_progressor_df['name'].apply(get_short_name)

# Initialize an empty dictionary to store away players different type of pass counts
away_progressor_counts = {'name': away_unique_players, 'Progressive Passes': [], 'Progressive Carries': [], 'LineBreaking Pass': []}
for name in away_unique_players:
    away_progressor_counts['Progressive Passes'].append(len(df[(df['name'] == name) & (df['prog_pass'] >= 9.144) & (df['x']>=35) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('CornerTaken|Freekick'))]))
    away_progressor_counts['Progressive Carries'].append(len(df[(df['name'] == name) & (df['prog_carry'] >= 9.144) & (df['endX']>=35)]))
    away_progressor_counts['LineBreaking Pass'].append(len(df[(df['name'] == name) & (df['type'] == 'Pass') & (df['qualifiers'].str.contains('Throughball'))]))
away_progressor_df = pd.DataFrame(away_progressor_counts)
away_progressor_df['total'] = away_progressor_df['Progressive Passes']+away_progressor_df['Progressive Carries']+away_progressor_df['LineBreaking Pass']
away_progressor_df = away_progressor_df.sort_values(by='total', ascending=False)
away_progressor_df.reset_index(drop=True, inplace=True)
away_progressor_df = away_progressor_df.head(5)
away_progressor_df['shortName'] = away_progressor_df['name'].apply(get_short_name)


# Top Threate Creators
# Initialize an empty dictionary to store home players different type of Carries counts
home_xT_counts = {'name': home_unique_players, 'xT from Pass': [], 'xT from Carry': []}
for name in home_unique_players:
    home_xT_counts['xT from Pass'].append((df[(df['name'] == name) & (df['type'] == 'Pass') & (df['xT']>=0) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('CornerTaken|Freekick|ThrowIn'))])['xT'].sum().round(2))
    home_xT_counts['xT from Carry'].append((df[(df['name'] == name) & (df['type'] == 'Carry') & (df['xT']>=0)])['xT'].sum().round(2))
home_xT_df = pd.DataFrame(home_xT_counts)
home_xT_df['total'] = home_xT_df['xT from Pass']+home_xT_df['xT from Carry']
home_xT_df = home_xT_df.sort_values(by='total', ascending=False)
home_xT_df.reset_index(drop=True, inplace=True)
home_xT_df = home_xT_df.head(5)
home_xT_df['shortName'] = home_xT_df['name'].apply(get_short_name)

# Initialize an empty dictionary to store home players different type of Carries counts
away_xT_counts = {'name': away_unique_players, 'xT from Pass': [], 'xT from Carry': []}
for name in away_unique_players:
    away_xT_counts['xT from Pass'].append((df[(df['name'] == name) & (df['type'] == 'Pass') & (df['xT']>=0) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('CornerTaken|Freekick|ThrowIn'))])['xT'].sum().round(2))
    away_xT_counts['xT from Carry'].append((df[(df['name'] == name) & (df['type'] == 'Carry') & (df['xT']>=0)])['xT'].sum().round(2))
away_xT_df = pd.DataFrame(away_xT_counts)
away_xT_df['total'] = away_xT_df['xT from Pass']+away_xT_df['xT from Carry']
away_xT_df = away_xT_df.sort_values(by='total', ascending=False)
away_xT_df.reset_index(drop=True, inplace=True)
away_xT_df = away_xT_df.head(5)
away_xT_df['shortName'] = away_xT_df['name'].apply(get_short_name)


# Shot Sequence Involvement
df_no_carry = df[df['type']!='Carry']
# Initialize an empty dictionary to store home players different type of shot sequence counts
home_shot_seq_counts = {'name': home_unique_players, 'Shots': [], 'Shot Assist': [], 'Buildup to shot': []}
# Putting counts in those lists
for name in home_unique_players:
    home_shot_seq_counts['Shots'].append(len(df[(df['name'] == name) & ((df['type']=='MissedShots') | (df['type']=='SavedShot') | (df['type']=='ShotOnPost') | (df['type']=='Goal'))]))
    home_shot_seq_counts['Shot Assist'].append(len(df[(df['name'] == name) & (df['type'] == 'Pass') & (df['qualifiers'].str.contains('KeyPass'))]))
    home_shot_seq_counts['Buildup to shot'].append(len(df_no_carry[(df_no_carry['name'] == name) & (df_no_carry['type'] == 'Pass') & (df_no_carry['qualifiers'].shift(-1).str.contains('KeyPass'))]))
# converting that list into a dataframe
home_sh_sq_df = pd.DataFrame(home_shot_seq_counts)
home_sh_sq_df['total'] = home_sh_sq_df['Shots']+home_sh_sq_df['Shot Assist']+home_sh_sq_df['Buildup to shot']
home_sh_sq_df = home_sh_sq_df.sort_values(by='total', ascending=False)
home_sh_sq_df.reset_index(drop=True, inplace=True)
home_sh_sq_df = home_sh_sq_df.head(5)
home_sh_sq_df['shortName'] = home_sh_sq_df['name'].apply(get_short_name)

# Initialize an empty dictionary to store away players different type of shot sequence counts
away_shot_seq_counts = {'name': away_unique_players, 'Shots': [], 'Shot Assist': [], 'Buildup to shot': []}
for name in away_unique_players:
    away_shot_seq_counts['Shots'].append(len(df[(df['name'] == name) & ((df['type']=='MissedShots') | (df['type']=='SavedShot') | (df['type']=='ShotOnPost') | (df['type']=='Goal'))]))
    away_shot_seq_counts['Shot Assist'].append(len(df[(df['name'] == name) & (df['type'] == 'Pass') & (df['qualifiers'].str.contains('KeyPass'))]))
    away_shot_seq_counts['Buildup to shot'].append(len(df_no_carry[(df_no_carry['name'] == name) & (df_no_carry['type'] == 'Pass') & (df_no_carry['qualifiers'].shift(-1).str.contains('KeyPass'))]))
away_sh_sq_df = pd.DataFrame(away_shot_seq_counts)
away_sh_sq_df['total'] = away_sh_sq_df['Shots']+away_sh_sq_df['Shot Assist']+away_sh_sq_df['Buildup to shot']
away_sh_sq_df = away_sh_sq_df.sort_values(by='total', ascending=False)
away_sh_sq_df.reset_index(drop=True, inplace=True)
away_sh_sq_df = away_sh_sq_df.head(5)
away_sh_sq_df['shortName'] = away_sh_sq_df['name'].apply(get_short_name)


# Top Defenders
# Initialize an empty dictionary to store home players different type of defensive actions counts
home_defensive_actions_counts = {'name': home_unique_players, 'Tackles': [], 'Interceptions': [], 'Clearance': []}
for name in home_unique_players:
    home_defensive_actions_counts['Tackles'].append(len(df[(df['name'] == name) & (df['type'] == 'Tackle') & (df['outcomeType']=='Successful')]))
    home_defensive_actions_counts['Interceptions'].append(len(df[(df['name'] == name) & (df['type'] == 'Interception')]))
    home_defensive_actions_counts['Clearance'].append(len(df[(df['name'] == name) & (df['type'] == 'Clearance')]))
home_defender_df = pd.DataFrame(home_defensive_actions_counts)
home_defender_df['total'] = home_defender_df['Tackles']+home_defender_df['Interceptions']+home_defender_df['Clearance']
home_defender_df = home_defender_df.sort_values(by='total', ascending=False)
home_defender_df.reset_index(drop=True, inplace=True)
home_defender_df = home_defender_df.head(5)
home_defender_df['shortName'] = home_defender_df['name'].apply(get_short_name)

# Initialize an empty dictionary to store away players different type of defensive actions counts
away_defensive_actions_counts = {'name': away_unique_players, 'Tackles': [], 'Interceptions': [], 'Clearance': []}
for name in away_unique_players:
    away_defensive_actions_counts['Tackles'].append(len(df[(df['name'] == name) & (df['type'] == 'Tackle') & (df['outcomeType']=='Successful')]))
    away_defensive_actions_counts['Interceptions'].append(len(df[(df['name'] == name) & (df['type'] == 'Interception')]))
    away_defensive_actions_counts['Clearance'].append(len(df[(df['name'] == name) & (df['type'] == 'Clearance')]))
away_defender_df = pd.DataFrame(away_defensive_actions_counts)
away_defender_df['total'] = away_defender_df['Tackles']+away_defender_df['Interceptions']+away_defender_df['Clearance']
away_defender_df = away_defender_df.sort_values(by='total', ascending=False)
away_defender_df.reset_index(drop=True, inplace=True)
away_defender_df = away_defender_df.head(5)
away_defender_df['shortName'] = away_defender_df['name'].apply(get_short_name)

El siguiente código está bien estructurado y se encarga de calcular varias estadísticas de los jugadores de fútbol en un partido. Aquí tienes un resumen y algunas sugerencias para mejorar su eficacia y claridad:

### Resumen del Código

1. **Jugadores Únicos**:
   - Obtienes una lista de jugadores únicos del DataFrame `df`.

2. **Top Ball Progressor**:
   - Inicializas un diccionario para contar pases progresivos, carries progresivos y pases que rompen líneas. Iteras sobre los jugadores para llenar este diccionario y luego lo conviertes en un DataFrame.

3. **Top Threat Creators**:
   - Contas el Expected Threat (xT) derivado de los pases y carries para cada jugador, similar a la sección anterior.

4. **Shot Sequence Involvement**:
   - Contas los tiros, asistencias de tiro y acciones de preparación para tiros, procesando los datos y almacenándolos en un DataFrame.

5. **Top Defenders**:
   - Cuentas las acciones defensivas (tackles, interceptions, clearances) para cada jugador y almacenas los resultados en otro DataFrame.


In [ ]:
# Get unique players
unique_players = df['name'].unique()


# Top Ball Progressor
# Initialize an empty dictionary to store both team's players different type of pass counts
progressor_counts = {'name': unique_players, 'Progressive Passes': [], 'Progressive Carries': [], 'LineBreaking Pass': []}
for name in unique_players:
    progressor_counts['Progressive Passes'].append(len(df[(df['name'] == name) & (df['prog_pass'] >= 9.144) & (df['x']>=35) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('CornerTaken|Freekick'))]))
    progressor_counts['Progressive Carries'].append(len(df[(df['name'] == name) & (df['prog_carry'] >= 9.144) & (df['endX']>=35)]))
    progressor_counts['LineBreaking Pass'].append(len(df[(df['name'] == name) & (df['type'] == 'Pass') & (df['qualifiers'].str.contains('Throughball'))]))
progressor_df = pd.DataFrame(progressor_counts)
progressor_df['total'] = progressor_df['Progressive Passes']+progressor_df['Progressive Carries']+progressor_df['LineBreaking Pass']
progressor_df = progressor_df.sort_values(by='total', ascending=False)
progressor_df.reset_index(drop=True, inplace=True)
progressor_df = progressor_df.head(10)
progressor_df['shortName'] = progressor_df['name'].apply(get_short_name)




# Top Threate Creators
# Initialize an empty dictionary to store both team's players different type of Carries counts
xT_counts = {'name': unique_players, 'xT from Pass': [], 'xT from Carry': []}
for name in unique_players:
    xT_counts['xT from Pass'].append((df[(df['name'] == name) & (df['type'] == 'Pass') & (df['xT']>=0) & (df['outcomeType']=='Successful') & (~df['qualifiers'].str.contains('CornerTaken|Freekick|ThrowIn'))])['xT'].sum().round(2))
    xT_counts['xT from Carry'].append((df[(df['name'] == name) & (df['type'] == 'Carry') & (df['xT']>=0)])['xT'].sum().round(2))
xT_df = pd.DataFrame(xT_counts)
xT_df['total'] = xT_df['xT from Pass']+xT_df['xT from Carry']
xT_df = xT_df.sort_values(by='total', ascending=False)
xT_df.reset_index(drop=True, inplace=True)
xT_df = xT_df.head(10)
xT_df['shortName'] = xT_df['name'].apply(get_short_name)




# Shot Sequence Involvement
df_no_carry = df[df['type']!='Carry']
# Initialize an empty dictionary to store both team's players different type of shot sequence counts
shot_seq_counts = {'name': unique_players, 'Shots': [], 'Shot Assist': [], 'Buildup to shot': []}
# Putting counts in those lists
for name in unique_players:
    shot_seq_counts['Shots'].append(len(df[(df['name'] == name) & ((df['type']=='MissedShots') | (df['type']=='SavedShot') | (df['type']=='ShotOnPost') | (df['type']=='Goal'))]))
    shot_seq_counts['Shot Assist'].append(len(df[(df['name'] == name) & (df['type'] == 'Pass') & (df['qualifiers'].str.contains('KeyPass'))]))
    shot_seq_counts['Buildup to shot'].append(len(df_no_carry[(df_no_carry['name'] == name) & (df_no_carry['type'] == 'Pass') & (df_no_carry['qualifiers'].shift(-1).str.contains('KeyPass'))]))
# converting that list into a dataframe
sh_sq_df = pd.DataFrame(shot_seq_counts)
sh_sq_df['total'] = sh_sq_df['Shots']+sh_sq_df['Shot Assist']+sh_sq_df['Buildup to shot']
sh_sq_df = sh_sq_df.sort_values(by='total', ascending=False)
sh_sq_df.reset_index(drop=True, inplace=True)
sh_sq_df = sh_sq_df.head(10)
sh_sq_df['shortName'] = sh_sq_df['name'].apply(get_short_name)




# Top Defenders
# Initialize an empty dictionary to store both team's players different type of defensive actions counts
defensive_actions_counts = {'name': unique_players, 'Tackles': [], 'Interceptions': [], 'Clearance': []}
for name in unique_players:
    defensive_actions_counts['Tackles'].append(len(df[(df['name'] == name) & (df['type'] == 'Tackle') & (df['outcomeType']=='Successful')]))
    defensive_actions_counts['Interceptions'].append(len(df[(df['name'] == name) & (df['type'] == 'Interception')]))
    defensive_actions_counts['Clearance'].append(len(df[(df['name'] == name) & (df['type'] == 'Clearance')]))
defender_df = pd.DataFrame(defensive_actions_counts)
defender_df['total'] = defender_df['Tackles']+defender_df['Interceptions']+defender_df['Clearance']
defender_df = defender_df.sort_values(by='total', ascending=False)
defender_df.reset_index(drop=True, inplace=True)
defender_df = defender_df.head(10)
defender_df['shortName'] = defender_df['name'].apply(get_short_name)

## Mapa de Pases del Mejor Pasador (Top Passer's PassMap)

Esta función sirve crear el mapa de pases de los jugadores de los equipos locales y visitantes está bien estructurada. Aquí tienes un resumen de su funcionamiento y algunos comentarios que podrían mejorar la claridad y funcionalidad del código:

### Funciones para Mapa de Pases de Jugadores

1. **Función `home_player_passmap(ax)`**:
   - Dibuja el mapa de pases del mejor pasador del equipo local.
   - Calcula y clasifica los pases exitosos, los pases progresivos, las asistencias y los pases clave.
   - Utiliza diferentes colores para representar los distintos tipos de pases.
   - Muestra estadísticas en el título y en el texto dentro del gráfico.

2. **Función `away_player_passmap(ax)`**:
   - Similar a la función anterior, pero para el mejor pasador del equipo visitante.
   - Invertir el eje x e y permite que el gráfico refleje la perspectiva del equipo visitante.
   - También muestra estadísticas relevantes en el gráfico.


In [ ]:
def home_player_passmap(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)

    # taking the top home passer and plotting his passmap
    home_player_name = home_progressor_df['name'].iloc[0]

    acc_pass = df[(df['name']==home_player_name) & (df['type']=='Pass') & (df['outcomeType']=='Successful')]
    pro_pass = acc_pass[(acc_pass['prog_pass']>=9.11) & (acc_pass['x']>=35) & (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    pro_carry = df[(df['name']==home_player_name) & (df['prog_carry']>=9.11) & (df['endX']>=35)]
    key_pass = acc_pass[acc_pass['qualifiers'].str.contains('KeyPass')]
    g_assist = acc_pass[acc_pass['qualifiers'].str.contains('GoalAssist')]

    pitch.lines(acc_pass.x, acc_pass.y, acc_pass.endX, acc_pass.endY, color=line_color, lw=2, alpha=0.15, comet=True, zorder=2, ax=ax)
    pitch.lines(pro_pass.x, pro_pass.y, pro_pass.endX, pro_pass.endY, color=hcol      , lw=3, alpha=1,    comet=True, zorder=3, ax=ax)
    pitch.lines(key_pass.x, key_pass.y, key_pass.endX, key_pass.endY, color=violet,     lw=4, alpha=1,    comet=True, zorder=4, ax=ax)
    pitch.lines(g_assist.x, g_assist.y, g_assist.endX, g_assist.endY, color='green',      lw=4, alpha=1,    comet=True, zorder=5, ax=ax)

    ax.scatter(acc_pass.endX, acc_pass.endY, s=30, color=bg_color,    edgecolor='gray', alpha=1, zorder=2)
    ax.scatter(pro_pass.endX, pro_pass.endY, s=40, color=bg_color,  edgecolor= hcol,  alpha=1, zorder=3)
    ax.scatter(key_pass.endX, key_pass.endY, s=50, color=bg_color,  edgecolor=violet, alpha=1, zorder=4)
    ax.scatter(g_assist.endX, g_assist.endY, s=50, color=bg_color,  edgecolor= 'green', alpha=1, zorder=5)

    for index, row in pro_carry.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), arrowstyle='->', color=hcol, zorder=4, mutation_scale=20, 
                                        alpha=0.9, linewidth=2, linestyle='--')
        ax.add_patch(arrow) 


    home_name_show = home_progressor_df['shortName'].iloc[0]
    ax.set_title(f"{home_name_show} PassMap", color=hcol, fontsize=25, fontweight='bold', y=1.03)
    ax.text(0,-3, f'Prog. Pass: {len(pro_pass)}          Prog. Carry: {len(pro_carry)}', fontsize=15, color=hcol, ha='left', va='center')
    ax_text(105,-3, s=f'Key Pass: {len(key_pass)}          <Assist: {len(g_assist)}>', fontsize=15, color=violet, ha='right', va='center',
            highlight_textprops=[{'color':'green'}], ax=ax)

def away_player_passmap(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)
    ax.invert_xaxis()
    ax.invert_yaxis()

    # taking the top away passer and plotting his passmap
    away_player_name = away_progressor_df['name'].iloc[0]
    
    acc_pass = df[(df['name']==away_player_name) & (df['type']=='Pass') & (df['outcomeType']=='Successful')]
    pro_pass = acc_pass[(acc_pass['prog_pass']>=9.11) & (acc_pass['x']>=35) & (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    pro_carry = df[(df['name']==away_player_name) & (df['prog_carry']>=9.11) & (df['endX']>=35)]
    key_pass = acc_pass[acc_pass['qualifiers'].str.contains('KeyPass')]
    g_assist = acc_pass[acc_pass['qualifiers'].str.contains('GoalAssist')]

    pitch.lines(acc_pass.x, acc_pass.y, acc_pass.endX, acc_pass.endY, color=line_color, lw=2, alpha=0.15, comet=True, zorder=2, ax=ax)
    pitch.lines(pro_pass.x, pro_pass.y, pro_pass.endX, pro_pass.endY, color=acol      , lw=3, alpha=1,    comet=True, zorder=3, ax=ax)
    pitch.lines(key_pass.x, key_pass.y, key_pass.endX, key_pass.endY, color=violet,     lw=4, alpha=1,    comet=True, zorder=4, ax=ax)
    pitch.lines(g_assist.x, g_assist.y, g_assist.endX, g_assist.endY, color='green',      lw=4, alpha=1,    comet=True, zorder=5, ax=ax)

    ax.scatter(acc_pass.endX, acc_pass.endY, s=30, color=bg_color,    edgecolor='gray', alpha=1, zorder=2)
    ax.scatter(pro_pass.endX, pro_pass.endY, s=40, color=bg_color,  edgecolor= acol,  alpha=1, zorder=3)
    ax.scatter(key_pass.endX, key_pass.endY, s=50, color=bg_color,  edgecolor=violet, alpha=1, zorder=4)
    ax.scatter(g_assist.endX, g_assist.endY, s=50, color=bg_color,  edgecolor= 'green', alpha=1, zorder=5)

    for index, row in pro_carry.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), arrowstyle='->', color=acol, zorder=4, mutation_scale=20, 
                                        alpha=0.9, linewidth=2, linestyle='--')
        ax.add_patch(arrow) 


    away_name_show = away_progressor_df['shortName'].iloc[0]
    ax.set_title(f"{away_name_show} PassMap", color=acol, fontsize=25, fontweight='bold', y=1.03)
    ax.text(0,71, f'Prog. Pass: {len(pro_pass)}          Prog. Carry: {len(pro_carry)}', fontsize=15, color=acol, ha='right', va='center')
    ax_text(105,71, s=f'Key Pass: {len(key_pass)}          <Assist: {len(g_assist)}>', fontsize=15, color=violet, ha='left', va='center',
            highlight_textprops=[{'color':'green'}], ax=ax)

# fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
# home_player_passmap(ax)

In [ ]:
fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
home_player_passmap(ax)

## Recepción de Pases Hacia Adelante (Forward Pass Receiving)

El siguiente código sirve para generar los mapas de pases recibidos de los jugadores de los equipos local y visitante está bien estructurado. Aquí te ofrezco un resumen de su funcionalidad y algunas recomendaciones para mejorar la claridad y la eficiencia del código.

### Resumen de las Funciones

1. **Función `get_short_name(full_name)`**:
   - Esta función toma el nombre completo de un jugador y devuelve una versión abreviada del mismo, utilizando las iniciales y el apellido.

2. **Función `home_passes_recieved(ax)`**:
   - Dibuja el mapa de pases recibidos del mejor pasador del equipo local.
   - Filtra los pases exitosos recibidos por el delantero central y los clasifica como pases clave y asistencias.
   - Utiliza diferentes colores para representar los tipos de pases en el gráfico.
   - Muestra estadísticas relevantes en el título y en el gráfico.

3. **Función `away_passes_recieved(ax)`**:
   - Similar a la función anterior, pero para el mejor pasador del equipo visitante.
   - Invertir los ejes permite que el gráfico refleje la perspectiva del equipo visitante.
   - También muestra estadísticas en el gráfico.


In [ ]:
def get_short_name(full_name):
    if not isinstance(full_name, str):
        return None  # Return None if the input is not a string
    parts = full_name.split()
    if len(parts) == 1:
        return full_name  # No need for short name if there's only one word
    elif len(parts) == 2:
        return parts[0][0] + ". " + parts[1]
    else:
        return parts[0][0] + ". " + parts[1][0] + ". " + " ".join(parts[2:])

def home_passes_recieved(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)

    # plotting the home center forward pass receiving
    name = home_average_locs_and_count_df.loc[home_average_locs_and_count_df['position'] == 'FW', 'name'].tolist()[0]
    name_show = get_short_name(name)
    filtered_rows = df[(df['type'] == 'Pass') & (df['outcomeType'] == 'Successful') & (df['name'].shift(-1) == name)]
    keypass_recieved_df = filtered_rows[filtered_rows['qualifiers'].str.contains('KeyPass')]
    assist_recieved_df = filtered_rows[filtered_rows['qualifiers'].str.contains('IntentionalGoalAssist')]
    pr = len(filtered_rows)
    kpr = len(keypass_recieved_df)
    ar = len(assist_recieved_df)

    lc1 = pitch.lines(filtered_rows.x, filtered_rows.y, filtered_rows.endX, filtered_rows.endY, lw=3, transparent=True, comet=True,color=hcol, ax=ax, alpha=0.5)
    lc2 = pitch.lines(keypass_recieved_df.x, keypass_recieved_df.y, keypass_recieved_df.endX, keypass_recieved_df.endY, lw=4, transparent=True, comet=True,color=violet, ax=ax, alpha=0.75)
    lc3 = pitch.lines(assist_recieved_df.x, assist_recieved_df.y, assist_recieved_df.endX, assist_recieved_df.endY, lw=4, transparent=True, comet=True,color='green', ax=ax, alpha=0.75)
    sc1 = pitch.scatter(filtered_rows.endX, filtered_rows.endY, s=30, edgecolor=hcol, linewidth=1, color=bg_color, zorder=2, ax=ax)
    sc2 = pitch.scatter(keypass_recieved_df.endX, keypass_recieved_df.endY, s=40, edgecolor=violet, linewidth=1.5, color=bg_color, zorder=2, ax=ax)
    sc3 = pitch.scatter(assist_recieved_df.endX, assist_recieved_df.endY, s=50, edgecolors='green', linewidths=1, marker='football', c=bg_color, zorder=2, ax=ax)

    avg_endY = filtered_rows['endY'].median()
    avg_endX = filtered_rows['endX'].median()
    ax.axvline(x=avg_endX, ymin=0, ymax=68, color='gray', linestyle='--', alpha=0.6, linewidth=2)
    ax.axhline(y=avg_endY, xmin=0, xmax=105, color='gray', linestyle='--', alpha=0.6, linewidth=2)
    ax.set_title(f"{name_show} Passes Recieved", color=hcol, fontsize=25, fontweight='bold', y=1.03)
    highlight_text=[{'color':violet}, {'color':'green'}]
    ax_text(52.5,-3, f'Passes Recieved:{pr+kpr} | <Keypasses Recieved:{kpr}> | <Assist Received: {ar}>', color=line_color, fontsize=15, ha='center', 
            va='center', highlight_textprops=highlight_text, ax=ax)

    return

def away_passes_recieved(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)
    ax.invert_xaxis()
    ax.invert_yaxis()

    # plotting the away center forward pass receiving
    name = away_average_locs_and_count_df.loc[away_average_locs_and_count_df['position'] == 'FW', 'name'].tolist()[0]
    name_show = get_short_name(name)
    filtered_rows = df[(df['type'] == 'Pass') & (df['outcomeType'] == 'Successful') & (df['name'].shift(-1) == name)]
    keypass_recieved_df = filtered_rows[filtered_rows['qualifiers'].str.contains('KeyPass')]
    assist_recieved_df = filtered_rows[filtered_rows['qualifiers'].str.contains('IntentionalGoalAssist')]
    pr = len(filtered_rows)
    kpr = len(keypass_recieved_df)
    ar = len(assist_recieved_df)

    lc1 = pitch.lines(filtered_rows.x, filtered_rows.y, filtered_rows.endX, filtered_rows.endY, lw=3, transparent=True, comet=True,color=acol, ax=ax, alpha=0.5)
    lc2 = pitch.lines(keypass_recieved_df.x, keypass_recieved_df.y, keypass_recieved_df.endX, keypass_recieved_df.endY, lw=4, transparent=True, comet=True,color=violet, ax=ax, alpha=0.75)
    lc3 = pitch.lines(assist_recieved_df.x, assist_recieved_df.y, assist_recieved_df.endX, assist_recieved_df.endY, lw=4, transparent=True, comet=True,color='green', ax=ax, alpha=0.75)
    sc1 = pitch.scatter(filtered_rows.endX, filtered_rows.endY, s=30, edgecolor=acol, linewidth=1, color=bg_color, zorder=2, ax=ax)
    sc2 = pitch.scatter(keypass_recieved_df.endX, keypass_recieved_df.endY, s=40, edgecolor=violet, linewidth=1.5, color=bg_color, zorder=2, ax=ax)
    sc3 = pitch.scatter(assist_recieved_df.endX, assist_recieved_df.endY, s=50, edgecolors='green', linewidths=1, marker='football', c=bg_color, zorder=2, ax=ax)

    avg_endX = filtered_rows['endX'].median()
    avg_endY = filtered_rows['endY'].median()
    ax.axvline(x=avg_endX, ymin=0, ymax=68, color='gray', linestyle='--', alpha=0.6, linewidth=2)
    ax.axhline(y=avg_endY, xmin=0, xmax=105, color='gray', linestyle='--', alpha=0.6, linewidth=2)
    ax.set_title(f"{name_show} Passes Recieved", color=acol, fontsize=25, fontweight='bold', y=1.03)
    highlight_text=[{'color':violet}, {'color':'green'}]
    ax_text(52.5,71, f'Passes Recieved:{pr+kpr} | <Keypasses Recieved:{kpr}> | <Assist Received: {ar}>', color=line_color, fontsize=15, ha='center', 
            va='center', highlight_textprops=highlight_text, ax=ax)

    return

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
home_passes_recieved(ax)

## Mejores defensas (Top Defenders)


### Resumen de Funciones

1. **Función `home_player_def_acts(ax)`**:
   - **Objetivo**: Visualizar las acciones defensivas del mejor defensor del equipo local en un gráfico.
   - **Detalles**:
     - Se configura un campo de fútbol con las especificaciones de la UEFA.
     - Se extrae el nombre del mejor defensor local y se filtran las acciones defensivas correspondientes desde el DataFrame `df`.
     - Se identifican y cuentan varias acciones defensivas: 
       - **Tackles**
       - **Intercepciones**
       - **Recuperaciones de balón**
       - **Despejes**
       - **Faltas**
       - **Aéreos defensivos**
     - Se representan gráficamente las acciones utilizando diferentes marcadores y colores, mostrando la cantidad de cada tipo de acción.
     - Se añade un título con el nombre abreviado del jugador y un resumen textual de las estadísticas.

2. **Función `away_player_def_acts(ax)`**:
   - **Objetivo**: Visualizar las acciones defensivas del mejor defensor del equipo visitante en un gráfico.
   - **Detalles**:
     - Similar a la función anterior, pero se invierte el eje X y el eje Y para adaptarse a la perspectiva del equipo visitante.
     - Se extrae el nombre del mejor defensor visitante y se filtran sus acciones defensivas.
     - Se grafican las mismas acciones defensivas con los mismos tipos de marcadores y colores, pero utilizando el color correspondiente al equipo visitante.
     - Se presenta un resumen textual de las estadísticas del jugador visitante, junto con un título.

Ambas funciones están diseñadas para mostrar visualmente las contribuciones defensivas clave de los jugadores en el contexto de un partido.

In [ ]:
def home_player_def_acts(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, line_zorder=2, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-12,68.5)

    # taking the top home defender and plotting his defensive actions
    home_player_name = home_defender_df['name'].iloc[0]
    home_playerdf = df[(df['name']==home_player_name)]

    hp_tk = home_playerdf[home_playerdf['type']=='Tackle']
    hp_intc = home_playerdf[(home_playerdf['type']=='Interception') | (home_playerdf['type']=='BlockedPass')]
    hp_br = home_playerdf[home_playerdf['type']=='BallRecovery']
    hp_cl = home_playerdf[home_playerdf['type']=='Clearance']
    hp_fl = home_playerdf[home_playerdf['type']=='Foul']
    hp_ar = home_playerdf[(home_playerdf['type']=='Aerial') & (home_playerdf['qualifiers'].str.contains('Defensive'))]

    sc1 = pitch.scatter(hp_tk.x, hp_tk.y, s=250, c=hcol, lw=2.5, edgecolor=hcol, marker='+', hatch='/////', ax=ax)
    sc2 = pitch.scatter(hp_intc.x, hp_intc.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='s', hatch='/////', ax=ax)
    sc3 = pitch.scatter(hp_br.x, hp_br.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='o', hatch='/////', ax=ax)
    sc4 = pitch.scatter(hp_cl.x, hp_cl.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='d', hatch='/////', ax=ax)
    sc5 = pitch.scatter(hp_fl.x, hp_fl.y, s=250, c=hcol, lw=2.5, edgecolor=hcol, marker='x', hatch='/////', ax=ax)
    sc6 = pitch.scatter(hp_ar.x, hp_ar.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='^', hatch='/////', ax=ax)

    sc7 =  pitch.scatter(2, -4, s=150, c=hcol, lw=2.5, edgecolor=hcol, marker='+', hatch='/////', ax=ax)
    sc8 =  pitch.scatter(2, -10, s=150, c='None', lw=2.5, edgecolor=hcol, marker='s', hatch='/////', ax=ax)
    sc9 =  pitch.scatter(41, -4, s=150, c='None', lw=2.5, edgecolor=hcol, marker='o', hatch='/////', ax=ax)
    sc10 = pitch.scatter(41, -10, s=150, c='None', lw=2.5, edgecolor=hcol, marker='d', hatch='/////', ax=ax)
    sc11 = pitch.scatter(103, -4, s=150, c=hcol, lw=2.5, edgecolor=hcol, marker='x', hatch='/////', ax=ax)
    sc12 = pitch.scatter(103, -10, s=150, c='None', lw=2.5, edgecolor=hcol, marker='^', hatch='/////', ax=ax)

    ax.text(5, -3, f"Tackle: {len(hp_tk)}\n\nInterception: {len(hp_intc)}", color=hcol, ha='left', va='top', fontsize=13)
    ax.text(43, -3, f"BallRecovery: {len(hp_br)}\n\nClearance: {len(hp_cl)}", color=hcol, ha='left', va='top', fontsize=13)
    ax.text(100, -3, f"{len(hp_fl)} Fouls\n\n{len(hp_ar)} Aerials", color=hcol, ha='right', va='top', fontsize=13)

    home_name_show = home_defender_df['shortName'].iloc[0]
    ax.set_title(f"{home_name_show} Defensive Actions", color=hcol, fontsize=25, fontweight='bold')

def away_player_def_acts(ax):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, line_zorder=2, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5,80)
    ax.invert_xaxis()
    ax.invert_yaxis()

    # taking the top home defender and plotting his defensive actions
    away_player_name = away_defender_df['name'].iloc[0]
    away_playerdf = df[(df['name']==away_player_name)]

    ap_tk = away_playerdf[away_playerdf['type']=='Tackle']
    ap_intc = away_playerdf[(away_playerdf['type']=='Interception') | (away_playerdf['type']=='BlockedPass')]
    ap_br = away_playerdf[away_playerdf['type']=='BallRecovery']
    ap_cl = away_playerdf[away_playerdf['type']=='Clearance']
    ap_fl = away_playerdf[away_playerdf['type']=='Foul']
    ap_ar = away_playerdf[(away_playerdf['type']=='Aerial') & (away_playerdf['qualifiers'].str.contains('Defensive'))]

    sc1 = pitch.scatter(ap_tk.x, ap_tk.y, s=250, c=acol, lw=2.5, edgecolor=acol, marker='+', hatch='/////', ax=ax)
    sc2 = pitch.scatter(ap_intc.x, ap_intc.y, s=250, c='None', lw=2.5, edgecolor=acol, marker='s', hatch='/////', ax=ax)
    sc3 = pitch.scatter(ap_br.x, ap_br.y, s=250, c='None', lw=2.5, edgecolor=acol, marker='o', hatch='/////', ax=ax)
    sc4 = pitch.scatter(ap_cl.x, ap_cl.y, s=250, c='None', lw=2.5, edgecolor=acol, marker='d', hatch='/////', ax=ax)
    sc5 = pitch.scatter(ap_fl.x, ap_fl.y, s=250, c=acol, lw=2.5, edgecolor=acol, marker='x', hatch='/////', ax=ax)
    sc6 = pitch.scatter(ap_ar.x, ap_ar.y, s=250, c='None', lw=2.5, edgecolor=acol, marker='^', hatch='/////', ax=ax)

    sc7 =  pitch.scatter(2, 72, s=150, c=acol, lw=2.5, edgecolor=acol, marker='+', hatch='/////', ax=ax)
    sc8 =  pitch.scatter(2, 78, s=150, c='None', lw=2.5, edgecolor=acol, marker='s', hatch='/////', ax=ax)
    sc9 =  pitch.scatter(41, 72, s=150, c='None', lw=2.5, edgecolor=acol, marker='o', hatch='/////', ax=ax)
    sc10 = pitch.scatter(41, 78, s=150, c='None', lw=2.5, edgecolor=acol, marker='d', hatch='/////', ax=ax)
    sc11 = pitch.scatter(103, 72, s=150, c=acol, lw=2.5, edgecolor=acol, marker='x', hatch='/////', ax=ax)
    sc12 = pitch.scatter(103, 78, s=150, c='None', lw=2.5, edgecolor=acol, marker='^', hatch='/////', ax=ax)

    ax.text(5, 71, f"Tackle: {len(ap_tk)}\n\nInterception: {len(ap_intc)}", color=acol, ha='right', va='top', fontsize=13)
    ax.text(43, 71, f"BallRecovery: {len(ap_br)}\n\nClearance: {len(ap_cl)}", color=acol, ha='right', va='top', fontsize=13)
    ax.text(100, 71, f"{len(ap_fl)} Fouls\n\n{len(ap_ar)} Aerials", color=acol, ha='left', va='top', fontsize=13)

    away_name_show = away_defender_df['shortName'].iloc[0]
    ax.set_title(f"{away_name_show} Defensive Actions", color=acol, fontsize=25, fontweight='bold')

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
away_player_def_acts(ax)

## Mapa de pases del portero (GoalKeeper PassMap)


### Resumen de Funciones

1. **Función `home_gk(ax)`**:
   - **Objetivo**: Visualizar el mapa de pases del portero del equipo local.
   - **Detalles**:
     - Se filtra el DataFrame `df` para obtener los datos del portero local y sus acciones de pase.
     - Se diferencian los pases en:
       - **Pases en juego abierto** (excluyendo tiros de meta y tiros libres).
       - **Pases de tiro de meta o tiro libre**.
     - Se dibuja el campo de fútbol y se representan gráficamente los pases, diferenciando entre exitosos y fallidos mediante colores y marcadores específicos.
     - Se calculan y muestran estadísticas de promedio de longitud y altura de los pases.
     - Se añade un título con el nombre abreviado del portero y se muestra un resumen de los pases recibidos.

2. **Función `away_gk(ax)`**:
   - **Objetivo**: Visualizar el mapa de pases del portero del equipo visitante.
   - **Detalles**:
     - Similar a la función `home_gk`, pero se invierte la dirección del campo para adaptarse a la perspectiva del equipo visitante.
     - Se filtran los pases del portero visitante y se representan gráficamente de la misma manera.
     - Se calculan y muestran las mismas estadísticas de promedio de longitud y altura de los pases, junto con un resumen textual.
     - Se añade un título con el nombre abreviado del portero visitante.

Ambas funciones están diseñadas para proporcionar una representación visual clara de las contribuciones del portero en términos de pases durante el partido.

In [ ]:
def home_gk(ax):
    df_gk = df[(df['teamName']==hteamName) & (df['position']=='GK')]
    gk_pass = df_gk[df_gk['type']=='Pass']
    op_pass = df_gk[(df_gk['type']=='Pass') & (~df_gk['qualifiers'].str.contains('GoalKick|FreekickTaken'))]
    sp_pass = df_gk[(df_gk['type']=='Pass') &  (df_gk['qualifiers'].str.contains('GoalKick|FreekickTaken'))]
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)
    gk_name = df_gk['shortName'].unique()[0]
    op_succ = sp_succ = 0
    for index, row in op_pass.iterrows():
        if row['outcomeType']=='Successful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=hcol, lw=4, comet=True, alpha=0.5, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=hcol, edgecolor=line_color, zorder=3)
            op_succ += 1
        if row['outcomeType']=='Unsuccessful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=hcol, lw=4, comet=True, alpha=0.5, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=bg_color, edgecolor=hcol, zorder=3)
    for index, row in sp_pass.iterrows():
        if row['outcomeType']=='Successful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=violet, lw=4, comet=True, alpha=0.5, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=violet, edgecolor=line_color, zorder=3)
            sp_succ += 1
        if row['outcomeType']=='Unsuccessful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=violet, lw=4, comet=True, alpha=0.35, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=bg_color, edgecolor=violet, zorder=3)

    gk_pass['length'] = np.sqrt((gk_pass['x']-gk_pass['endX'])**2 + (gk_pass['y']-gk_pass['endY'])**2)
    avg_len = gk_pass['length'].mean().round(2)
    avg_hgh = gk_pass['endX'].mean().round(2)
    
    ax.set_title(f'{gk_name} PassMap', color=hcol, fontsize=25, fontweight='bold', y=1.07)
    ax.text(52.5, -3, f'Avg. Passing Length: {avg_len}     |     Avg. Passign Height: {avg_hgh}', color=line_color, fontsize=15, ha='center', va='center')
    ax_text(52.5, 70, s=f'<Open-play Pass (Acc.): {len(op_pass)} ({op_succ})>     |     <GoalKick/Freekick (Acc.): {len(sp_pass)} ({sp_succ})>', 
            fontsize=15, highlight_textprops=[{'color':hcol}, {'color':violet}], ha='center', va='center', ax=ax)

    return

def away_gk(ax):
    df_gk = df[(df['teamName']==ateamName) & (df['position']=='GK')]
    gk_pass = df_gk[df_gk['type']=='Pass']
    op_pass = df_gk[(df_gk['type']=='Pass') & (~df_gk['qualifiers'].str.contains('GoalKick|FreekickTaken'))]
    sp_pass = df_gk[(df_gk['type']=='Pass') &  (df_gk['qualifiers'].str.contains('GoalKick|FreekickTaken'))]
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)
    ax.invert_xaxis()
    ax.invert_yaxis()
    gk_name = df_gk['shortName'].unique()[0]
    op_succ = sp_succ = 0
    for index, row in op_pass.iterrows():
        if row['outcomeType']=='Successful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=acol, lw=4, comet=True, alpha=0.5, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=acol, edgecolor=line_color, zorder=3)
            op_succ += 1
        if row['outcomeType']=='Unsuccessful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=acol, lw=4, comet=True, alpha=0.5, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=bg_color, edgecolor=acol, zorder=3)
    for index, row in sp_pass.iterrows():
        if row['outcomeType']=='Successful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=violet, lw=4, comet=True, alpha=0.5, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=violet, edgecolor=line_color, zorder=3)
            sp_succ += 1
        if row['outcomeType']=='Unsuccessful':
            pitch.lines(row['x'], row['y'], row['endX'], row['endY'], color=violet, lw=4, comet=True, alpha=0.35, zorder=2, ax=ax)
            ax.scatter(row['endX'], row['endY'], s=40, color=bg_color, edgecolor=violet, zorder=3)

    gk_pass['length'] = np.sqrt((gk_pass['x']-gk_pass['endX'])**2 + (gk_pass['y']-gk_pass['endY'])**2)
    avg_len = gk_pass['length'].mean().round(2)
    avg_hgh = gk_pass['endX'].mean().round(2)

    ax.set_title(f'{gk_name} PassMap', color=acol, fontsize=25, fontweight='bold', y=1.07)
    ax.text(52.5, 71, f'Avg. Passing Length: {avg_len}     |     Avg. Passing Height: {avg_hgh}', color=line_color, fontsize=15, ha='center', va='center')
    ax_text(52.5, -2, s=f'<Open-play Pass (Acc.): {len(op_pass)} ({op_succ})>     |     <GoalKick/Freekick (Acc.): {len(sp_pass)} ({sp_succ})>', 
            fontsize=15, highlight_textprops=[{'color':acol}, {'color':violet}], ha='center', va='center', ax=ax)

    return

fig,ax=plt.subplots(figsize=(10,10), facecolor=bg_color)
home_gk(ax)

## Gráfico de barras (Bar Charts)


### Resumen de Funciones

1. **Función `sh_sq_bar(ax)`**:
   - **Objetivo**: Crear un gráfico de barras horizontales que muestre la participación de los jugadores en las secuencias de tiro.
   - **Detalles**:
     - Se seleccionan los 10 jugadores con menor participación total en las secuencias de tiro.
     - Se organizan las estadísticas de tiros, asistencias y construcciones de tiro para estos jugadores.
     - Se utilizan gráficos de barras apiladas para representar:
       - Tiros
       - Asistencias a tiro
       - Construcción de tiro
     - Se añaden etiquetas con las cuentas en el medio de las barras.
     - Se trazan líneas de referencia en el eje x.
     - Se personalizan colores y estilos del gráfico, incluyendo el título y la leyenda.

2. **Función `passer_bar(ax)`**:
   - **Objetivo**: Visualizar las estadísticas de los 10 mejores pasadores en un gráfico de barras horizontales.
   - **Detalles**:
     - Se seleccionan los 10 mejores jugadores en base a su participación total.
     - Se agrupan las estadísticas de pases progresivos, llevadas progresivas y pases rompientes.
     - Se crean gráficos de barras apiladas para mostrar cada tipo de pase.
     - Se añaden etiquetas con los conteos correspondientes.
     - Se ajustan colores y estilos visuales, incluyendo título y leyenda.

3. **Función `defender_bar(ax)`**:
   - **Objetivo**: Crear un gráfico de barras horizontales que ilustre las acciones defensivas de los 10 mejores defensores.
   - **Detalles**:
     - Se seleccionan los 10 defensores con mayor cantidad de acciones defensivas.
     - Se agrupan las estadísticas de tackles, intercepciones y despejes.
     - Se representan con gráficos de barras apiladas.
     - Se incluyen etiquetas de conteo y se ajustan los estilos visuales, título y leyenda.

4. **Función `threat_creators(ax)`**:
   - **Objetivo**: Visualizar los jugadores que generan más "expected threat" (xT) en un gráfico de barras horizontales.
   - **Detalles**:
     - Se seleccionan los 10 jugadores con el mayor xT total.
     - Se diferencian las contribuciones de xT por pase y por llevar el balón.
     - Se representan con gráficos de barras apiladas.
     - Se añaden etiquetas de conteo y se personalizan los estilos visuales, título y leyenda.

Estas funciones están diseñadas para proporcionar una visualización clara y efectiva de las contribuciones de los jugadores en diferentes aspectos del juego, utilizando gráficos de barras para facilitar la comparación.

In [ ]:
from matplotlib.ticker import MaxNLocator
def sh_sq_bar(ax):
  top10_sh_sq = sh_sq_df.nsmallest(10, 'total')['shortName'].tolist()

  shsq_sh = sh_sq_df.nsmallest(10, 'total')['Shots'].tolist()
  shsq_sa = sh_sq_df.nsmallest(10, 'total')['Shot Assist'].tolist()
  shsq_bs = sh_sq_df.nsmallest(10, 'total')['Buildup to shot'].tolist()

  left1 = [w + x for w, x in zip(shsq_sh, shsq_sa)]

  ax.barh(top10_sh_sq, shsq_sh, label='Shot', color=col1, left=0)
  ax.barh(top10_sh_sq, shsq_sa, label='Shot Assist', color=violet, left=shsq_sh)
  ax.barh(top10_sh_sq, shsq_bs, label='Buildup to Shot', color=col2, left=left1)

  # Add counts in the middle of the bars (if count > 0)
  for i, player in enumerate(top10_sh_sq):
      for j, count in enumerate([shsq_sh[i], shsq_sa[i], shsq_bs[i]]):
          if count > 0:
              x_position = sum([shsq_sh[i], shsq_sa[i]][:j]) + count / 2
              ax.text(x_position, i, str(count), ha='center', va='center', color=bg_color, fontsize=13, fontweight='bold')

  max_x = sh_sq_df['total'].iloc()[0]
  x_coord = [2 * i for i in range(1, int(max_x/2))]
  for x in x_coord:
      ax.axvline(x=x, color='gray', linestyle='--', zorder=2, alpha=0.5)

  ax.set_facecolor(bg_color)
  ax.tick_params(axis='x', colors=line_color, labelsize=15)
  ax.tick_params(axis='y', colors=line_color, labelsize=15)
  ax.xaxis.label.set_color(line_color)
  ax.yaxis.label.set_color(line_color)
  for spine in ax.spines.values():
    spine.set_edgecolor(bg_color)

  ax.set_title(f"Shot Sequence Involvement", color=line_color, fontsize=25, fontweight='bold')
  ax.legend()

def passer_bar(ax):
  top10_passers = progressor_df.nsmallest(10, 'total')['shortName'].tolist()

  passers_pp = progressor_df.nsmallest(10, 'total')['Progressive Passes'].tolist()
  passers_tp = progressor_df.nsmallest(10, 'total')['Progressive Carries'].tolist()
  passers_kp = progressor_df.nsmallest(10, 'total')['LineBreaking Pass'].tolist()

  left1 = [w + x for w, x in zip(passers_pp, passers_tp)]

  ax.barh(top10_passers, passers_pp, label='Prog. Pass', color=col1, left=0)
  ax.barh(top10_passers, passers_tp, label='Prog. Carries', color=col2, left=passers_pp)
  ax.barh(top10_passers, passers_kp, label='Through Pass', color=violet, left=left1)

  # Add counts in the middle of the bars (if count > 0)
  for i, player in enumerate(top10_passers):
      for j, count in enumerate([passers_pp[i], passers_tp[i], passers_kp[i]]):
          if count > 0:
              x_position = sum([passers_pp[i], passers_tp[i]][:j]) + count / 2
              ax.text(x_position, i, str(count), ha='center', va='center', color=bg_color, fontsize=13, fontweight='bold')

  max_x = progressor_df['total'].iloc()[0]
  x_coord = [2 * i for i in range(1, int(max_x/2))]
  for x in x_coord:
      ax.axvline(x=x, color='gray', linestyle='--', zorder=2, alpha=0.5)

  ax.set_facecolor(bg_color)
  ax.tick_params(axis='x', colors=line_color, labelsize=15)
  ax.tick_params(axis='y', colors=line_color, labelsize=15)
  ax.xaxis.label.set_color(line_color)
  ax.yaxis.label.set_color(line_color)
  for spine in ax.spines.values():
    spine.set_edgecolor(bg_color)

  ax.set_title(f"Top10 Ball Progressors", color=line_color, fontsize=25, fontweight='bold')
  ax.legend()


def defender_bar(ax):
  top10_defenders = defender_df.nsmallest(10, 'total')['shortName'].tolist()

  defender_tk = defender_df.nsmallest(10, 'total')['Tackles'].tolist()
  defender_in = defender_df.nsmallest(10, 'total')['Interceptions'].tolist()
  defender_ar = defender_df.nsmallest(10, 'total')['Clearance'].tolist()

  left1 = [w + x for w, x in zip(defender_tk, defender_in)]

  ax.barh(top10_defenders, defender_tk, label='Tackle', color=col1, left=0)
  ax.barh(top10_defenders, defender_in, label='Interception', color=violet, left=defender_tk)
  ax.barh(top10_defenders, defender_ar, label='Clearance', color=col2, left=left1)

  # Add counts in the middle of the bars (if count > 0)
  for i, player in enumerate(top10_defenders):
      for j, count in enumerate([defender_tk[i], defender_in[i], defender_ar[i]]):
          if count > 0:
              x_position = sum([defender_tk[i], defender_in[i]][:j]) + count / 2
              ax.text(x_position, i, str(count), ha='center', va='center', color=bg_color, fontsize=13, fontweight='bold')

  max_x = defender_df['total'].iloc()[0]
  x_coord = [2 * i for i in range(1, int(max_x/2))]
  for x in x_coord:
      ax.axvline(x=x, color='gray', linestyle='--', zorder=2, alpha=0.5)

  ax.set_facecolor(bg_color)
  ax.tick_params(axis='x', colors=line_color, labelsize=15)
  ax.tick_params(axis='y', colors=line_color, labelsize=15)
  ax.xaxis.label.set_color(line_color)
  ax.yaxis.label.set_color(line_color)
  for spine in ax.spines.values():
    spine.set_edgecolor(bg_color)


  ax.set_title(f"Top10 Defenders", color=line_color, fontsize=25, fontweight='bold')
  ax.legend()


def threat_creators(ax):
  top10_xT = xT_df.nsmallest(10, 'total')['shortName'].tolist()

  xT_pass = xT_df.nsmallest(10, 'total')['xT from Pass'].tolist()
  xT_carry = xT_df.nsmallest(10, 'total')['xT from Carry'].tolist()

  left1 = [w + x for w, x in zip(xT_pass, xT_carry)]

  ax.barh(top10_xT, xT_pass, label='xT_pass', color=col1, left=0)
  ax.barh(top10_xT, xT_carry, label='xT_carry', color=violet, left=xT_pass)

  # Add counts in the middle of the bars (if count > 0)
  for i, player in enumerate(top10_xT):
      for j, count in enumerate([xT_pass[i], xT_carry[i]]):
          if count > 0:
              x_position = sum([xT_pass[i], xT_carry[i]][:j]) + count / 2
              ax.text(x_position, i, str(count), ha='center', va='center', color=line_color, fontsize=13, rotation=45, fontweight='bold')

  # max_x = xT_df['total'].iloc()[0]
  # x_coord = [2 * i for i in range(1, int(max_x/2))]
  # for x in x_coord:
  #     ax.axvline(x=x, color='gray', linestyle='--', zorder=2, alpha=0.5)

  ax.set_facecolor(bg_color)
  ax.tick_params(axis='x', colors=line_color, labelsize=15)
  ax.tick_params(axis='y', colors=line_color, labelsize=15)
  ax.xaxis.label.set_color(line_color)
  ax.yaxis.label.set_color(line_color)
  for spine in ax.spines.values():
    spine.set_edgecolor(bg_color)


  ax.set_title(f"Top10 Threatening Players", color=line_color, fontsize=25, fontweight='bold')
  ax.legend()

fig,ax=plt.subplots(figsize=(10,10))
sh_sq_bar(ax)

# Top Players Dashboard


### Resumen del Código

El código crea una visualización de los mejores jugadores del partido utilizando un conjunto de gráficos organizados en un layout de 4 filas y 3 columnas. A continuación se detallan las partes principales:

1. **Creación de la Figura y Ejes**:
   - Se crea una figura (`fig`) con subgráficos (`axs`) de tamaño 35x35 utilizando un fondo de color personalizado (`bg_color`).

2. **Visualizaciones en los Subgráficos**:
   - Cada subgráfico representa diferentes estadísticas o análisis de jugadores:
     - **Fila 1**:
       - **Pasos del jugador local** (`home_player_passmap`): Muestra el mapa de pases del mejor pasador del equipo local.
       - **Gráfico de Pasadores** (`passer_bar`): Muestra las estadísticas de los pasadores del partido.
       - **Pasos del jugador visitante** (`away_player_passmap`): Muestra el mapa de pases del mejor pasador del equipo visitante.
     - **Fila 2**:
       - **Pases recibidos por el jugador local** (`home_passes_recieved`): Visualiza los pases que ha recibido el mejor delantero del equipo local.
       - **Participación en Secuencias de Tiro** (`sh_sq_bar`): Muestra las estadísticas de participación en las secuencias de tiro de los jugadores.
       - **Pases recibidos por el jugador visitante** (`away_passes_recieved`): Visualiza los pases que ha recibido el mejor delantero del equipo visitante.
     - **Fila 3**:
       - **Acciones defensivas del jugador local** (`home_player_def_acts`): Muestra las acciones defensivas del mejor defensor del equipo local.
       - **Gráfico de Defensores** (`defender_bar`): Muestra las estadísticas de los defensores del partido.
       - **Acciones defensivas del jugador visitante** (`away_player_def_acts`): Muestra las acciones defensivas del mejor defensor del equipo visitante.
     - **Fila 4**:
       - **Portero local** (`home_gk`): Muestra las estadísticas de pases del portero del equipo local.
       - **Creadores de Amenazas** (`threat_creators`): Muestra las estadísticas de los jugadores que generan más amenazas.
       - **Portero visitante** (`away_gk`): Muestra las estadísticas de pases del portero del equipo visitante.

3. **Títulos y Subtítulos**:
   - Se añade un título general que muestra el resultado del partido.
   - Se incluyen subtítulos que indican la jornada, la liga y los datos de origen.

4. **Dirección de Ataque**:
   - Se añaden textos que indican la dirección del ataque para ambos equipos.

5. **Logos de Equipos**:
   - Se cargan e insertan las imágenes de los logotipos de los equipos locales y visitantes.

6. **Guardar la Figura**:
   - Se incluye una línea de código (comentada) para guardar la figura como una imagen PNG con un nombre basado en el ID del partido.

Este código proporciona una visualización completa y detallada del rendimiento de los mejores jugadores en un partido, mostrando diferentes aspectos del juego de una manera clara y atractiva.

In [ ]:
fig, axs = plt.subplots(4,3, figsize=(35,35), facecolor=bg_color)

home_player_passmap(axs[0,0])
passer_bar(axs[0,1])
away_player_passmap(axs[0,2])
home_passes_recieved(axs[1,0])
sh_sq_bar(axs[1,1])
away_passes_recieved(axs[1,2])
home_player_def_acts(axs[2,0])
defender_bar(axs[2,1])
away_player_def_acts(axs[2,2])
home_gk(axs[3,0])
threat_creators(axs[3,1])
away_gk(axs[3,2])

# Headings
highlight_text = [{'color':hcol}, {'color':acol}]
fig_text(0.5, 0.98, f"<{hteamName} {hgoal_count}> - <{agoal_count} {ateamName}>", color=line_color, fontsize=70, fontweight='bold',
            highlight_textprops=highlight_text, ha='center', va='center', ax=fig)

# Subtitles
fig.text(0.5, 0.95, f"GW 1 , EPL 2024-25 | Top Players of the Match", color=line_color, fontsize=30, ha='center', va='center')
fig.text(0.5, 0.93, f"Data from: Opta | code by: @adnaaan433", color=line_color, fontsize=22.5, ha='center', va='center')

fig.text(0.125,0.097, 'Attacking Direction ------->', color=hcol, fontsize=25, ha='left', va='center')
fig.text(0.9,0.097, '<------- Attacking Direction', color=acol, fontsize=25, ha='right', va='center')

# himage_url = himage_url.replace(' ', '%20')
himage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{hometeamId.iloc[0]['teamId']}.png")
himage = Image.open(himage)
ax_himage = add_image(himage, fig, left=0.125, bottom=0.94, width=0.05, height=0.05)

# aimage_url = aimage_url.replace(' ', '%20')
aimage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{awayteamId.iloc[0]['teamId']}.png")
aimage = Image.open(aimage)
ax_aimage = add_image(aimage, fig, left=0.85, bottom=0.94, width=0.05, height=0.05)

# plt.savefig(f"{match_id}_Top_Players_Dashboard.png", bbox_inches='tight')

# Individual Player Dashboard Functions


### Resumen de la Función `individual_passMap`

La función `individual_passMap` genera un mapa de pases para un jugador específico en un partido de fútbol, utilizando una visualización de un campo de fútbol. Los pasos principales de la función son:

1. **Configuración del Campo**:
   - Se crea un campo de fútbol utilizando la biblioteca `Pitch`, con un diseño de tipo UEFA, colores de fondo y líneas personalizados.
   - Se establece el rango de los ejes del gráfico.

2. **Filtrado de Datos**:
   - Se filtran los datos para obtener solo los pases realizados por el jugador cuyo nombre se pasa como argumento (`pname`).
   - Se dividen los pases en varias categorías: 
     - **Pases exitosos**.
     - **Pases no exitosos**.
     - **Pases progresivos** (pases que cumplen ciertos criterios).
     - **Pases clave**, **asistencias**, **balones recuperados**, **pases largos**, **pases cruzados**, y otros tipos relevantes.

3. **Cálculos Estadísticos**:
   - Se calculan métricas como el porcentaje de precisión en los pases, la longitud promedio de los pases, y el xT (Expected Threat) total generado por los pases del jugador.

4. **Visualización de Pases**:
   - Se dibujan líneas para representar los pases exitosos y no exitosos en el campo, utilizando diferentes colores para diferenciarlos:
     - Pases exitosos en color principal.
     - Pases progresivos en un color específico.
     - Pases clave y asistencias en otros colores.
   - Se representan los puntos finales de los pases en el campo como puntos.

5. **Título y Estadísticas**:
   - Se establece un título en el gráfico que muestra el nombre del jugador.
   - Se agregan textos que indican estadísticas importantes sobre los pases realizados, incluyendo la cantidad total de pases, la cantidad de pases progresivos, las oportunidades creadas, las asistencias, y más.

6. **Retorno**:
   - La función no retorna ningún valor; su objetivo es crear una visualización en el eje proporcionado.

### Uso
La función se puede llamar con un eje específico y el nombre del jugador para visualizar su mapa de pases en un gráfico.

Este resumen debería ayudarte a comprender rápidamente la funcionalidad de la función `individual_passMap` y cómo contribuye a la visualización del rendimiento de un jugador en el contexto de sus pases durante un partido de fútbol.

In [ ]:
df.head()

In [ ]:
pname = 'Kevin De Bruyne'

In [ ]:
def individual_passMap(ax, pname):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)

    dfpass = df[(df['type']=='Pass') & (df['name']==pname)]
    acc_pass = dfpass[dfpass['outcomeType']=='Successful']
    iac_pass = dfpass[dfpass['outcomeType']=='Unsuccessful']
    
    if len(dfpass) != 0:
        accurate_pass_perc = round((len(acc_pass)/len(dfpass))*100, 2)
    else:
        accurate_pass_perc = 0
    
    pro_pass = acc_pass[(acc_pass['prog_pass']>=9.11) & (acc_pass['x']>=35) &
                        (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Thr_ball = dfpass[(dfpass['qualifiers'].str.contains('Throughball')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Thr_ball_acc = Thr_ball[Thr_ball['outcomeType']=='Successful']
    Lng_ball = dfpass[(dfpass['qualifiers'].str.contains('Longball')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Lng_ball_acc = Lng_ball[Lng_ball['outcomeType']=='Successful']
    Crs_pass = dfpass[(dfpass['qualifiers'].str.contains('Cross')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Crs_pass_acc = Crs_pass[Crs_pass['outcomeType']=='Successful']
    key_pass = dfpass[dfpass['qualifiers'].str.contains('KeyPass')]
    big_chnc = dfpass[dfpass['qualifiers'].str.contains('BigChanceCreated')]
    df_no_carry = df[df['type']!='Carry'].reset_index(drop=True)
    pre_asst = df_no_carry[(df_no_carry['qualifiers'].shift(-1).str.contains('IntentionalGoalAssist')) & (df_no_carry['type']=='Pass') & 
                           (df_no_carry['outcomeType']=='Successful') &  (df_no_carry['name']==pname)]
    shot_buildup = df_no_carry[(df_no_carry['qualifiers'].shift(-1).str.contains('KeyPass')) & (df_no_carry['type']=='Pass') & 
                           (df_no_carry['outcomeType']=='Successful') &  (df_no_carry['name']==pname)]
    g_assist = dfpass[dfpass['qualifiers'].str.contains('IntentionalGoalAssist')]
    fnl_thd = acc_pass[(acc_pass['endX']>=70) & (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    pen_box = acc_pass[(acc_pass['endX']>=88.5) & (acc_pass['endY']>=13.6) & (acc_pass['endY']<=54.4) & 
                       (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    frwd_pass = dfpass[(dfpass['pass_or_carry_angle']>= -85) & (dfpass['pass_or_carry_angle']<= 85) & 
                       (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    back_pass = dfpass[((dfpass['pass_or_carry_angle']>= 95) & (dfpass['pass_or_carry_angle']<= 180) | 
                        (dfpass['pass_or_carry_angle']>= -180) & (dfpass['pass_or_carry_angle']<= -95)) &
                       (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    side_pass = dfpass[((dfpass['pass_or_carry_angle']>= 85) & (dfpass['pass_or_carry_angle']<= 95) | 
                        (dfpass['pass_or_carry_angle']>= -95) & (dfpass['pass_or_carry_angle']<= -85)) & 
                       (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    frwd_pass_acc = frwd_pass[frwd_pass['outcomeType']=='Successful']
    back_pass_acc = back_pass[back_pass['outcomeType']=='Successful']
    side_pass_acc = side_pass[side_pass['outcomeType']=='Successful']
    corners = dfpass[dfpass['qualifiers'].str.contains('CornerTaken')]
    corners_acc = corners[corners['outcomeType']=='Successful']
    freekik = dfpass[dfpass['qualifiers'].str.contains('Freekick')]
    freekik_acc = freekik[freekik['outcomeType']=='Successful']
    thins = dfpass[dfpass['qualifiers'].str.contains('ThrowIn')]
    thins_acc = thins[thins['outcomeType']=='Successful']
    lngball = dfpass[(dfpass['qualifiers'].str.contains('Longball')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    lngball_acc = lngball[lngball['outcomeType']=='Successful']

    if len(frwd_pass) != 0:
        Forward_Pass_Accuracy = round((len(frwd_pass_acc)/len(frwd_pass))*100, 2)
    else:
        Forward_Pass_Accuracy = 0
    
    df_xT_inc = dfpass[dfpass['xT']>0]
    df_xT_dec = dfpass[dfpass['xT']<0]
    xT_by_Pass = dfpass['xT'].sum().round(2)

    pitch.lines(iac_pass.x, iac_pass.y, iac_pass.endX, iac_pass.endY, color=line_color, lw=4, alpha=0.15, comet=True, zorder=2, ax=ax)
    pitch.lines(acc_pass.x, acc_pass.y, acc_pass.endX, acc_pass.endY, color=line_color, lw=2, alpha=0.15, comet=True, zorder=2, ax=ax)
    pitch.lines(pro_pass.x, pro_pass.y, pro_pass.endX, pro_pass.endY, color=hcol      , lw=3, alpha=1,    comet=True, zorder=3, ax=ax)
    pitch.lines(key_pass.x, key_pass.y, key_pass.endX, key_pass.endY, color=violet,     lw=4, alpha=1,    comet=True, zorder=4, ax=ax)
    pitch.lines(g_assist.x, g_assist.y, g_assist.endX, g_assist.endY, color='green',      lw=4, alpha=1,    comet=True, zorder=5, ax=ax)

    ax.scatter(acc_pass.endX, acc_pass.endY, s=30, color=bg_color,    edgecolor='gray', alpha=1, zorder=2)
    ax.scatter(pro_pass.endX, pro_pass.endY, s=40, color=bg_color,  edgecolor= hcol,  alpha=1, zorder=3)
    ax.scatter(key_pass.endX, key_pass.endY, s=50, color=bg_color,  edgecolor=violet, alpha=1, zorder=4)
    ax.scatter(g_assist.endX, g_assist.endY, s=50, color=bg_color,  edgecolor= 'green', alpha=1, zorder=5)


    ax.set_title(f"PassMap", color=line_color, fontsize=25, fontweight='bold', y=1.03)
    ax_text(0, -3, f'''Accurate Pass: {len(acc_pass)}/{len(dfpass)} ({accurate_pass_perc}%) | <Progressive Pass: {len(pro_pass)}> | <Chances Created: {len(key_pass)}>
Big Chances Created: {len(big_chnc)} | <Assists: {len(g_assist)}> | Pre-Assist: {len(pre_asst)} | Build-up to Shot: {len(shot_buildup)}
Final third Passes: {len(fnl_thd)} | Passes into Penalty box: {len(pen_box)} | Crosses (Acc.): {len(Crs_pass)} ({len(Crs_pass_acc)})
Longballs (Acc.): {len(lngball)} ({len(lngball_acc)}) | xT from Pass: {xT_by_Pass}
''', color=line_color, highlight_textprops=[{'color':hcol}, {'color':violet}, {'color':'green'}], fontsize=15, ha='left', va='top', ax=ax)
    # Open-Play Forward Pass (Acc.): {len(frwd_pass)} ({len(frwd_pass_acc)})
    # Open-Play Side Pass (Acc.): {len(side_pass)} ({len(side_pass_acc)})
    # Open-Play Back Pass (Acc.): {len(back_pass)} ({len(back_pass_acc)})
    # xT decreased as passer: {df_xT_dec['xT'].sum().round(2)}
    
    return


fig,ax=plt.subplots(figsize=(10,10))
individual_passMap(ax, pname)


### Resumen de la Función `individual_carry`

La función `individual_carry` genera un mapa visual que muestra las acciones de "carries" (llevadas del balón) de un jugador específico en un partido de fútbol, utilizando una visualización de un campo de fútbol. A continuación se describen las etapas clave de la función:

1. **Configuración del Campo**:
   - Se crea un campo de fútbol utilizando la clase `Pitch` con un diseño de tipo UEFA y se establecen los límites de los ejes del gráfico.

2. **Filtrado de Datos**:
   - Se filtran los datos para obtener solo las "carries" realizadas por el jugador cuyo nombre se pasa como argumento (`pname`).
   - Se clasifican las carries en diferentes categorías:
     - **Llevadas que llevan a un tiro** (separando entre tiros a puerta y tiros de cualquier tipo).
     - **Llevadas que llevan a un gol** (separando entre asistencias y goles).
     - **Llevadas progresivas** (que cumplen ciertos criterios).
     - **Llevadas que entran en el área final** y otras características relevantes.

3. **Cálculos Estadísticos**:
   - Se calculan métricas como la longitud promedio de las carries y el total de longitud de todas las carries.
   - Se calcula la tasa de éxito de los "take-ons" (intentos de desbordar al defensor).

4. **Visualización de Carry**:
   - Se dibujan flechas que representan cada "carry" en el campo, utilizando diferentes colores para distinguir entre:
     - **Llevadas normales** en color claro.
     - **Llevadas progresivas** en un color específico.
     - **Llevadas que llevan a un tiro** en color violeta.
     - **Llevadas que llevan a un gol** en color verde.
   - Se representan puntos para los "take-ons" exitosos y no exitosos.

5. **Título y Estadísticas**:
   - Se establece un título en el gráfico que muestra las acciones de "carries" del jugador.
   - Se agregan textos que indican estadísticas importantes sobre las carries, como el total de carries, las que llevaron a tiros o goles, la longitud promedio, el xT (Expected Threat) generado, y el éxito en los "take-ons".

6. **Retorno**:
   - La función no retorna ningún valor; su objetivo es crear una visualización en el eje proporcionado.

### Uso
La función se puede invocar pasando un eje específico y el nombre del jugador para visualizar sus carries en un gráfico.

Este resumen debería ayudarte a comprender cómo funciona la función `individual_carry` y cómo contribuye a la visualización del rendimiento de un jugador en el contexto de sus acciones con el balón durante un partido de fútbol.

In [ ]:
def individual_carry(ax,pname):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5,105.5)
    ax.set_ylim(-0.5,68.5)

    df_carry = df[(df['type']=='Carry') & (df['name']==pname)]
    led_shot1 = df[(df['type']=='Carry') & (df['name']==pname) & (df['qualifiers'].shift(-1).str.contains('KeyPass'))]
    led_shot2 = df[(df['type']=='Carry') & (df['name']==pname) & (df['type'].shift(-1).str.contains('Shot'))]
    led_shot = pd.concat([led_shot1, led_shot2])
    led_goal1 = df[(df['type']=='Carry') & (df['name']==pname) & (df['qualifiers'].shift(-1).str.contains('IntentionalGoalAssist'))]
    led_goal2 = df[(df['type']=='Carry') & (df['name']==pname) & (df['type'].shift(-1)=='Goal')]
    led_goal = pd.concat([led_goal1, led_goal2])
    pro_carry = df_carry[(df_carry['prog_carry']>=9.11) & (df_carry['x']>=35)]
    fth_carry = df_carry[(df_carry['x']<70) & (df_carry['endX']>=70)]
    box_entry = df_carry[(df_carry['endX']>=88.5) & (df_carry['endY']>=13.6) & (df_carry['endY']<=54.4) &
                 ~((df_carry['x']>=88.5) & (df_carry['y']>=13.6) & (df_carry['y']<=54.6))]
    disp = df[(df['type']=='Carry') & (df['name']==pname) & (df['type'].shift(-1)=='Dispossessed')]
    df_to = df[(df['type']=='TakeOn') & (df['name']==pname)]
    t_ons = df_to[df_to['outcomeType']=='Successful']
    t_onu = df_to[df_to['outcomeType']=='Unsuccessful']
    df_xT_inc = df_carry[df_carry['xT']>0]
    df_xT_dec = df_carry[df_carry['xT']<0]
    xT_by_Carry = df_carry['xT'].sum().round(2)
    df_carry = df_carry.copy()
    df_carry.loc[:, 'Length'] = np.sqrt((df_carry['x'] - df_carry['endX'])**2 + (df_carry['y'] - df_carry['endY'])**2)
    median_length = round(df_carry['Length'].median(),2)
    total_length = round(df_carry['Length'].sum(),2)
    if len(df_to)!=0:
        success_rate = round((len(t_ons)/len(df_to))*100, 2)
    else:
        success_rate = 0

    for index, row in df_carry.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), color=line_color, alpha=0.25, arrowstyle='->', linestyle='--', 
                                   linewidth=2, mutation_scale=15, zorder=2)
        ax.add_patch(arrow)
    for index, row in pro_carry.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), color=hcol, alpha=1, arrowstyle='->', linestyle='--', linewidth=3, 
                                   mutation_scale=20, zorder=3)
        ax.add_patch(arrow)
    for index, row in led_shot.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), color=violet, alpha=1, arrowstyle='->', linestyle='--', linewidth=4, 
                                   mutation_scale=20, zorder=4)
        ax.add_patch(arrow)
    for index, row in led_goal.iterrows():
        arrow = patches.FancyArrowPatch((row['x'], row['y']), (row['endX'], row['endY']), color='green', alpha=1, arrowstyle='->', linestyle='--', linewidth=4, 
                                   mutation_scale=20, zorder=4)
        ax.add_patch(arrow)

    ax.scatter(t_ons.x, t_ons.y, s=250, color='orange', edgecolor=line_color, lw=2, zorder=5)
    ax.scatter(t_onu.x, t_onu.y, s=250, color='None', edgecolor='orange', hatch='/////', lw=2.5, zorder=5)

    ax.set_title(f"Carries & TakeOns", color='k', fontsize=25, fontweight='bold', y=1.03)
    ax_text(0, -3, f'''Total Carries: {len(df_carry)} | <Progressive Carries: {len(pro_carry)}> | <Carries Led to Shot: {len(led_shot)}>
<Carries Led to Goal: {len(led_goal)}> | Carrier Dispossessed: {len(disp)} | Carries into Final third: {len(fth_carry)}
Carries into pen.box: {len(box_entry)} | Avg. Carry Length: {median_length} m | Total Carry: {total_length} m
xT from Carries: {xT_by_Carry} | <Successful TakeOns: {len(t_ons)}/{len(df_to)} ({success_rate}%)>
''', highlight_textprops=[{'color':hcol}, {'color':violet}, {'color':'green'}, {'color':'darkorange'}], fontsize=15, ha='left', 
            va='top', ax=ax)
    # xT decreased as carrier: {df_xT_dec['xT'].sum().round(2)}
    return

fig,ax=plt.subplots(figsize=(10,10))
individual_carry(ax, pname)

In [ ]:
fig,ax=plt.subplots(figsize=(10,10))
individual_carry(ax, pname)


### Resumen de la Función `Individual_ShotMap`

La función `Individual_ShotMap` se utiliza para crear un mapa de tiros que representa las acciones de tiro de un jugador específico en un partido de fútbol. Aquí están las etapas clave de la función:

1. **Configuración del Campo**:
   - Se inicializa un campo de fútbol utilizando la clase `Pitch`, con configuraciones para el diseño, colores de fondo y líneas. Se establece el límite de los ejes del gráfico.

2. **Filtrado de Datos**:
   - Se filtra el DataFrame `shots_df` para obtener los tiros del jugador cuyo nombre se pasa como argumento (`pname`).
   - Se clasifica cada tiro en diferentes categorías:
     - **Tiros durante el juego regular**.
     - **Goles**.
     - **Tiros fallidos**.
     - **Tiros guardados**.
     - **Tiros bloqueados**.
     - **Tiros en el poste**.
     - **Grandes oportunidades creadas**.

3. **Cálculos Estadísticos**:
   - Se calculan estadísticas como el total de tiros, tiros dentro y fuera del área, goles, y la distancia promedio de los tiros.
   - Se calcula el total de goles esperados (xG) y los goles esperados en tiros a puerta (xGOT).

4. **Visualización de Tiros**:
   - Se representan diferentes tipos de tiros en el gráfico utilizando distintos colores:
     - **Goles** en verde.
     - **Tiros en el poste** y **bloqueados** en color específico (hcol).
     - **Tiros guardados** en color hcol.
     - **Tiros fallidos** en color hcol con un fondo transparente.
   - Se añaden flechas que indican la dirección de cada tiro.

5. **Estadísticas en el Gráfico**:
   - Se añade un título al gráfico y se muestran estadísticas relevantes sobre los tiros en la parte inferior, como el total de tiros, los tiros exitosos y fallidos, y otras métricas importantes relacionadas con el rendimiento del jugador.

6. **Retorno**:
   - La función no retorna ningún valor; su objetivo es crear una visualización en el eje proporcionado.

### Uso
Esta función se puede utilizar pasando un eje específico y el nombre del jugador para visualizar sus tiros en un gráfico, lo que permite analizar el rendimiento del jugador en situaciones de tiro durante un partido.

Este resumen ofrece una visión clara de la función `Individual_ShotMap` y su propósito en la visualización del rendimiento de los jugadores en términos de sus acciones de tiro.

In [ ]:
def Individual_ShotMap(ax,pname):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5,105.5)
    ax.set_ylim(-0.5, 68.5)

    op_sh = shots_df[(shots_df['playerName']==pname) & (shots_df['situation']=='RegularPlay')]

    goal = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='Goal')]
    miss = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='Miss')]
    save = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='AttemptSaved') & (shots_df['isBlocked']==0)]
    blok = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='AttemptSaved') & (shots_df['isBlocked']==1)]
    post = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='Post')]

    goal_bc = df[(df['name']==pname) & (df['type']=='Goal') & (df['qualifiers'].str.contains('BigChance'))]
    miss_bc = df[(df['name']==pname) & (df['type']=='MissedShots') & (df['qualifiers'].str.contains('BigChance'))]
    save_bc = df[(df['name']==pname) & (df['type']=='SavedShot') & (df['qualifiers'].str.contains('BigChance'))]
    post_bc = df[(df['name']==pname) & (df['type']=='ShotOnPost') & (df['qualifiers'].str.contains('BigChance'))]

    shots = df[(df['name']==pname) & ((df['type']=='Goal') | (df['type']=='MissedShots') | (df['type']=='SavedShot') | (df['type']=='ShotOnPost'))]
    out_box = shots[shots['qualifiers'].str.contains('OutOfBox')]
    shots = shots.copy()
    shots.loc[:, 'Length'] = np.sqrt((shots['x'] - 105)**2 + (shots['y'] - 34)**2)
    avg_dist = round(shots['Length'].mean(), 2)
    xG = shots_df[(shots_df['playerName']==pname)]['expectedGoals'].sum().round(2)
    xGOT = shots_df[(shots_df['playerName']==pname)]['expectedGoalsOnTarget'].sum().round(2)

    pitch.scatter(goal.x,goal.y, s=goal['expectedGoals']*1000+100, marker='football', c='None', edgecolors='green', zorder=5, ax=ax)
    pitch.scatter(post.x,post.y, s=post['expectedGoals']*1000+100, marker='o', c='None', edgecolors=hcol, hatch='+++', zorder=4, ax=ax)
    pitch.scatter(blok.x,blok.y, s=blok['expectedGoals']*1000+100, marker='o', c='None', edgecolors=hcol, hatch='/////', zorder=4, ax=ax)
    pitch.scatter(save.x,save.y, s=save['expectedGoals']*1000+100, marker='o', c=hcol, edgecolors=line_color, zorder=3, ax=ax)
    pitch.scatter(miss.x,miss.y, s=miss['expectedGoals']*1000+100, marker='o', c='None', edgecolors=hcol, zorder=2, ax=ax)

    # pitch.scatter(goal_bc.x,goal_bc.y, s=250, marker='football', c='None', edgecolors='green', zorder=5, ax=ax)
    # pitch.scatter(post_bc.x,post_bc.y, s=190, marker='o', c='None', edgecolors=hcol, hatch='/////', zorder=4, ax=ax)
    # pitch.scatter(save_bc.x,save_bc.y, s=190, marker='o', c=hcol, edgecolors=line_color, zorder=3, ax=ax)
    # pitch.scatter(miss_bc.x,miss_bc.y, s=165, marker='o', c='None', edgecolors=hcol, zorder=2, ax=ax)

    yhalf = [-0.5, -0.5, 68.5, 68.5]
    xhalf = [-0.5, 52.5, 52.5, -0.5]
    ax.fill(xhalf, yhalf, bg_color, alpha=1)

    pitch.scatter(2,56-(0*4), s=200, marker='football', c='None', edgecolors='green', zorder=5, ax=ax)
    pitch.scatter(2,56-(1*4), s=150, marker='o', c='None', edgecolors=hcol, hatch='+++', zorder=4, ax=ax)
    pitch.scatter(2,56-(2*4), s=150, marker='o', c=hcol, edgecolors=line_color, zorder=3, ax=ax)
    pitch.scatter(2,56-(3*4), s=130, marker='o', c='None', edgecolors=hcol, zorder=2, ax=ax)
    pitch.scatter(2,56-(4*4), s=130, marker='o', c='None', edgecolors=hcol, hatch='/////', zorder=2, ax=ax)

    ax.text(0, 71, f"Shooting Stats", color=line_color, fontsize=25, fontweight='bold')
    ax.text(7,64-(0*4), f'Total Shots: {len(shots)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(1*4), f'Open-play Shots: {len(op_sh)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(2*4), f'Goals: {len(goal)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(3*4), f'Shot on Post: {len(post)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(4*4), f'Shots on Target: {len(save)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(5*4), f'Shots off Target: {len(miss)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(6*4), f'Shots Blocked: {len(blok)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(7*4), f'Big Chances: {len(goal_bc)+len(miss_bc)+len(save_bc)+len(post_bc)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(8*4), f'Big Chances Missed: {len(miss_bc)+len(save_bc)+len(post_bc)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(9*4), f'Shots outside box: {len(out_box)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(10*4), f'Shots inside box: {len(shots) - len(out_box)}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(11*4), f'Avg. Shot Distance: {avg_dist} m', fontsize=15, ha='left', va='center')
    ax.text(7,64-(12*4), f'xG: {xG}', fontsize=15, ha='left', va='center')
    ax.text(7,64-(13*4), f'xGOT: {xGOT}', fontsize=15, ha='left', va='center')

    ax.text(80, 71, f"Shot Map", color=line_color, fontsize=25, fontweight='bold', ha='center')
    return

fig,ax=plt.subplots(figsize=(10,10))
Individual_ShotMap(ax, pname)

In [ ]:
fig,ax=plt.subplots(figsize=(10,10))
Individual_ShotMap(ax, pname)


### Resumen de la Función `individual_passes_recieved`

La función `individual_passes_recieved` se utiliza para crear un gráfico que muestra las acciones de recepción de pases de un jugador específico en un partido de fútbol. A continuación se describen las etapas clave de la función:

1. **Configuración del Campo**:
   - Se inicializa un campo de fútbol utilizando la clase `Pitch`, configurando su diseño, colores de fondo y líneas. Se establecen los límites de los ejes del gráfico.

2. **Filtrado de Datos**:
   - Se filtra el DataFrame `df` para obtener solo los pases exitosos que el jugador especificado (`pname`) ha recibido.
   - Se clasifican los pases recibidos en diferentes categorías:
     - **Pases clave recibidos**.
     - **Asistencias recibidas**.
     - **Pases recibidos en el último tercio**.
     - **Pases recibidos dentro del área del oponente**.
     - **Pases progresivos recibidos**.
     - **Centros recibidos**.
     - **Largueros recibidos**.
     - **Pases de retorno de corte**.

3. **Cálculos Estadísticos**:
   - Se calcula la tasa de retención de balones (proporción de pases exitosos recibidos respecto al total de pases recibidos).
   - Se determina el jugador que le pasó más veces el balón al jugador especificado y se cuentan las veces que se produjo esta acción.

4. **Visualización de Pases**:
   - Se representan los pases en el gráfico utilizando diferentes colores:
     - **Pases recibidos** en un color específico (hcol).
     - **Pases clave** en violeta.
     - **Asistencias** en verde.
   - Se añaden líneas que indican la dirección de los pases recibidos.

5. **Estadísticas en el Gráfico**:
   - Se agrega un título al gráfico y se muestran estadísticas relevantes sobre los pases en la parte inferior, como el número total de pases recibidos, pases clave, asistencias, y otras métricas importantes.

6. **Retorno**:
   - La función no retorna ningún valor; su objetivo es crear una visualización en el eje proporcionado.

### Uso
Esta función se puede utilizar para analizar la recepción de pases de un jugador en un partido, proporcionando información visual sobre su rendimiento en este aspecto del juego. Esto es útil para entrenadores, analistas y aficionados que deseen evaluar la eficacia del jugador en la recepción de pases.

Este resumen resume las características y el propósito de la función `individual_passes_recieved`, destacando su utilidad en el análisis del rendimiento de los jugadores.

In [ ]:
def individual_passes_recieved(ax, pname):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5, 68.5)

    
    dfp = df[(df['type']=='Pass') & (df['outcomeType']=='Successful') & (df['name'].shift(-1)==pname)]
    dfkp = dfp[dfp['qualifiers'].str.contains('KeyPass')]
    dfas = dfp[dfp['qualifiers'].str.contains('IntentionalGoalAssist')]
    dfnt = dfp[dfp['endX']>=70]
    dfpen = dfp[(dfp['endX']>=87.5) & (dfp['endY']>=13.6) & (dfp['endY']<=54.6)]
    dfpro = dfp[(dfp['x']>=35) & (dfp['prog_pass']>=9.11) & (~dfp['qualifiers'].str.contains('CornerTaken|Frerkick'))]
    dfcros = dfp[(dfp['qualifiers'].str.contains('Cross')) & (~dfp['qualifiers'].str.contains('CornerTaken|Frerkick'))]
    dfxT = dfp[dfp['xT']>=0]
    dflb = dfp[(dfp['qualifiers'].str.contains('Longball'))]
    cutback = dfp[((dfp['x'] >= 88.54) & (dfp['x'] <= 105) & 
                       ((dfp['y'] >= 40.8) & (dfp['y'] <= 54.4) | (dfp['y'] >= 13.6) & (dfp['y'] <= 27.2)) & 
                       (dfp['endY'] >= 27.2) & (dfp['endY'] <= 40.8) & (dfp['endX'] >= 81.67))]
    next_act = df[(df['name']==pname) & (df['type'].shift(1)=='Pass') & (df['outcomeType'].shift(1)=='Successful')]
    ball_retain = next_act[(next_act['outcomeType']=='Successful') & ((next_act['type']!='Foul') | (next_act['type']!='Dispossessed'))]
    if len(next_act) != 0:
        ball_retention = round((len(ball_retain)/len(next_act))*100, 2)
    else:
        ball_retention = 0

    if len(dfp) != 0:
        name_counts = dfp['shortName'].value_counts()
        name_counts_df = name_counts.reset_index()
        name_counts_df.columns = ['name', 'count']
        name_counts_df = name_counts_df.sort_values(by='count', ascending=False)  
        name_counts_df = name_counts_df.reset_index()
        r_name = name_counts_df['name'][0]
        r_count = name_counts_df['count'][0]
    else:
        r_name = 'None'
        r_count = 0        

    pitch.lines(dfp.x, dfp.y, dfp.endX, dfp.endY, lw=3, transparent=True, comet=True,color=hcol, ax=ax, alpha=0.5)
    pitch.lines(dfkp.x, dfkp.y, dfkp.endX, dfkp.endY, lw=4, transparent=True, comet=True,color=violet, ax=ax, alpha=0.75)
    pitch.lines(dfas.x, dfas.y, dfas.endX, dfas.endY, lw=4, transparent=True, comet=True,color='green', ax=ax, alpha=0.75)
    pitch.scatter(dfp.endX, dfp.endY, s=30, edgecolor=hcol, linewidth=1, color=bg_color, zorder=2, ax=ax)
    pitch.scatter(dfkp.endX, dfkp.endY, s=40, edgecolor=violet, linewidth=1.5, color=bg_color, zorder=2, ax=ax)
    pitch.scatter(dfas.endX, dfas.endY, s=50, edgecolors='green', linewidths=1, marker='football', c=bg_color, zorder=2, ax=ax)

    avg_endY = dfp['endY'].median()
    avg_endX = dfp['endX'].median()
    ax.axvline(x=avg_endX, ymin=0, ymax=68, color='gray', linestyle='--', alpha=0.6, linewidth=2)
    ax.axhline(y=avg_endY, xmin=0, xmax=105, color='gray', linestyle='--', alpha=0.6, linewidth=2)
    ax.set_title(f"Passes Recieved", color=line_color, fontsize=25, fontweight='bold', y=1.03)
    ax_text(0, -3, f'''<Passes Received: {len(dfp)}> | <Key Passes Received: {len(dfkp)}> | <Assists Received: {len(dfas)}>
Passes Received in Final third: {len(dfnt)} | Passes Received in Opponent box: {len(dfpen)}
Progressive Passes Received: {len(dfpro)} | Crosses Received: {len(dfcros)} | Longballs Received: {len(dflb)}
Cutbacks Received: {len(cutback)} | Ball Retention: {ball_retention} % | xT Received: {dfxT['xT'].sum().round(2)}
Avg. Distance of Pass Receiving from Opponent Goal line: {round(105-dfp['endX'].median(),2)}m
Most Passes from: {r_name} ({r_count})''', fontsize=15, ha='left', va='top', ax=ax,
           highlight_textprops=[{'color':hcol}, {'color':violet}, {'color':'green'}])

    return

# fig,ax=plt.subplots(figsize=(10,10))
# individual_passes_recieved(ax, pname)

In [ ]:
fig,ax=plt.subplots(figsize=(10,10))
individual_passes_recieved(ax, pname)


### Resumen de la Función `individual_def_acts`

La función `individual_def_acts` se utiliza para visualizar las acciones defensivas de un jugador específico en un partido de fútbol. A continuación, se describen las etapas clave de la función:

1. **Configuración del Campo**:
   - Se inicializa un campo de fútbol utilizando la clase `Pitch`, configurando su diseño y colores. Se establecen los límites de los ejes del gráfico.

2. **Filtrado de Datos**:
   - Se filtra el DataFrame `df` para obtener las acciones defensivas del jugador especificado (`pname`).
   - Se clasifican las acciones defensivas en diferentes categorías:
     - **Tackles (ganados y fallidos)**.
     - **Intercepciones**.
     - **Recuperaciones de balón**.
     - **Despejes**.
     - **Faltas**.
     - **Duelo aéreo (ganados y fallidos)**.
     - **Pases bloqueados**.
     - **Disparos bloqueados**.
     - **Dribles pasados**.

3. **Cálculos Estadísticos**:
   - Se calcula el porcentaje de retención de balón posterior a una recuperación (proporción de recuperaciones de balón exitosas respecto a la cantidad total de intercepciones y recuperaciones).

4. **Visualización de Acciones**:
   - Se representan las acciones defensivas en el gráfico utilizando diferentes marcadores y colores:
     - **Tackles exitosos** en color específico (hcol).
     - **Tackles fallidos** en gris.
     - **Intercepciones** en forma de cuadrado.
     - **Recuperaciones de balón** en forma de círculo.
     - **Despejes** en forma de rombo.
     - **Faltas** en forma de cruz.
     - **Duelo aéreo** en forma de triángulo.

5. **Estadísticas en el Gráfico**:
   - Se agrega un título al gráfico y se muestran estadísticas relevantes sobre las acciones defensivas en la parte inferior, como el número total de tackles, intercepciones, recuperaciones, y otras métricas importantes.

6. **Retorno**:
   - La función no retorna ningún valor; su objetivo es crear una visualización en el eje proporcionado.

### Uso
Esta función permite a entrenadores, analistas y aficionados analizar el rendimiento defensivo de un jugador específico en un partido, proporcionando información visual sobre su efectividad en diferentes aspectos del juego defensivo. Es una herramienta útil para evaluar la contribución de un jugador a la defensa del equipo.

Este resumen destaca las características y el propósito de la función `individual_def_acts`, resaltando su utilidad en el análisis del rendimiento defensivo de los jugadores.

In [ ]:
def individual_def_acts(ax, pname):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, line_zorder=2, linewidth=2)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5, 105.5)
    ax.set_ylim(-0.5,68.5)

    playerdf = df[(df['name']==pname)]
    ball_wins = playerdf[(playerdf['type']=='Interception') | (playerdf['type']=='BallRecovery')]
    f_third = ball_wins[ball_wins['x']>=70]
    m_third = ball_wins[(ball_wins['x']>35) & (ball_wins['x']<70)]
    d_third = ball_wins[ball_wins['x']<=35]

    hp_tk = playerdf[(playerdf['type']=='Tackle')]
    hp_tk_u = playerdf[(playerdf['type']=='Tackle') & (playerdf['outcomeType']=='Unsuccessful')]
    hp_intc = playerdf[(playerdf['type']=='Interception')]
    hp_br = playerdf[playerdf['type']=='BallRecovery']
    hp_cl = playerdf[playerdf['type']=='Clearance']
    hp_fl = playerdf[playerdf['type']=='Foul']
    hp_ar = playerdf[(playerdf['type']=='Aerial') & (playerdf['qualifiers'].str.contains('Defensive'))]
    hp_ar_u = playerdf[(playerdf['type']=='Aerial') & (playerdf['outcomeType']=='Unsuccessful') & (playerdf['qualifiers'].str.contains('Defensive'))]
    pass_bl = playerdf[playerdf['type']=='BlockedPass']
    shot_bl = playerdf[playerdf['type']=='Save']
    drb_pst = playerdf[playerdf['type']=='Challenge']
    drb_tkl = df[(df['name']==pname) & (df['type']=='Tackle') & (df['type'].shift(1)=='TakeOn') & (df['outcomeType'].shift(1)=='Unsuccessful')]
    err_lat = playerdf[playerdf['qualifiers'].str.contains('LeadingToAttempt')]
    err_lgl = playerdf[playerdf['qualifiers'].str.contains('LeadingToGoal')]
    dan_frk = playerdf[(playerdf['type']=='Foul') & (playerdf['x']>16.5) & (playerdf['x']<35) & (playerdf['y']>13.6) & (playerdf['y']<54.4)]
    prbr = df[(df['name']==pname) & ((df['type']=='BallRecovery') | (df['type']=='Interception')) & (df['name'].shift(-1)==pname) & 
              (df['outcomeType'].shift(-1)=='Successful') &
              ((df['type'].shift(-1)!='Foul') | (df['type'].shift(-1)!='Dispossessed'))]
    if (len(hp_br)+len(hp_intc)) != 0:
        post_rec_ball_retention = round((len(prbr)/(len(hp_br)+len(hp_intc)))*100, 2)
    else:
        post_rec_ball_retention = 0

    pitch.scatter(hp_tk.x, hp_tk.y, s=250, c=hcol, lw=2.5, edgecolor=hcol, marker='+', hatch='/////', ax=ax)
    pitch.scatter(hp_tk_u.x, hp_tk_u.y, s=250, c='gray', lw=2.5, edgecolor='gray', marker='+', hatch='/////', ax=ax)
    pitch.scatter(hp_intc.x, hp_intc.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='s', hatch='/////', ax=ax)
    pitch.scatter(hp_br.x, hp_br.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='o', hatch='/////', ax=ax)
    pitch.scatter(hp_cl.x, hp_cl.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='d', hatch='/////', ax=ax)
    pitch.scatter(hp_fl.x, hp_fl.y, s=250, c=hcol, lw=2.5, edgecolor=hcol, marker='x', hatch='/////', ax=ax)
    pitch.scatter(hp_ar.x, hp_ar.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='^', hatch='/////', ax=ax)
    pitch.scatter(hp_ar_u.x, hp_ar_u.y, s=250, c='None', lw=2.5, edgecolor='gray', marker='^', hatch='/////', ax=ax)
    pitch.scatter(drb_pst.x, drb_pst.y, s=250, c='None', lw=2.5, edgecolor=hcol, marker='h', hatch='|||||', ax=ax)

    ax.set_title(f"Defensive Actions", color=line_color, fontsize=25, fontweight='bold', y=1.03)
    ax_text(0, -3, f'''Tackle (Win): {len(hp_tk)} ({len(hp_tk) - len(hp_tk_u)}) | Dribblers Tackled: {len(drb_tkl)} | Dribble past: {len(drb_pst)} | Interception: {len(hp_intc)}
Ball Recovery: {len(hp_br)} | Post Recovery Ball Retention: {post_rec_ball_retention} %  | Pass Block: {len(pass_bl)}
Ball Clearances: {len(hp_cl)} | Shots Blocked: {len(shot_bl)} | Aerial Duels (Win): {len(hp_ar)} ({len(hp_ar) - len(hp_ar_u)}) | Fouls: {len(hp_fl)}
Fouls infront of Penalty Box: {len(dan_frk)} | Error Led to Shot/Led to Goal: {len(err_lat)}/{len(err_lgl)}
Possession Win in Final third/Mid third/Defensive third: {len(f_third)}/{len(m_third)}/{len(d_third)}
''', fontsize=15, ha='left', va='top', ax=ax)
    return

# fig,ax=plt.subplots(figsize=(10,10))
# individual_def_acts(ax, pname)

In [ ]:
fig,ax=plt.subplots(figsize=(10,10))
individual_def_acts(ax, pname)


### Resumen de la Función `heatMap`

La función `heatMap` se utiliza para visualizar los toques y la distribución espacial de un jugador en un partido de fútbol a través de un mapa de calor en el campo. A continuación se describen las etapas clave de la función:

1. **Configuración del Campo**:
   - Se inicializa un campo de fútbol utilizando la clase `Pitch`, configurando el tipo de campo, colores y líneas. Se establecen los límites de los ejes del gráfico.

2. **Filtrado de Datos**:
   - Se filtra el DataFrame `df` para obtener las acciones del jugador especificado (`pname`).
   - Se excluyen tipos de eventos no relevantes como sustituciones, tarjetas y ciertos tipos de acciones.
   - Se crea un DataFrame adicional para representar el área fuera del campo.

3. **Cálculo de Estadísticas**:
   - Se calculan diferentes métricas relacionadas con los toques del jugador, como el número de toques en el área penal, en la última línea y la cantidad de toques totales.
   - Se calcula el área cubierta por el jugador utilizando el método de envoltura convexa (Convex Hull).
   - Se determinan las distancias recorridas en diferentes periodos del juego (primera mitad, segunda mitad, prórrogas) y se suman para obtener la distancia total cubierta.

4. **Visualización del Mapa de Calor**:
   - Se genera un mapa de calor utilizando un histograma 2D de los toques del jugador.
   - Se superponen los toques del jugador en el campo, utilizando diferentes marcadores y colores para representar distintos tipos de toques y eventos (e.g., goles, fallos, intentos guardados).

5. **Estadísticas en el Gráfico**:
   - Se agrega un título al gráfico y se presentan estadísticas relevantes sobre los toques, la distancia recorrida, el área cubierta y otros aspectos del rendimiento del jugador.

6. **Retorno**:
   - La función no retorna ningún valor; su objetivo es crear una visualización en el eje proporcionado.

### Uso
Esta función permite a entrenadores, analistas y aficionados analizar el rendimiento y la actividad de un jugador en el campo, ofreciendo una representación visual clara de su influencia en el juego a través de los toques y la movilidad. Es útil para evaluar cómo un jugador contribuye en diferentes fases del juego y su efectividad en áreas clave del campo.

Este resumen destaca las características y el propósito de la función `heatMap`, resaltando su utilidad en el análisis del rendimiento de los jugadores.

In [ ]:
def heatMap(ax,pname):
    pitch = Pitch(pitch_type='uefa', corner_arcs=True, pitch_color=bg_color, line_color=line_color, linewidth=2, line_zorder=3)
    pitch.draw(ax=ax)
    ax.set_xlim(-0.5,105.5)
    ax.set_ylim(-0.5,68.5) 

    df_player = df[df['name']==pname]
    df_player = df_player[~df_player['type'].str.contains('SubstitutionOff|SubstitutionOn|Card|Carry')]
    new_data = pd.DataFrame({'x': [-5, -5, 110, 110], 'y': [-5, 73, 73, -5]})
    df_player = pd.concat([df_player, new_data], ignore_index=True)
    flamingo_cmap = LinearSegmentedColormap.from_list("Flamingo - 100 colors", [bg_color, green, 'yellow', hcol, 'red'], N=500)
    # Create heatmap
    # pitch.kdeplot(df_player.x, df_player.y, ax=ax, fill=True, levels=5000, thresh=0.02, cut=4, cmap=flamingo_cmap)
    
    heatmap, xedges, yedges = np.histogram2d(df_player.x, df_player.y, bins=(12,12))
    extent = [xedges[0], xedges[-1], yedges[-1], yedges[0]]
    # extent = [0,105,68,0]
    ax.imshow(heatmap.T, extent=extent, cmap=flamingo_cmap, interpolation='bilinear')

    touches = df_player[df_player['isTouch']==1]
    final_third = touches[touches['x']>=70]
    pen_box = touches[(touches['x']>=88.5) & (touches['y']>=13.6) & (touches['y']<=54.4)]

    ax.scatter(touches.x, touches.y, marker='o', s=10, c='gray')

    points = touches[['x', 'y']].values
    hull = ConvexHull(points)
    # # Plotting the convex hull
    # ax.plot(points[:,0], points[:,1], 'o')
    # for simplex in hull.simplices:
    #     ax.plot(points[simplex, 0], points[simplex, 1], 'k-')
    # ax.fill(points[hull.vertices,0], points[hull.vertices,1], 'c', alpha=0.3)
    area_covered = round(hull.volume,2)
    area_perc = round((area_covered/(105*68))*100, 2)

    df_player = df[df['name']==pname]
    df_player = df_player[~df_player['type'].str.contains('CornerTaken|FreekickTaken|Card|CornerAwarded|SubstitutionOff|SubstitutionOn')]
    df_player = df_player[['x', 'y', 'period']]
    dfp_fh = df_player[df_player['period']=='FirstHalf']
    dfp_sh = df_player[df_player['period']=='SecondHalf']
    dfp_fpet = df_player[df_player['period']=='FirstPeriodOfExtraTime']
    dfp_spet = df_player[df_player['period']=='SecondPeriodOfExtraTime']
    
    dfp_fh['distance_covered'] = np.sqrt((dfp_fh['x'] - dfp_fh['x'].shift(-1))**2 + (dfp_fh['y'] - dfp_fh['y'].shift(-1))**2)
    dist_cov_fh = (dfp_fh['distance_covered'].sum()/1000).round(2)
    dfp_sh['distance_covered'] = np.sqrt((dfp_sh['x'] - dfp_sh['x'].shift(-1))**2 + (dfp_sh['y'] - dfp_sh['y'].shift(-1))**2)
    dist_cov_sh = (dfp_sh['distance_covered'].sum()/1000).round(2)
    dfp_fpet['distance_covered'] = np.sqrt((dfp_fpet['x'] - dfp_fpet['x'].shift(-1))**2 + (dfp_fpet['y'] - dfp_fpet['y'].shift(-1))**2)
    dist_cov_fpet = (dfp_fpet['distance_covered'].sum()/1000).round(2)
    dfp_spet['distance_covered'] = np.sqrt((dfp_spet['x'] - dfp_spet['x'].shift(-1))**2 + (dfp_spet['y'] - dfp_spet['y'].shift(-1))**2)
    dist_cov_spet = (dfp_spet['distance_covered'].sum()/1000).round(2)
    tot_dist_cov = dist_cov_fh + dist_cov_sh + dist_cov_fpet + dist_cov_spet

    ax.set_title(f"Touches and Heatmap", color=line_color, fontsize=25, fontweight='bold', y=1.03)
    ax.text(52.5, -3, f'''Touches: {len(touches)}  |  Touches in Final third: {len(final_third)}  |  Touches in Penalty Area: {len(pen_box)}
    Total Distances Covered: {round(tot_dist_cov,2)} Km
    Total Area Covered: {area_covered} sq.m ({area_perc}% of the Total Field Area)
    ''',
            fontsize=15, ha='center', va='top')

    return

# fig,ax=plt.subplots(figsize=(10,10))
# heatMap(ax, pname)

In [ ]:
fig,ax=plt.subplots(figsize=(10,10))
heatMap(ax, pname)

# Individual Player Dashboard

In [ ]:
df['name'].unique()

La función `playing_time` se utiliza para calcular el tiempo de juego de un jugador en un partido de fútbol. A continuación, se presenta un resumen de su funcionamiento:

### Resumen de la Función `playing_time`

1. **Entrada**: 
   - La función recibe como parámetro `pname`, que es el nombre del jugador para el cual se desea calcular el tiempo de juego.

2. **Filtrado de Datos**: 
   - Se filtra el DataFrame `df` para obtener solo las acciones del jugador correspondiente al nombre dado (`pname`).

3. **Manejo de Datos Faltantes**: 
   - La columna `isFirstEleven` (que indica si el jugador fue titular) se rellena con ceros en caso de que tenga valores faltantes.

4. **Identificación de Sustituciones**:
   - Se identifican las filas donde el jugador fue sustituido (`SubstitutionOff`) y las filas donde el jugador entró al partido (`SubstitutionOn`).

5. **Cálculo del Tiempo Jugado**:
   - Se determina el tiempo jugado en función de las siguientes condiciones:
     - **Titular sin sustituciones**: Si el jugador es titular y no ha sido sustituido, se asigna el tiempo total del partido (`max_min`).
     - **Titular con sustitución**: Si el jugador es titular y ha sido sustituido, se toma el minuto en que fue sustituido.
     - **Suplente**: Si el jugador no es titular, se calcula el tiempo jugado restando el minuto en que entró del tiempo total del partido.
     - **Por defecto**: Si no se cumplen las condiciones anteriores, se asigna cero como tiempo jugado.

6. **Retorno**:
   - La función retorna el tiempo jugado en minutos como un número entero.

### Ejemplo de Uso
Esta función es útil para obtener estadísticas de tiempo de juego de un jugador en un análisis post-partido, permitiendo evaluar su contribución al juego en términos de minutos jugados. 

### Posibles Mejora
- **Control de Errores**: Se podría agregar un manejo de errores para garantizar que `pname` siempre se encuentre en el DataFrame y que haya datos suficientes para calcular el tiempo jugado.
- **Flexibilidad en el Formato de Entrada**: Podría ser útil permitir el ingreso del nombre del jugador en diferentes formatos o convenciones, aumentando la robustez de la función. 

Este resumen describe la lógica de la función `playing_time`, destacando su propósito y funcionalidad en el contexto del análisis de rendimiento de los jugadores.

In [ ]:
# Function to get playing time in the match
def playing_time(pname):
    df_player = df[df['name']==pname]
    df_player['isFirstEleven'] = df_player['isFirstEleven'].fillna(0)
    df_sub_off = df_player[df_player['type']=='SubstitutionOff']
    df_sub_on  = df_player[df_player['type']=='SubstitutionOn']
    max_min = df['minute'].max()
    if df_player['isFirstEleven'].unique() == 1 and len(df_sub_off)==0:
        mins_played = max_min
    elif df_player['isFirstEleven'].unique() == 1 and len(df_sub_off)==1:
        mins_played = int(df_sub_off['minute'].unique())
    elif df_player['isFirstEleven'].unique()==0:
        mins_played = int(max_min - df_sub_on['minute'].unique())
    else:
        mins_played = 0

    return int(mins_played)

Este código genera y guarda figuras para todos los jugadores titulares en un partido de fútbol, capturando sus métricas individuales de rendimiento, como pases, carreras, tiros y acciones defensivas. Aquí tienes un desglose de cómo funciona, junto con algunas posibles mejoras:

### Descripción General del Código

1. **Identificar Jugadores Titulares**:
   - El código filtra el DataFrame `df` para extraer los nombres únicos de los jugadores que comenzaron el partido para los equipos local y visitante (`hteamName` y `ateamName`).
   - Se crean dos arreglos, `h_pnames` y `a_pnames`, que contienen los nombres de los jugadores titulares de cada equipo.

2. **Definición de la Función**:
   - La función `generate_and_save_figure` se define para crear una figura de múltiples paneles para cada jugador.
   - Acepta los parámetros `pname` (el nombre del jugador) y `team_name` (el equipo del jugador).

3. **Generar Subgráficas**:
   - Dentro de la función, se crea una cuadrícula de subgráficas de 2x3 para visualizar varias métricas del jugador.
   - Se llama a la función `playing_time` para calcular el tiempo total jugado por el jugador.

4. **Generar Gráficas Individuales**:
   - La función llama a varias funciones de trazado para generar métricas individuales para el jugador: 
     - **Mapa de Pases**: Visualiza los pases del jugador.
     - **Carreras**: Muestra las carreras realizadas por el jugador.
     - **Mapa de Tiros**: Muestra los tiros realizados.
     - **Pases Recibidos**: Ilustra los pases recibidos con éxito.
     - **Acciones Defensivas**: Resalta las acciones defensivas.
     - **Mapa de Calor**: Representa el mapa de calor del jugador basado en los toques.

5. **Añadir Anotaciones e Imágenes**:
   - Se añaden el nombre del jugador, el nombre del equipo, los goles marcados y los minutos jugados como anotaciones de texto en la figura.
   - Se incluyen los logotipos del equipo local y de la liga utilizando la función `add_image`.

6. **Guardar la Figura**:
   - La figura se puede guardar con el nombre del jugador y el ID del partido como parte del nombre del archivo (actualmente comentado).

7. **Recorrer Jugadores**:
   - La parte final del código itera a través de ambas listas de nombres de jugadores (`h_pnames` y `a_pnames`), generando figuras para cada jugador utilizando la función `generate_and_save_figure`.

### Posibles Mejoras

- **Manejo de Errores**: Considera agregar manejo de errores para gestionar escenarios donde los datos del jugador puedan estar faltantes o incompletos. Esto podría prevenir que el programa se detenga si ciertos jugadores no tienen datos.

- **URLs Dinámicas para los Logotipos**: Asegúrate de que las URLs para los logotipos del equipo y la liga sean válidas y accesibles. Considera añadir una opción de respaldo en caso de que no se puedan obtener las imágenes.

- **Guardado de Archivos**: Actualmente, la línea para guardar las figuras está comentada. Asegúrate de descomentar esta línea y manejar las rutas de archivo correctamente para evitar sobrescribir archivos existentes.

- **Optimización del Rendimiento**: Si el DataFrame es grande y el número de jugadores es alto, considera optimizar cómo se generan los gráficos para mejorar el rendimiento. Por ejemplo, podrías querer almacenar en caché resultados o graficar en lotes.

- **Parametrización**: Podrías parametrizar los tamaños y posiciones de las imágenes en la función `add_image` para hacerlas más flexibles para diferentes configuraciones de visualización.

### Ejemplo de Uso
Puedes llamar a este bloque de código en un script o en un cuaderno de Jupyter donde tu DataFrame `df` y las funciones necesarias para graficar (como `individual_passMap`, `individual_carry`, etc.) estén definidas. Asegúrate de que las bibliotecas requeridas (como `matplotlib`, `PIL` y `numpy`) estén importadas también.

Este código está bien estructurado para generar informes visuales completos sobre el rendimiento de los jugadores, adecuados para analistas, entrenadores o aficionados interesados en estadísticas detalladas del partido.

In [ ]:
h_starters = df[(df['isFirstEleven']==1) & (df['teamName']==hteamName)]
h_pnames = h_starters['name'].unique()
a_starters = df[(df['isFirstEleven']==1) & (df['teamName']==ateamName)]
a_pnames = a_starters['name'].unique()

# Function to generate and save the figure
def generate_and_save_figure(pname, team_name):
    fig, axs = plt.subplots(2, 3, figsize=(35, 18), facecolor='#f5f5f5')
    
    # Calculate minutes played
    mins_played = playing_time(pname)
    
    # Generate individual plots
    individual_passMap(axs[0, 0], pname)
    individual_carry(axs[0, 1], pname)
    Individual_ShotMap(axs[0, 2], pname)
    individual_passes_recieved(axs[1, 0], pname)
    individual_def_acts(axs[1, 1], pname)
    heatMap(axs[1, 2], pname)
    
    # Add text and images to the figure
    fig.text(0.2, 0.99, f'{pname}', fontsize=70, fontweight='bold', ha='left', va='center')
    fig.text(0.2, 0.94, f'in {hteamName} {hgoal_count} - {agoal_count} {ateamName}, GW 1, EPL 2024-25  |  Minutes played: {mins_played}', 
             fontsize=30, ha='left', va='center')

    himage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{hometeamId.iloc[0]['teamId']}.png")
    himage = Image.open(himage)
    add_image(himage, fig, left=0.115, bottom=0.94, width=0.085, height=0.085)
    
    timage = urlopen("file:FData/Leagues_logos/EPL_Logo.png")
    timage = Image.open(timage)
    add_image(timage, fig, left=0.825, bottom=0.94, width=0.085, height=0.085)

    
    # Save the figure with the player's name
    # fig.savefig(f"{pname}_in_{match_id}.png", bbox_inches='tight')
    plt.close(fig)

# Loop through all player names and generate figures for all the starting players
for pname in h_pnames:
    generate_and_save_figure(pname, hteamName)

for pname in a_pnames:
    generate_and_save_figure(pname, ateamName)

El código que has proporcionado genera y guarda una figura que muestra el rendimiento individual de un jugador, en este caso, "Phil Foden". Aquí te doy una descripción general de su funcionamiento y algunas recomendaciones para asegurarte de que funcione correctamente:

### Descripción del Código

1. **Función `generate_and_save_figure`**:
   - Esta función toma dos parámetros: `pname` (el nombre del jugador) y `team_name` (el nombre del equipo del jugador).
   - Crea una figura con subgráficas (2 filas y 3 columnas) utilizando Matplotlib.

2. **Cálculo del Tiempo Jugado**:
   - Llama a la función `playing_time` para calcular los minutos que el jugador ha estado en el campo durante el partido.

3. **Generación de Gráficas Individuales**:
   - Se generan varias gráficas relacionadas con el rendimiento del jugador:
     - Mapa de Pases
     - Carreras
     - Mapa de Tiros
     - Pases Recibidos
     - Acciones Defensivas
     - Mapa de Calor

4. **Añadir Texto e Imágenes**:
   - Se añaden anotaciones de texto a la figura, que incluyen el nombre del jugador, el resultado del partido y los minutos jugados.
   - Se cargan e insertan los logotipos del equipo y de la liga.

5. **Guardar la Figura**:
   - La línea para guardar la figura está comentada. Deberías descomentarla para guardar la imagen generada.

6. **Ejemplo de Uso**:
   - Al final del código, se llama a la función `generate_and_save_figure` con "Phil Foden" como nombre del jugador y `ateamName` como nombre del equipo.


In [ ]:
# Function to generate and save the figure
def generate_and_save_figure(pname, team_name):
    fig, axs = plt.subplots(2, 3, figsize=(35, 18), facecolor='#f5f5f5')
    
    # Calculate minutes played
    mins_played = playing_time(pname)
    
    # Generate individual plots
    individual_passMap(axs[0, 0], pname)
    individual_carry(axs[0, 1], pname)
    Individual_ShotMap(axs[0, 2], pname)
    individual_passes_recieved(axs[1, 0], pname)
    individual_def_acts(axs[1, 1], pname)
    heatMap(axs[1, 2], pname)
    
    # Add text and images to the figure
    fig.text(0.2, 0.99, f'{pname}', fontsize=70, fontweight='bold', ha='left', va='center')
    fig.text(0.2, 0.94, f'in {hteamName} {hgoal_count} - {agoal_count} {ateamName}, GW 1, EPL 2024-25  |  Minutes played: {mins_played} | code by: MPAD', 
             fontsize=30, ha='left', va='center')
    
    himage = urlopen(f"https://images.fotmob.com/image_resources/logo/teamlogo/{hometeamId.iloc[0]['teamId']}.png")
    himage = Image.open(himage)
    add_image(himage, fig, left=0.115, bottom=0.94, width=0.085, height=0.085)
    
    timage = urlopen("file:FData/Leagues_logos/EPL_Logo.png")
    timage = Image.open(timage)
    add_image(timage, fig, left=0.825, bottom=0.94, width=0.085, height=0.085)

    # Save the figure with the player's name
    # fig.savefig(f"{pname}_in_{match_id}.png", bbox_inches='tight')

generate_and_save_figure('Phil Foden', ateamName)

# Player Stats Counting

In [ ]:
def players_passing_stats(pname):
    dfpass = df[(df['type']=='Pass') & (df['name']==pname)]
    acc_pass = dfpass[dfpass['outcomeType']=='Successful']
    iac_pass = dfpass[dfpass['outcomeType']=='Unsuccessful']
    
    if len(dfpass) != 0:
        accurate_pass_perc = round((len(acc_pass)/len(dfpass))*100, 2)
    else:
        accurate_pass_perc = 0
    
    pro_pass = acc_pass[(acc_pass['prog_pass']>=9.11) & (acc_pass['x']>=35) &
                        (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Thr_ball = dfpass[(dfpass['qualifiers'].str.contains('Throughball')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Thr_ball_acc = Thr_ball[Thr_ball['outcomeType']=='Successful']
    Lng_ball = dfpass[(dfpass['qualifiers'].str.contains('Longball')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Lng_ball_acc = Lng_ball[Lng_ball['outcomeType']=='Successful']
    Crs_pass = dfpass[(dfpass['qualifiers'].str.contains('Cross')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    Crs_pass_acc = Crs_pass[Crs_pass['outcomeType']=='Successful']
    key_pass = dfpass[dfpass['qualifiers'].str.contains('KeyPass')]
    big_chnc = dfpass[dfpass['qualifiers'].str.contains('BigChanceCreated')]
    df_no_carry = df[df['type']!='Carry'].reset_index(drop=True)
    pre_asst = df_no_carry[(df_no_carry['qualifiers'].shift(-1).str.contains('IntentionalGoalAssist')) & (df_no_carry['type']=='Pass') & 
                           (df_no_carry['outcomeType']=='Successful') &  (df_no_carry['name']==pname)]
    shot_buildup = df_no_carry[(df_no_carry['qualifiers'].shift(-1).str.contains('KeyPass')) & (df_no_carry['type']=='Pass') & 
                           (df_no_carry['outcomeType']=='Successful') &  (df_no_carry['name']==pname)]
    g_assist = dfpass[dfpass['qualifiers'].str.contains('IntentionalGoalAssist')]
    fnl_thd = acc_pass[(acc_pass['endX']>=70) & (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    pen_box = acc_pass[(acc_pass['endX']>=88.5) & (acc_pass['endY']>=13.6) & (acc_pass['endY']<=54.4) & 
                       (~acc_pass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    frwd_pass = dfpass[(dfpass['pass_or_carry_angle']>= -85) & (dfpass['pass_or_carry_angle']<= 85) & 
                       (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    back_pass = dfpass[((dfpass['pass_or_carry_angle']>= 95) & (dfpass['pass_or_carry_angle']<= 180) | 
                        (dfpass['pass_or_carry_angle']>= -180) & (dfpass['pass_or_carry_angle']<= -95)) &
                       (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    side_pass = dfpass[((dfpass['pass_or_carry_angle']>= 85) & (dfpass['pass_or_carry_angle']<= 95) | 
                        (dfpass['pass_or_carry_angle']>= -95) & (dfpass['pass_or_carry_angle']<= -85)) & 
                       (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    frwd_pass_acc = frwd_pass[frwd_pass['outcomeType']=='Successful']
    back_pass_acc = back_pass[back_pass['outcomeType']=='Successful']
    side_pass_acc = side_pass[side_pass['outcomeType']=='Successful']
    corners = dfpass[dfpass['qualifiers'].str.contains('CornerTaken')]
    corners_acc = corners[corners['outcomeType']=='Successful']
    freekik = dfpass[dfpass['qualifiers'].str.contains('Freekick')]
    freekik_acc = freekik[freekik['outcomeType']=='Successful']
    thins = dfpass[dfpass['qualifiers'].str.contains('ThrowIn')]
    thins_acc = thins[thins['outcomeType']=='Successful']
    lngball = dfpass[(dfpass['qualifiers'].str.contains('Longball')) & (~dfpass['qualifiers'].str.contains('CornerTaken|Freekick'))]
    lngball_acc = lngball[lngball['outcomeType']=='Successful']

    if len(frwd_pass) != 0:
        Forward_Pass_Accuracy = round((len(frwd_pass_acc)/len(frwd_pass))*100, 2)
    else:
        Forward_Pass_Accuracy = 0
    
    df_xT_inc = (dfpass[dfpass['xT']>0])['xT'].sum().round(2)
    df_xT_dec = (dfpass[dfpass['xT']<0])['xT'].sum().round(2)
    total_xT = dfpass['xT'].sum().round(2)
    
    return {
        'Name': pname,
        'Total_Passes': len(dfpass),
        'Accurate_Passes': len(acc_pass),
        'Miss_Pass': len(iac_pass),
        'Passing_Accuracy': accurate_pass_perc,
        'Progressive_Passes': len(pro_pass),
        'Chances_Created': len(key_pass),
        'Big_Chances_Created': len(big_chnc),
        'Assists': len(g_assist),
        'Pre-Assists': len(pre_asst),
        'Buil-up_to_Shot': len(shot_buildup),
        'Final_Third_Passes': len(fnl_thd),
        'Passes_In_Penalty_Box': len(pen_box),
        'Through_Pass_Attempts': len(Thr_ball),
        'Accurate_Through_Passes': len(Thr_ball_acc),
        'Crosses_Attempts': len(Crs_pass),
        'Accurate_Crosses': len(Crs_pass_acc),
        'Longballs_Attempts': len(lngball),
        'Accurate_Longballs': len(lngball_acc),
        'Corners_Taken': len(corners),
        'Accurate_Corners': len(corners_acc),
        'Freekicks_Taken': len(freekik),
        'Accurate_Freekicks': len(freekik_acc),
        'ThrowIns_Taken': len(thins),
        'Accurate_ThrowIns': len(thins_acc),
        'Forward_Pass_Attempts': len(frwd_pass),
        'Accurate_Forward_Pass': len(frwd_pass_acc),
        'Side_Pass_Attempts': len(side_pass),
        'Accurate_Side_Pass': len(side_pass_acc),
        'Back_Pass_Attempts': len(back_pass),
        'Accurate_Back_Pass': len(back_pass_acc),
        'xT_Increased_From_Pass': df_xT_inc,
        'xT_Decreased_From_Pass': df_xT_dec,
        'Total_xT_From_Pass': total_xT 
    }


pnames = df['name'].unique()

# Create a list of dictionaries to store the counts for each player
data = []

for pname in pnames:
    counts = players_passing_stats(pname)
    data.append(counts)

# Convert the list of dictionaries to a DataFrame
passing_stats_df = pd.DataFrame(data)

# Sort the DataFrame by 'pr_count' in descending order
passing_stats_df = passing_stats_df.sort_values(by='xT_Decreased_From_Pass', ascending=False).reset_index(drop=True)
passing_stats_df

In [ ]:
def player_carry_stats(pname):
    df_carry = df[(df['type']=='Carry') & (df['name']==pname)]
    led_shot1 = df[(df['type']=='Carry') & (df['name']==pname) & (df['qualifiers'].shift(-1).str.contains('KeyPass'))]
    led_shot2 = df[(df['type']=='Carry') & (df['name']==pname) & (df['type'].shift(-1).str.contains('Shot'))]
    led_shot = pd.concat([led_shot1, led_shot2])
    led_goal1 = df[(df['type']=='Carry') & (df['name']==pname) & (df['qualifiers'].shift(-1).str.contains('IntentionalGoalAssist'))]
    led_goal2 = df[(df['type']=='Carry') & (df['name']==pname) & (df['type'].shift(-1)=='Goal')]
    led_goal = pd.concat([led_goal1, led_goal2])
    pro_carry = df_carry[(df_carry['prog_carry']>=9.11) & (df_carry['x']>=35)]
    fth_carry = df_carry[(df_carry['x']<70) & (df_carry['endX']>=70)]
    box_entry = df_carry[(df_carry['endX']>=88.5) & (df_carry['endY']>=13.6) & (df_carry['endY']<=54.4) &
                 ~((df_carry['x']>=88.5) & (df_carry['y']>=13.6) & (df_carry['y']<=54.6))]
    disp = df[(df['type']=='Carry') & (df['name']==pname) & (df['type'].shift(-1)=='Dispossessed')]
    df_to = df[(df['type']=='TakeOn') & (df['name']==pname)]
    t_ons = df_to[df_to['outcomeType']=='Successful']
    t_onu = df_to[df_to['outcomeType']=='Unsuccessful']
    df_xT_inc = df_carry[df_carry['xT']>0]
    df_xT_dec = df_carry[df_carry['xT']<0]
    total_xT = df_carry['xT'].sum().round(2)
    df_carry = df_carry.copy()
    df_carry.loc[:, 'Length'] = np.sqrt((df_carry['x'] - df_carry['endX'])**2 + (df_carry['y'] - df_carry['endY'])**2)
    median_length = round(df_carry['Length'].median(),2)
    total_length = round(df_carry['Length'].sum(),2)
    if len(df_to)!=0:
        success_rate = round((len(t_ons)/len(df_to))*100, 2)
    else:
        success_rate = 0
    
    return {
        'Name': pname,
        'Total_Carries': len(df_carry),
        'Progressive_Carries': len(pro_carry),
        'Carries_Led_to_Shot': len(led_shot),
        'Carries_Led_to_Goal': len(led_goal),
        'Carrier_Dispossessed': len(disp),
        'Carries_Into_Final_Third': len(fth_carry),
        'Carries_Into_Penalty_Box': len(box_entry),
        'Avg._Carry_Length': median_length,
        'Total_Carry_Length': total_length,
        'xT_Increased_From_Carries': df_xT_inc['xT'].sum().round(2),
        'xT_Decreased_From_Carries': df_xT_dec['xT'].sum().round(2),
        'Total_xT_From_Carries': total_xT,
        'TakeOn_Attempts': len(df_to),
        'Successful_TakeOns': len(t_ons)
    }

pnames = df['name'].unique()

# Create a list of dictionaries to store the counts for each player
data = []

for pname in pnames:
    counts = player_carry_stats(pname)
    data.append(counts)

# Convert the list of dictionaries to a DataFrame
carrying_stats_df = pd.DataFrame(data)

# Sort the DataFrame by 'pr_count' in descending order
carrying_stats_df = carrying_stats_df.sort_values(by='Total_Carries', ascending=False).reset_index(drop=True)
carrying_stats_df

In [ ]:
def player_shooting_stats(pname):

    op_sh = shots_df[(shots_df['playerName']==pname) & (shots_df['situation']=='RegularPlay')]

    goal = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='Goal')]
    miss = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='Miss')]
    save = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='AttemptSaved') & (shots_df['isBlocked']==0)]
    blok = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='AttemptSaved') & (shots_df['isBlocked']==1)]
    post = shots_df[(shots_df['playerName']==pname) & (shots_df['eventType']=='Post')]

    goal_bc = df[(df['name']==pname) & (df['type']=='Goal') & (df['qualifiers'].str.contains('BigChance'))]
    miss_bc = df[(df['name']==pname) & (df['type']=='MissedShots') & (df['qualifiers'].str.contains('BigChance'))]
    save_bc = df[(df['name']==pname) & (df['type']=='SavedShot') & (df['qualifiers'].str.contains('BigChance'))]
    post_bc = df[(df['name']==pname) & (df['type']=='ShotOnPost') & (df['qualifiers'].str.contains('BigChance'))]

    shots = df[(df['name']==pname) & ((df['type']=='Goal') | (df['type']=='MissedShots') | (df['type']=='SavedShot') | (df['type']=='ShotOnPost'))]
    out_box = shots[shots['qualifiers'].str.contains('OutOfBox')]
    shots = shots.copy()
    shots.loc[:, 'Length'] = np.sqrt((shots['x'] - 105)**2 + (shots['y'] - 34)**2)
    avg_dist = round(shots['Length'].mean(), 2)
    xG = shots_df[(shots_df['playerName']==pname)]['expectedGoals'].sum().round(2)
    xGOT = shots_df[(shots_df['playerName']==pname)]['expectedGoalsOnTarget'].sum().round(2)

    return {
        'Name': pname, 
        'Total_Shots': len(shots),
        'Open_Play_Shots': len(op_sh),
        'Goals': len(goal),
        'Shot_On_Post': len(post),
        'Shot_On_Target': len(save),
        'Shot_Off_Target': len(miss),
        'Shot_Blocked': len(blok),
        'Big_Chances': len(goal_bc)+len(miss_bc)+len(save_bc)+len(post_bc),
        'Big_Chances_Missed': len(miss_bc)+len(save_bc)+len(post_bc),
        'Shots_From_Outside_The_Box': len(out_box),
        'Shots_From_Inside_The_Box': len(shots) - len(out_box),
        'Avg._Shot_Distance': avg_dist,
        'xG': xG,
        'xGOT': xGOT
    }

pnames = df['name'].unique()

# Create a list of dictionaries to store the counts for each player
data = []

for pname in pnames:
    counts = player_shooting_stats(pname)
    data.append(counts)

# Convert the list of dictionaries to a DataFrame
shooting_stats_df = pd.DataFrame(data)

# Sort the DataFrame by 'pr_count' in descending order
shooting_stats_df = shooting_stats_df.sort_values(by='Total_Shots', ascending=False).reset_index(drop=True)
shooting_stats_df

In [ ]:
def player_pass_receiving_stats(pname):
    dfp = df[(df['type']=='Pass') & (df['outcomeType']=='Successful') & (df['name'].shift(-1)==pname)]
    dfkp = dfp[dfp['qualifiers'].str.contains('KeyPass')]
    dfas = dfp[dfp['qualifiers'].str.contains('IntentionalGoalAssist')]
    dfnt = dfp[dfp['endX']>=70]
    dfpen = dfp[(dfp['endX']>=87.5) & (dfp['endY']>=13.6) & (dfp['endY']<=54.6)]
    dfpro = dfp[(dfp['x']>=35) & (dfp['prog_pass']>=9.11) & (~dfp['qualifiers'].str.contains('CornerTaken|Frerkick'))]
    dfcros = dfp[(dfp['qualifiers'].str.contains('Cross')) & (~dfp['qualifiers'].str.contains('CornerTaken|Frerkick'))]
    dfxT = dfp[dfp['xT']>=0]
    dflb = dfp[(dfp['qualifiers'].str.contains('Longball'))]
    cutback = dfp[((dfp['x'] >= 88.54) & (dfp['x'] <= 105) & 
                       ((dfp['y'] >= 40.8) & (dfp['y'] <= 54.4) | (dfp['y'] >= 13.6) & (dfp['y'] <= 27.2)) & 
                       (dfp['endY'] >= 27.2) & (dfp['endY'] <= 40.8) & (dfp['endX'] >= 81.67))]
    next_act = df[(df['name']==pname) & (df['type'].shift(1)=='Pass') & (df['outcomeType'].shift(1)=='Successful')]
    ball_retain = next_act[(next_act['outcomeType']=='Successful') & ((next_act['type']!='Foul') | (next_act['type']!='Dispossessed'))]
    if len(next_act) != 0:
        ball_retention = round((len(ball_retain)/len(next_act))*100, 2)
    else:
        ball_retention = 0

    if len(dfp) != 0:
        name_counts = dfp['shortName'].value_counts()
        name_counts_df = name_counts.reset_index()
        name_counts_df.columns = ['name', 'count']
        name_counts_df = name_counts_df.sort_values(by='count', ascending=False)  
        name_counts_df = name_counts_df.reset_index()
        r_name = name_counts_df['name'][0]
        r_count = name_counts_df['count'][0]
    else:
        r_name = 'None'
        r_count = 0        

    return {
        'Name': pname,
        'Total_Passes_Received': len(dfp),
        'Key_Passes_Received': len(dfkp), 
        'Assists_Received': len(dfas),
        'Progressive_Passes_Received': len(dfpro),
        'Passes_Received_In_Final_Third': len(dfnt),
        'Passes_Received_In_Opponent_Penalty_Box': len(dfpen),
        'Crosses_Received': len(dfcros),
        'Longballs_Received': len(dflb),
        'Cutbacks_Received': len(cutback),
        'Ball_Retention': ball_retention,
        'xT_Received': dfxT['xT'].sum().round(2),
        'Avg._Distance_Of_Pass_Receiving_Form_Oppo._Goal_Line': round(105-dfp['endX'].median(),2),
        'Most_Passes_Received_From_(Count)': f'{r_name}({r_count})'
    }

pnames = df['name'].unique()

# Create a list of dictionaries to store the counts for each player
data = []

for pname in pnames:
    counts = player_pass_receiving_stats(pname)
    data.append(counts)

# Convert the list of dictionaries to a DataFrame
player_pass_receiving_stats_df = pd.DataFrame(data)

# Sort the DataFrame by 'pr_count' in descending order
player_pass_receiving_stats_df = player_pass_receiving_stats_df.sort_values(by='Total_Passes_Received', ascending=False).reset_index(drop=True)
player_pass_receiving_stats_df

In [ ]:
def player_defensive_stats(pname):
    playerdf = df[(df['name']==pname)]
    ball_wins = playerdf[(playerdf['type']=='Interception') | (playerdf['type']=='BallRecovery')]
    f_third = ball_wins[ball_wins['x']>=70]
    m_third = ball_wins[(ball_wins['x']>35) & (ball_wins['x']<70)]
    d_third = ball_wins[ball_wins['x']<=35]

    tk = playerdf[(playerdf['type']=='Tackle')]
    tk_u = playerdf[(playerdf['type']=='Tackle') & (playerdf['outcomeType']=='Unsuccessful')]
    intc = playerdf[(playerdf['type']=='Interception')]
    br = playerdf[playerdf['type']=='BallRecovery']
    cl = playerdf[playerdf['type']=='Clearance']
    fl = playerdf[playerdf['type']=='Foul']
    ar = playerdf[(playerdf['type']=='Aerial') & (playerdf['qualifiers'].str.contains('Defensive'))]
    ar_u = playerdf[(playerdf['type']=='Aerial') & (playerdf['outcomeType']=='Unsuccessful') & (playerdf['qualifiers'].str.contains('Defensive'))]
    pass_bl = playerdf[playerdf['type']=='BlockedPass']
    shot_bl = playerdf[playerdf['type']=='SavedShot']
    drb_pst = playerdf[playerdf['type']=='Challenge']
    drb_tkl = df[(df['name']==pname) & (df['type']=='Tackle') & (df['type'].shift(1)=='TakeOn') & (df['outcomeType'].shift(1)=='Unsuccessful')]
    err_lat = playerdf[playerdf['qualifiers'].str.contains('LeadingToAttempt')]
    err_lgl = playerdf[playerdf['qualifiers'].str.contains('LeadingToGoal')]
    dan_frk = playerdf[(playerdf['type']=='Foul') & (playerdf['x']>16.5) & (playerdf['x']<35) & (playerdf['y']>13.6) & (playerdf['y']<54.4)]
    prbr = df[(df['name']==pname) & ((df['type']=='BallRecovery') | (df['type']=='Interception')) & (df['name'].shift(-1)==pname) & 
              (df['outcomeType'].shift(-1)=='Successful') &
              ((df['type'].shift(-1)!='Foul') | (df['type'].shift(-1)!='Dispossessed'))]
    if (len(br)+len(intc)) != 0:
        post_rec_ball_retention = round((len(prbr)/(len(br)+len(intc)))*100, 2)
    else:
        post_rec_ball_retention = 0

    return {
        'Name': pname,
        'Total_Tackles': len(tk),
        'Tackles_Won': len(tk_u),
        'Dribblers_Tackled': len(drb_tkl),
        'Dribble_Past': len(drb_pst),
        'Interception': len(intc),
        'Ball_Recoveries': len(br), 
        'Post_Recovery_Ball_Retention': post_rec_ball_retention,
        'Pass_Blocked': len(pass_bl),
        'Ball_Clearances': len(cl),
        'Shots_Blocked': len(shot_bl),
        'Aerial_Duels': len(ar),
        'Aerial_Duels_Won': len(ar) - len(ar_u),
        'Fouls_Committed': len(fl),
        'Fouls_Infront_Of_Own_Penalty_Box': len(dan_frk),
        'Error_Led_to_Shot': len(err_lat),
        'Error_Led_to_Goal': len(err_lgl),
        'Possession Win in Final third': len(f_third),
        'Possession Win in Mid third': len(m_third),
        'Possession Win in Defensive third': len(d_third)
    }

pnames = df['name'].unique()

# Create a list of dictionaries to store the counts for each player
data = []

for pname in pnames:
    counts = player_defensive_stats(pname)
    data.append(counts)

# Convert the list of dictionaries to a DataFrame
player_defensive_stats_df = pd.DataFrame(data)

# Sort the DataFrame by 'pr_count' in descending order
player_defensive_stats_df = player_defensive_stats_df.sort_values(by='Total_Tackles', ascending=False).reset_index(drop=True)
player_defensive_stats_df

In [ ]:
def touches_stats(pname):
    df_player = df[df['name']==pname]
    df_player = df_player[~df_player['type'].str.contains('SubstitutionOff|SubstitutionOn|Card|Carry')]

    touches = df_player[df_player['isTouch']==1]
    final_third = touches[touches['x']>=70]
    pen_box = touches[(touches['x']>=88.5) & (touches['y']>=13.6) & (touches['y']<=54.4)]

    points = touches[['x', 'y']].values
    hull = ConvexHull(points)
    area_covered = round(hull.volume,2)
    area_perc = round((area_covered/(105*68))*100, 2)

    df_player = df[df['name']==pname]
    df_player = df_player[~df_player['type'].str.contains('CornerTaken|FreekickTaken|Card|CornerAwarded|SubstitutionOff|SubstitutionOn')]
    df_player = df_player[['x', 'y', 'period']]
    dfp_fh = df_player[df_player['period']=='FirstHalf']
    dfp_sh = df_player[df_player['period']=='SecondHalf']
    dfp_fpet = df_player[df_player['period']=='FirstPeriodOfExtraTime']
    dfp_spet = df_player[df_player['period']=='SecondPeriodOfExtraTime']
    
    dfp_fh['distance_covered'] = np.sqrt((dfp_fh['x'] - dfp_fh['x'].shift(-1))**2 + (dfp_fh['y'] - dfp_fh['y'].shift(-1))**2)
    dist_cov_fh = (dfp_fh['distance_covered'].sum()/1000).round(2)
    dfp_sh['distance_covered'] = np.sqrt((dfp_sh['x'] - dfp_sh['x'].shift(-1))**2 + (dfp_sh['y'] - dfp_sh['y'].shift(-1))**2)
    dist_cov_sh = (dfp_sh['distance_covered'].sum()/1000).round(2)
    dfp_fpet['distance_covered'] = np.sqrt((dfp_fpet['x'] - dfp_fpet['x'].shift(-1))**2 + (dfp_fpet['y'] - dfp_fpet['y'].shift(-1))**2)
    dist_cov_fpet = (dfp_fpet['distance_covered'].sum()/1000).round(2)
    dfp_spet['distance_covered'] = np.sqrt((dfp_spet['x'] - dfp_spet['x'].shift(-1))**2 + (dfp_spet['y'] - dfp_spet['y'].shift(-1))**2)
    dist_cov_spet = (dfp_spet['distance_covered'].sum()/1000).round(2)
    tot_dist_cov = dist_cov_fh + dist_cov_sh + dist_cov_fpet + dist_cov_spet

    # df_speed = df.copy()
    # df_speed['len_Km'] = np.where((df_speed['type']=='Carry'), 
    #                        np.sqrt((df_speed['x'] - df_speed['endX'])**2 + (df_speed['y'] - df_speed['endY'])**2)/1000, 0)
    # df_speed['speed'] = np.where((df_speed['type']=='Carry'), 
    #                              (df_speed['len_Km']*60)/(df_speed['cumulative_mins'].shift(-1) - df_speed['cumulative_mins'].shift(1)) , 0)
    # speed_df = df_speed[(df_speed['name']==pname)]
    # avg_speed = round(speed_df['speed'].median(),2)

    return {
        'Name': pname,
        'Total_Touches': len(touches),
        'Touches_In_Final_Third': len(final_third),
        'Touches_In_Opponent_Penalty_Box': len(pen_box),
        'Total_Distance_Covered(Km)': tot_dist_cov,
        'Total_Area_Covered': area_covered
    }

starters = df[df['isFirstEleven']==1]
pnames = starters['name'].unique()[1:]

# Create a list of dictionaries to store the counts for each player
data = []

for pname in pnames:
    counts = touches_stats(pname)
    data.append(counts)

# Convert the list of dictionaries to a DataFrame
touches_stats_df = pd.DataFrame(data)

# Sort the DataFrame by 'pr_count' in descending order
touches_stats_df = touches_stats_df.sort_values(by='Total_Area_Covered', ascending=False).reset_index(drop=True)
touches_stats_df

In [ ]:
player_stats_df = shooting_stats_df.merge(passing_stats_df, on='Name', how='left')
player_stats_df = player_stats_df.merge(carrying_stats_df, on='Name', how='left')
player_stats_df = player_stats_df.merge(player_pass_receiving_stats_df, on='Name', how='left')
player_stats_df = player_stats_df.merge(player_defensive_stats_df, on='Name', how='left')
player_stats_df = player_stats_df.merge(touches_stats_df, on='Name', how='left')
player_stats_df

In [ ]:
player_stats_df.to_csv(os.path.join("FData", f"{fotmob_matchId}_player_stats_fotmob.csv"))